# Runtime and GPU check

Select your Colab kernel in VS Code, then run all cells. The output describes the selected kernel's machine. A successful GPU check lists an NVIDIA GPU and, when PyTorch is installed with CUDA support, completes a small calculation on it.

In [1]:
import platform
import sys
from pathlib import Path

print('Python:', sys.version)
print('System:', platform.platform())
print('Working directory:', Path.cwd())
print('Kernel connection works.')

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
System: Linux-6.6.122+-x86_64-with-glibc2.39
Working directory: /content
Kernel connection works.


In [2]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
        capture_output=True, text=True, timeout=30,
    )
    print(result.stdout or result.stderr)
    if result.returncode:
        print('GPU inspection failed; check the selected runtime.')
else:
    print('nvidia-smi is unavailable. Check that you selected a GPU runtime.')

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07



In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    if exc.name != 'torch':
        raise
    print('PyTorch is not installed in this kernel; skipping the tensor check.')
else:
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        values = torch.tensor([1.0, 2.0, 3.0], device='cuda')
        result = (values * values).sum().item()
        assert result == 14.0, f'Unexpected result: {result}'
        print('GPU calculation passed:', result)
    else:
        print('GPU calculation skipped. Check the runtime and PyTorch CUDA support.')

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU calculation passed: 14.0


In [ ]:
from datasets import Dataset, DatasetDict

def short_value(value):
    # Avoid printing image data itself
    if hasattr(value, "mode") and hasattr(value, "size"):
        return f"<image: mode={value.mode}, size={value.size}>"

    text = repr(value)
    return text if len(text) <= 300 else text[:300] + " ..."

found_data = False

for name, value in list(globals().items()):
    if isinstance(value, DatasetDict):
        found_data = True
        print(f"\nVariable: {name} (DatasetDict)")
        print("Splits:", list(value.keys()))

        if "train" in value:
            train_split = value["train"]
            print("Train rows:", len(train_split))
            print("Train columns:", train_split.column_names)

            row = train_split[0]
            print("First training example:")
            for key, item in row.items():
                print(f"  {key}: {short_value(item)}")

    elif isinstance(value, Dataset):
        found_data = True
        print(f"\nVariable: {name} (Dataset)")
        print("Rows:", len(value))
        print("Columns:", value.column_names)

        row = value[0]
        print("First example:")
        for key, item in row.items():
            print(f"  {key}: {short_value(item)}")

if not found_data:
    print("No Hugging Face Dataset or DatasetDict found in the notebook.")
    print("Tell me what variable you used to load SLAKE, or paste your data-loading cell.")

: 

In [1]:
from datasets import Dataset, DatasetDict

def short_value(value):
    # Avoid printing image data itself
    if hasattr(value, "mode") and hasattr(value, "size"):
        return f"<image: mode={value.mode}, size={value.size}>"

    text = repr(value)
    return text if len(text) <= 300 else text[:300] + " ..."

found_data = False

for name, value in list(globals().items()):
    if isinstance(value, DatasetDict):
        found_data = True
        print(f"\nVariable: {name} (DatasetDict)")
        print("Splits:", list(value.keys()))

        if "train" in value:
            train_split = value["train"]
            print("Train rows:", len(train_split))
            print("Train columns:", train_split.column_names)

            row = train_split[0]
            print("First training example:")
            for key, item in row.items():
                print(f"  {key}: {short_value(item)}")

    elif isinstance(value, Dataset):
        found_data = True
        print(f"\nVariable: {name} (Dataset)")
        print("Rows:", len(value))
        print("Columns:", value.column_names)

        row = value[0]
        print("First example:")
        for key, item in row.items():
            print(f"  {key}: {short_value(item)}")

if not found_data:
    print("No Hugging Face Dataset or DatasetDict found in the notebook.")
    print("Tell me what variable you used to load SLAKE, or paste your data-loading cell.")

No Hugging Face Dataset or DatasetDict found in the notebook.
Tell me what variable you used to load SLAKE, or paste your data-loading cell.


In [2]:
%pip -q install huggingface_hub

from huggingface_hub import hf_hub_download
import json

train_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="train.json",
    repo_type="dataset",
)

with open(train_path, "r", encoding="utf-8") as file:
    slake_train_data = json.load(file)

print("Loaded file:", train_path)
print("Top-level type:", type(slake_train_data).__name__)

if isinstance(slake_train_data, list):
    print("Number of training examples:", len(slake_train_data))
    sample = slake_train_data[0]

elif isinstance(slake_train_data, dict):
    print("Top-level keys:", list(slake_train_data.keys()))

    # Look for a list of examples inside the JSON object
    rows = next(
        (
            value for value in slake_train_data.values()
            if isinstance(value, list) and value
        ),
        None,
    )

    if rows is not None:
        print("Number of examples in first list:", len(rows))
        sample = rows[0]
    else:
        sample = slake_train_data

else:
    sample = slake_train_data

print("\nFirst example:")
print(json.dumps(sample, indent=2, ensure_ascii=False)[:3000])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

Loaded file: /root/.cache/huggingface/hub/datasets--BoKelvin--SLAKE/snapshots/a9083ce6c34ac3ffb17671a605962924d8a8f9e9/train.json
Top-level type: list
Number of training examples: 9835

First example:
{
  "img_id": 1,
  "img_name": "xmlab1/source.jpg",
  "question": "What modality is used to take this image?",
  "answer": "MRI",
  "q_lang": "en",
  "location": "Abdomen",
  "modality": "MRI",
  "answer_type": "OPEN",
  "base_type": "vqa",
  "content_type": "Modality",
  "triple": [
    "vhead",
    "_",
    "_"
  ],
  "qid": 0
}


In [3]:
from huggingface_hub import hf_hub_download
from zipfile import ZipFile
import json

image_folder = slake_train_data[0]["img_name"].split("/")[0]
print("Image folder:", image_folder)

imgs_zip_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="imgs.zip",
    repo_type="dataset",
)

with ZipFile(imgs_zip_path) as archive:
    matching_files = []

    for path in archive.namelist():
        normalized_path = path.replace("\\", "/").strip("/")
        parts = normalized_path.split("/")

        if image_folder in parts:
            matching_files.append(normalized_path)

    print("\nFiles for this image:")
    for path in matching_files:
        print(" ", path)

    detection_files = [
        path for path in matching_files
        if path.lower().endswith("/detection.json")
        or path.lower() == "detection.json"
    ]

    if detection_files:
        detection_data = json.loads(archive.read(detection_files[0]))
        print("\nDetection annotation:")
        print(json.dumps(detection_data, indent=2, ensure_ascii=False)[:3000])
    else:
        print("\nNo detection.json found for this image folder.")

Image folder: xmlab1


imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            


Files for this image:
  imgs/xmlab1
  imgs/xmlab1/question.json
  imgs/xmlab1/source.jpg
  imgs/xmlab1/mask.png
  imgs/xmlab1/detection.json

Detection annotation:
[
  {
    "Liver": [
      54.0,
      106.0,
      30.0,
      31.0
    ]
  }
]


In [4]:
from zipfile import ZipFile
from io import BytesIO
from PIL import Image

first_row = slake_train_data[0]
image_id = first_row["img_id"]
image_name = first_row["img_name"]

print("Image ID:", image_id)
print("Image name:", image_name)

print("\nQuestions for this image:")
for row in slake_train_data:
    if row.get("img_id") == image_id and row.get("q_lang") == "en":
        print(
            f"QID {row.get('qid')}: "
            f"{row.get('question')} "
            f"-> {row.get('answer')} "
            f"[{row.get('content_type')}]"
        )

source_file = next(
    (path for path in matching_files if path.lower().endswith("/source.jpg")),
    None,
)

if source_file:
    with ZipFile(imgs_zip_path) as archive:
        image = Image.open(BytesIO(archive.read(source_file)))
        print("\nImage dimensions (width, height):", image.size)
else:
    print("\nCould not find source.jpg in the matching files.")

Image ID: 1
Image name: xmlab1/source.jpg

Questions for this image:
QID 0: What modality is used to take this image? -> MRI [Modality]
QID 1: Which part of the body does this image belong to? -> Abdomen [Position]
QID 2: What is the mr weighting in this image? -> T2 [Modality]
QID 3: Does the picture contain liver? -> Yes [Organ]
QID 4: Does the picture contain kidney? -> No [Organ]
QID 5: Does the picture contain spleen? -> No [Organ]
QID 6: Where is the liver? -> Right [Position]

Image dimensions (width, height): (256, 256)


In [5]:
import json
from zipfile import ZipFile
from pathlib import PurePosixPath

image_name = "xmlab1/source.jpg"
image_folder = PurePosixPath(image_name).parent.as_posix()

# Find this image's detection.json inside the downloaded imgs.zip
with ZipFile(imgs_zip_path) as archive:
    detection_path = next(
        name for name in archive.namelist()
        if name.replace("\\", "/").endswith(
            f"{image_folder}/detection.json"
        )
    )
    detection_data = json.loads(archive.read(detection_path))

# Extract the Liver box from the annotation
liver_box = None
for item in detection_data:
    if "Liver" in item:
        liver_box = item["Liver"]
        break

# Make one label record per English question for this image
image_questions = [
    row for row in slake_train_data
    if row.get("img_name") == image_name
    and row.get("q_lang") == "en"
]

labels = []

for row in image_questions:
    question = row["question"].lower()
    answer = row["answer"].lower()

    asks_about_liver = "liver" in question
    has_positive_liver_answer = answer in {"yes", "right", "left"}

    use_liver_box = (
        asks_about_liver
        and has_positive_liver_answer
        and liver_box is not None
    )

    labels.append({
        "img_id": row["img_id"],
        "img_name": row["img_name"],
        "qid": row["qid"],
        "question": row["question"],
        "answer": row["answer"],
        "bbox_xywh_pixels": liver_box if use_liver_box else None,
        "grounding_status": (
            "grounded"
            if use_liver_box
            else "no_positive_region"
            if "does the picture contain" in question and answer == "no"
            else "not_required"
        )
    })

print(json.dumps(labels, indent=2))

[
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 0,
    "question": "What modality is used to take this image?",
    "answer": "MRI",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 1,
    "question": "Which part of the body does this image belong to?",
    "answer": "Abdomen",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 2,
    "question": "What is the mr weighting in this image?",
    "answer": "T2",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 3,
    "question": "Does the picture contain liver?",
    "answer": "Yes",
    "bbox_xywh_pixels": [
      54.0,
      106.0,
      30.0,
      31.0
    ],
    "grounding_status": "grounded"
  },
  {
    "img_id": 1,
    "img_name": "xm

In [6]:
for item in labels:
    print(
        f"QID {item['qid']}: answer={item['answer']}, "
        f"status={item['grounding_status']}, "
        f"box={item['bbox_xywh_pixels']}"
    )

QID 0: answer=MRI, status=not_required, box=None
QID 1: answer=Abdomen, status=not_required, box=None
QID 2: answer=T2, status=not_required, box=None
QID 3: answer=Yes, status=grounded, box=[54.0, 106.0, 30.0, 31.0]
QID 4: answer=No, status=no_positive_region, box=None
QID 5: answer=No, status=no_positive_region, box=None
QID 6: answer=Right, status=grounded, box=[54.0, 106.0, 30.0, 31.0]


In [7]:
from zipfile import ZipFile
from pathlib import PurePosixPath

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

with ZipFile(imgs_zip_path) as archive:
    files = [
        name.replace("\\", "/").lstrip("./")
        for name in archive.namelist()
    ]

    source_files = {
        name.removeprefix("imgs/")
        for name in files
        if name.endswith("/source.jpg")
    }

    detection_folders = {
        str(PurePosixPath(name).parent).removeprefix("imgs/")
        for name in files
        if name.endswith("/detection.json")
    }

question_images = {
    row["img_name"]
    for row in english_rows
}

question_folders = {
    PurePosixPath(name).parent.as_posix()
    for name in question_images
}

print("English training questions:", len(english_rows))
print("Unique images with English questions:", len(question_images))
print("Question images found in archive:", len(question_images & source_files))
print("Question image folders with detection.json:", len(question_folders & detection_folders))
print("Example question image paths:", sorted(question_images)[:5])

English training questions: 4919
Unique images with English questions: 450
Question images found in archive: 450
Question image folders with detection.json: 450
Example question image paths: ['xmlab1/source.jpg', 'xmlab100/source.jpg', 'xmlab101/source.jpg', 'xmlab104/source.jpg', 'xmlab107/source.jpg']


In [8]:
import json
import os
import re
from pathlib import PurePosixPath
from zipfile import ZipFile

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

training_records = []
grounded_count = 0
missing_images = []

with ZipFile(imgs_zip_path) as archive:
    archive_files = [
        name.replace("\\", "/")
        for name in archive.namelist()
    ]

    for row in english_rows:
        image_name = row["img_name"]
        folder = PurePosixPath(image_name).parent.as_posix()

        image_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + image_name)),
            None
        )
        detection_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + folder + "/detection.json")),
            None
        )

        if image_zip_path is None:
            missing_images.append(image_name)
            continue

        # Extract the image so the training dataset can load it by path.
        image_path = os.path.join("/content/slake_images", image_name)
        os.makedirs(os.path.dirname(image_path), exist_ok=True)

        if not os.path.exists(image_path):
            with open(image_path, "wb") as image_file:
                image_file.write(archive.read(image_zip_path))

        # Read this image's organ boxes.
        detections = []
        if detection_zip_path is not None:
            detections = json.loads(archive.read(detection_zip_path))

        question = row["question"].lower()
        answer = row["answer"].strip().lower()
        bbox = None
        matched_organ = None

        # Attach a box only when the question names that organ and
        # the answer indicates it is present or asks where it is.
        for item in detections:
            for organ, box in item.items():
                organ_words = re.findall(r"[a-z]+", organ.lower())
                question_words = re.findall(r"[a-z]+", question)

                if organ_words and all(word in question_words for word in organ_words):
                    asks_presence = any(
                        word in question
                        for word in ["contain", "present", "visible", "is there"]
                    )
                    asks_location = any(
                        word in question
                        for word in ["where", "which side", "position"]
                    )

                    positive_answer = answer in {
                        "yes", "right", "left", "present"
                    }

                    if positive_answer and (asks_presence or asks_location):
                        bbox = box
                        matched_organ = organ
                        break
            if bbox is not None:
                break

        if bbox is not None:
            grounded_count += 1

        training_records.append({
            "img_id": row["img_id"],
            "img_name": image_name,
            "image_path": image_path,
            "qid": row["qid"],
            "question": row["question"],
            "answer": row["answer"],
            "bbox_xywh_pixels": bbox,
            "grounded_organ": matched_organ,
        })

print("Training records created:", len(training_records))
print("Records with a matched box:", grounded_count)
print("Missing images:", len(missing_images))
print("Example record:", json.dumps(training_records[0], indent=2))

Training records created: 4919
Records with a matched box: 174
Missing images: 0
Example record: {
  "img_id": 1,
  "img_name": "xmlab1/source.jpg",
  "image_path": "/content/slake_images/xmlab1/source.jpg",
  "qid": 0,
  "question": "What modality is used to take this image?",
  "answer": "MRI",
  "bbox_xywh_pixels": null,
  "grounded_organ": null
}


In [9]:
from collections import Counter
from PIL import Image

grounded_records = [
    row for row in training_records
    if row["bbox_xywh_pixels"] is not None
]

print("Grounded examples:")
for row in grounded_records[:5]:
    print(
        f"QID {row['qid']}: {row['question']} "
        f"-> {row['answer']} | organ={row['grounded_organ']} "
        f"| box={row['bbox_xywh_pixels']}"
    )

valid_boxes = 0
invalid_boxes = []

for row in grounded_records:
    x, y, width, height = row["bbox_xywh_pixels"]

    with Image.open(row["image_path"]) as image:
        image_width, image_height = image.size

    is_valid = (
        x >= 0 and y >= 0 and width > 0 and height > 0
        and x + width <= image_width
        and y + height <= image_height
    )

    if is_valid:
        valid_boxes += 1
    else:
        invalid_boxes.append(row["img_name"])

print("\nGrounded records:", len(grounded_records))
print("Boxes inside image boundaries:", valid_boxes)
print("Invalid boxes:", len(invalid_boxes))
print("Grounded organ counts:", dict(Counter(
    row["grounded_organ"] for row in grounded_records
)))

Grounded examples:
QID 3: Does the picture contain liver? -> Yes | organ=Liver | box=[54.0, 106.0, 30.0, 31.0]
QID 6: Where is the liver? -> Right | organ=Liver | box=[54.0, 106.0, 30.0, 31.0]
QID 68: Does the picture contain liver? -> Yes | organ=Liver | box=[50.0, 102.0, 109.0, 151.0]
QID 70: Does the picture contain spleen? -> Yes | organ=Spleen | box=[190.0, 175.0, 75.0, 69.0]
QID 244: Does the picture contain liver? -> Yes | organ=Liver | box=[48.0, 141.0, 55.0, 106.0]

Grounded records: 174
Boxes inside image boundaries: 174
Invalid boxes: 0
Grounded organ counts: {'Liver': 119, 'Spleen': 20, 'Brain Stem': 1, 'Spinal Cord': 11, 'Esophagus': 2, 'Brain Edema': 1, 'Stomach': 3, 'Small Bowel': 7, 'Heart': 2, 'Bladder': 1, 'Colon': 7}


In [ ]:
from collections import Counter
from PIL import Image

grounded_records = [
    row for row in training_records
    if row["bbox_xywh_pixels"] is not None
]

print("Grounded examples:")
for row in grounded_records[:5]:
    print(
        f"QID {row['qid']}: {row['question']} "
        f"-> {row['answer']} | organ={row['grounded_organ']} "
        f"| box={row['bbox_xywh_pixels']}"
    )

valid_boxes = 0
invalid_boxes = []

for row in grounded_records:
    x, y, width, height = row["bbox_xywh_pixels"]

    with Image.open(row["image_path"]) as image:
        image_width, image_height = image.size

    is_valid = (
        x >= 0 and y >= 0 and width > 0 and height > 0
        and x + width <= image_width
        and y + height <= image_height
    )

    if is_valid:
        valid_boxes += 1
    else:
        invalid_boxes.append(row["img_name"])

print("\nGrounded records:", len(grounded_records))
print("Boxes inside image boundaries:", valid_boxes)
print("Invalid boxes:", len(invalid_boxes))
print("Grounded organ counts:", dict(Counter(
    row["grounded_organ"] for row in grounded_records
)))

In [1]:
import importlib.metadata as metadata
import torch

print("PyTorch:", torch.__version__)

for package in ["transformers", "peft", "bitsandbytes", "trl"]:
    try:
        print(f"{package}:", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU memory (GB):", round(gpu.total_memory / 1024**3, 2))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

for name in ["model", "gemma_model", "processor", "gemma_processor"]:
    value = globals().get(name)
    if value is not None:
        print(f"{name} type:", type(value).__name__)

PyTorch: 2.11.0+cu130
transformers: 5.18.0
peft: 0.21.1
bitsandbytes: not installed
trl: not installed

CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56
BF16 supported: True


In [2]:
print("GPU compute capability:", torch.cuda.get_device_capability(0))
print("Native BF16 capability check:",
      torch.cuda.get_device_capability(0) >= (8, 0))

print("\nNotebook variables containing model, processor, or gemma:")
for name, value in list(globals().items()):
    if any(word in name.lower() for word in ["model", "processor", "gemma"]):
        if not name.startswith("_"):
            print(f"{name}: {type(value).__name__}")

GPU compute capability: (7, 5)
Native BF16 capability check: False

Notebook variables containing model, processor, or gemma:


In [3]:
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.3 MB/s eta 0:00:00:00:0100:01


In [4]:
import bitsandbytes as bnb
import torch

print("bitsandbytes version:", bnb.__version__)
print("4-bit layer available:", hasattr(bnb.nn, "Linear4bit"))
print("CUDA available:", torch.cuda.is_available())

bitsandbytes version: 0.50.2
4-bit layer available: True
CUDA available: True


In [5]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)

print("Model class:", type(model).__name__)
print("Processor class:", type(processor).__name__)
print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("Device map:", getattr(model, "hf_device_map", "not available"))
print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model class: Gemma4ForConditionalGeneration
Processor class: Gemma4Processor
Loaded in 4-bit: True
Device map: not available
GPU memory used (GB): 6.28


In [6]:
from collections import Counter
from torch.nn import Linear
from bitsandbytes.nn import Linear4bit

linear_modules = [
    (name, type(module).__name__)
    for name, module in model.named_modules()
    if isinstance(module, (Linear, Linear4bit))
]

print("Model type:", model.config.model_type)
print("Linear layer count:", len(linear_modules))
print("Layer name endings:", dict(Counter(
    name.split(".")[-1] for name, _ in linear_modules
)))

print("\nFirst 30 linear layer names:")
for name, module_type in linear_modules[:30]:
    print(f"{name} [{module_type}]")

Model type: gemma4
Linear layer count: 526
Layer name endings: {'input_proj': 1, 'linear': 232, 'q_proj': 35, 'k_proj': 15, 'v_proj': 15, 'o_proj': 35, 'gate_proj': 35, 'up_proj': 35, 'down_proj': 35, 'per_layer_input_gate': 35, 'per_layer_projection': 35, 'per_layer_model_projection': 1, 'input_proj_linear': 1, 'relative_k_proj': 12, 'output_proj': 1, 'embedding_projection': 2, 'lm_head': 1}

First 30 linear layer names:
model.vision_tower.patch_embedder.input_proj [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.q_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.k_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.v_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.o_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.mlp.gate_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.mlp.up_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.mlp.down_proj.linear [Linear4bit]
model.vision_tow

In [7]:
for name, module in model.named_modules():
    if "language_model" in name and (
        name.endswith(".q_proj") or name.endswith(".q_proj.linear")
    ):
        print(name, "->", type(module).__name__)

model.language_model.layers.0.self_attn.q_proj -> Linear4bit
model.language_model.layers.1.self_attn.q_proj -> Linear4bit
model.language_model.layers.2.self_attn.q_proj -> Linear4bit
model.language_model.layers.3.self_attn.q_proj -> Linear4bit
model.language_model.layers.4.self_attn.q_proj -> Linear4bit
model.language_model.layers.5.self_attn.q_proj -> Linear4bit
model.language_model.layers.6.self_attn.q_proj -> Linear4bit
model.language_model.layers.7.self_attn.q_proj -> Linear4bit
model.language_model.layers.8.self_attn.q_proj -> Linear4bit
model.language_model.layers.9.self_attn.q_proj -> Linear4bit
model.language_model.layers.10.self_attn.q_proj -> Linear4bit
model.language_model.layers.11.self_attn.q_proj -> Linear4bit
model.language_model.layers.12.self_attn.q_proj -> Linear4bit
model.language_model.layers.13.self_attn.q_proj -> Linear4bit
model.language_model.layers.14.self_attn.q_proj -> Linear4bit
model.language_model.layers.15.self_attn.q_proj -> Linear4bit
model.language_mod

In [8]:
from collections import Counter
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

target_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj"}

lora_targets = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and ".self_attn." in name
    and name.rsplit(".", 1)[-1] in target_suffixes
]

print("LoRA target count:", len(lora_targets))
print("Targets by type:", dict(Counter(
    name.rsplit(".", 1)[-1] for name in lora_targets
)))

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=lora_targets,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

LoRA target count: 100
Targets by type: {'q_proj': 35, 'k_proj': 15, 'v_proj': 15, 'o_proj': 35}


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.75 GiB. GPU 0 has a total capacity of 14.56 GiB of which 6.58 GiB is free. Including non-PyTorch memory, this process has 7.98 GiB memory in use. Of the allocated memory 7.06 GiB is allocated by PyTorch, and 825.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [9]:
import gc
import torch
from transformers import AutoModelForMultimodalLM

# Remove the model that may have been partly changed by the failed call.
del model
gc.collect()
torch.cuda.empty_cache()

# Reload the clean 4-bit model.
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)

# Freeze the base model without converting its weights to FP32.
for parameter in model.parameters():
    parameter.requires_grad = False

model.config.use_cache = False

print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("Trainable base parameters:",
      sum(parameter.numel() for parameter in model.parameters()
          if parameter.requires_grad))
print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [10]:
import torch

free_bytes, total_bytes = torch.cuda.mem_get_info()

print("GPU total (GB):", round(total_bytes / 1024**3, 2))
print("GPU currently free (GB):", round(free_bytes / 1024**3, 2))
print("PyTorch allocated (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))
print("PyTorch reserved (GB):", round(
    torch.cuda.memory_reserved() / 1024**3, 2
))

GPU total (GB): 14.56
GPU currently free (GB): 6.58
PyTorch allocated (GB): 7.06
PyTorch reserved (GB): 7.87


In [11]:
import json
import os

checkpoint_path = "/content/slake_training_records.json"

with open(checkpoint_path, "w", encoding="utf-8") as file:
    json.dump(training_records, file)

print("Saved records:", len(training_records))
print("Checkpoint:", checkpoint_path)
print("File size (MB):", round(
    os.path.getsize(checkpoint_path) / 1024**2, 2
))

NameError: name 'training_records' is not defined

In [12]:
for name in [
    "training_records",
    "slake_train_data",
    "imgs_zip_path",
    "model",
    "processor",
    "torch",
]:
    print(f"{name}:", "available" if name in globals() else "missing")

if "torch" in globals() and torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU free (GB):", round(free_bytes / 1024**3, 2))
    print("PyTorch allocated (GB):", round(
        torch.cuda.memory_allocated() / 1024**3, 2
    ))

training_records: missing
slake_train_data: missing
imgs_zip_path: missing
model: missing
processor: available
torch: available
GPU free (GB): 6.58
PyTorch allocated (GB): 7.06


In [2]:
import torch

free_bytes, total_bytes = torch.cuda.mem_get_info()
print("GPU total (GB):", round(total_bytes / 1024**3, 2))
print("GPU free (GB):", round(free_bytes / 1024**3, 2))
print("PyTorch allocated (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

AssertionError: Torch not compiled with CUDA enabled

In [3]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Python: /usr/bin/python3
PyTorch: 2.11.0+cpu
CUDA available: False
/bin/bash: line 1: nvidia-smi: command not found


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU free (GB):", round(free_bytes / 1024**3, 2))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU free (GB): 14.46


In [2]:
import importlib.metadata as metadata
import os
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("\nPackage versions:")

for package in ["transformers", "peft", "bitsandbytes", "datasets"]:
    try:
        print(f"{package}:", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("\nFiles currently in /content:")
print(os.listdir("/content")[:20])

GPU: Tesla T4

Package versions:
transformers: 5.18.0
peft: 0.21.1
bitsandbytes: not installed
datasets: 4.8.5

Files currently in /content:
['.config', 'sample_data']


In [3]:
%pip install -q bitsandbytes==0.50.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.9 MB/s eta 0:00:00:00:0100:01


In [4]:
import torch
import bitsandbytes
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model: Gemma4ForConditionalGeneration
Processor: Gemma4Processor
Loaded in 4-bit: True
GPU memory used (GB): 6.28


In [5]:
from collections import Counter
from peft import LoraConfig, TaskType, get_peft_model

# Freeze the quantized base model without changing its weight precision.
for parameter in model.parameters():
    parameter.requires_grad = False

model.config.use_cache = False

# Target attention projections in the language model only.
target_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj"}
lora_targets = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and ".self_attn." in name
    and name.rsplit(".", 1)[-1] in target_suffixes
]

print("LoRA target count:", len(lora_targets))
print("Targets by type:", dict(Counter(
    name.rsplit(".", 1)[-1] for name in lora_targets
)))

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=lora_targets,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

LoRA target count: 100
Targets by type: {'q_proj': 35, 'k_proj': 15, 'v_proj': 15, 'o_proj': 35}
trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525
GPU memory used (GB): 6.29


In [1]:
import json
import os
from huggingface_hub import hf_hub_download

train_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="train.json",
    repo_type="dataset",
)

imgs_zip_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="imgs.zip",
    repo_type="dataset",
)

with open(train_path, "r", encoding="utf-8") as file:
    slake_train_data = json.load(file)

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

print("Training examples:", len(slake_train_data))
print("English training questions:", len(english_rows))
print("Images archive found:", os.path.exists(imgs_zip_path))
print("Images archive size (MB):", round(
    os.path.getsize(imgs_zip_path) / 1024**2, 2
))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

Training examples: 9835
English training questions: 4919
Images archive found: True
Images archive size (MB): 202.51


In [ ]:
import importlib
import os
import subprocess
import sys

# The notebook runs on a Colab kernel, so the repository code must be cloned there.
REPO_URL = "https://github.com/Thushi404/grounded-med-vqa-gemma4-slake.git"
REPO_DIR = "/content/grounded-med-vqa-gemma4-slake"
REPO_BRANCH = "dev"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import src.data
importlib.reload(src.data)
from src.data import build_records, to_target_text

In [ ]:
from collections import Counter

# Box matching rules live in src/data.py so that every split uses the same logic.
training_records, missing_images = build_records(english_rows, imgs_zip_path)

grounded_training = [row for row in training_records if row["regions"]]

print("Training records:", len(training_records))
print("Records with at least one box:", len(grounded_training))
print("Records with more than one box:", sum(len(row["regions"]) > 1 for row in grounded_training))
print("Missing images:", len(missing_images))
print("Grounded records by question type:", dict(Counter(row["content_type"] for row in grounded_training)))
print("Example:", grounded_training[0])

In [3]:
import json
from huggingface_hub import hf_hub_download

validation_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="validation.json",
    repo_type="dataset",
)

with open(validation_path, "r", encoding="utf-8") as file:
    slake_validation_data = json.load(file)

validation_english_rows = [
    row for row in slake_validation_data
    if row.get("q_lang") == "en"
]

validation_images = {
    row["img_name"] for row in validation_english_rows
}

print("Validation examples:", len(slake_validation_data))
print("English validation questions:", len(validation_english_rows))
print("Unique validation images:", len(validation_images))
print("Example:", validation_english_rows[0])

validation.json:   0%|          | 0.00/639k [00:00<?, ?B/s]

Validation examples: 2099
English validation questions: 1053
Unique validation images: 96
Example: {'img_id': 0, 'img_name': 'xmlab0/source.jpg', 'question': 'What modality is used to take this image?', 'answer': 'MRI', 'q_lang': 'en', 'location': 'Abdomen', 'modality': 'MRI', 'answer_type': 'OPEN', 'base_type': 'vqa', 'content_type': 'Modality', 'triple': ['vhead', '_', '_'], 'qid': 9835}


In [ ]:
from collections import Counter

validation_records, missing_validation_images = build_records(validation_english_rows, imgs_zip_path)

grounded_validation = [row for row in validation_records if row["regions"]]

print("Validation records:", len(validation_records))
print("Records with at least one box:", len(grounded_validation))
print("Records with more than one box:", sum(len(row["regions"]) > 1 for row in grounded_validation))
print("Missing images:", len(missing_validation_images))
print("Grounded records by question type:", dict(Counter(row["content_type"] for row in grounded_validation)))

In [ ]:
from PIL import Image

image_sizes = {}

def add_target_text(row):
    image_path = row["image_path"]

    if image_path not in image_sizes:
        with Image.open(image_path) as image:
            image_sizes[image_path] = image.size

    image_width, image_height = image_sizes[image_path]

    example = row.copy()
    example["target_text"] = to_target_text(row, image_width, image_height)
    return example

training_examples = [
    add_target_text(row) for row in training_records
]
validation_examples = [
    add_target_text(row) for row in validation_records
]

print("Example without a box:")
print(training_examples[0]["target_text"])

grounded_example = next(
    row for row in training_examples
    if row["regions"]
)
print("\nExample with a box:")
print(grounded_example["question"])
print(grounded_example["target_text"])

In [6]:
sample = training_examples[0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": sample["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {sample['question']}"
                ),
            },
        ],
    },
    {
        "role": "assistant",
        "content": sample["target_text"],
    },
]

encoded = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=False,
    return_assistant_tokens_mask=True,
)

print("Question:", sample["question"])
print("Target:", sample["target_text"])
print("Processor output keys:", list(encoded.keys()))

for key, value in encoded.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print("Assistant mask available:", "assistant_masks" in encoded)
if "assistant_masks" in encoded:
    print("Assistant tokens marked:", int(encoded["assistant_masks"].sum()))

NameError: name 'processor' is not defined

In [7]:
from transformers import AutoProcessor

MODEL_ID = "google/gemma-4-E2B-it"
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Processor:", type(processor).__name__)

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

Processor: Gemma4Processor


In [8]:
sample = training_examples[0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": sample["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {sample['question']}"
                ),
            },
        ],
    },
    {
        "role": "assistant",
        "content": sample["target_text"],
    },
]

encoded = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=False,
    return_assistant_tokens_mask=True,
)

print("Question:", sample["question"])
print("Target:", sample["target_text"])
print("Processor output keys:", list(encoded.keys()))

for key, value in encoded.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print("Assistant mask available:", "assistant_masks" in encoded)
if "assistant_masks" in encoded:
    print("Assistant tokens marked:", int(encoded["assistant_masks"].sum()))

[transformers] return_assistant_tokens_mask==True but chat template does not contain `{% generation %}` keyword.


Question: What modality is used to take this image?
Target: {"answer":"MRI","region":null}
Processor output keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'assistant_masks']
input_ids shape: (1, 301)
attention_mask shape: (1, 301)
mm_token_type_ids shape: (1, 301)
pixel_values shape: (1, 2520, 768)
image_position_ids shape: (1, 2520, 2)
assistant_masks shape: (1, 301)
Assistant mask available: True
Assistant tokens marked: 0


In [9]:
prompt_encoded = processor.apply_chat_template(
    messages[:1],  # user message only
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
)

prompt_ids = prompt_encoded["input_ids"][0]
full_ids = encoded["input_ids"][0]

same_prefix = (
    len(full_ids) >= len(prompt_ids)
    and (full_ids[:len(prompt_ids)] == prompt_ids).all().item()
)

print("Prompt token count:", len(prompt_ids))
print("Full example token count:", len(full_ids))
print("Prompt matches start of full example:", same_prefix)

Prompt token count: 290
Full example token count: 301
Prompt matches start of full example: True


In [10]:
labels = encoded["input_ids"].clone()
labels[:, :len(prompt_ids)] = -100

supervised_ids = labels[0][labels[0] != -100]

print("Prompt tokens masked:", int((labels == -100).sum()))
print("Answer tokens used for training:", len(supervised_ids))
print(
    "Text used for training:",
    processor.tokenizer.decode(supervised_ids, skip_special_tokens=False),
)

Prompt tokens masked: 290
Answer tokens used for training: 11
Text used for training: {"answer":"MRI","region":null}<turn|>



In [11]:
def collate_vqa(batch):
    if len(batch) != 1:
        raise ValueError("This collator currently expects batch size 1.")

    example = batch[0]

    user_message = {
        "role": "user",
        "content": [
            {"type": "image", "url": example["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {example['question']}"
                ),
            },
        ],
    }

    full_messages = [
        user_message,
        {"role": "assistant", "content": example["target_text"]},
    ]

    full = processor.apply_chat_template(
        full_messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=False,
    )

    prompt = processor.apply_chat_template(
        [user_message],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )

    prompt_ids = prompt["input_ids"][0]
    full_ids = full["input_ids"][0]

    if not torch.equal(full_ids[:len(prompt_ids)], prompt_ids):
        raise ValueError("Prompt tokens do not match the start of the full example.")

    labels = full["input_ids"].clone()
    labels[:, :len(prompt_ids)] = -100
    full["labels"] = labels

    return dict(full)


test_batch = collate_vqa([training_examples[0]])

print("Batch keys:", list(test_batch.keys()))
for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print("Answer tokens used for training:",
      int((test_batch["labels"] != -100).sum()))

NameError: name 'torch' is not defined

In [12]:
import torch

test_batch = collate_vqa([training_examples[0]])

print("Batch keys:", list(test_batch.keys()))
for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print(
    "Answer tokens used for training:",
    int((test_batch["labels"] != -100).sum()),
)

Batch keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'labels']
input_ids shape: (1, 301)
attention_mask shape: (1, 301)
mm_token_type_ids shape: (1, 301)
pixel_values shape: (1, 2520, 768)
image_position_ids shape: (1, 2520, 2)
labels shape: (1, 301)
Answer tokens used for training: 11


In [13]:
from transformers import Trainer, TrainingArguments

print("Training examples:", len(training_examples))
print("Validation examples:", len(validation_examples))
print("Model:", type(model).__name__)

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/gemma4_slake_checkpoints",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="no",
    save_strategy="no",
    logging_steps=10,
    optim="adamw_torch",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_examples,
    eval_dataset=validation_examples,
    data_collator=collate_vqa,
    processing_class=processor,
)

print("Trainer created successfully.")
print("Training has not started.")

Training examples: 4919
Validation examples: 1053


NameError: name 'model' is not defined

In [14]:
print("Model-like variables currently loaded:")

for name, value in list(globals().items()):
    if ("model" in name.lower() or "gemma" in name.lower()):
        if hasattr(value, "parameters"):
            print(name, "->", type(value).__name__)

print("LoRA config available:", "lora_config" in globals())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU free (GB):", round(free_bytes / 1024**3, 2))

Model-like variables currently loaded:
LoRA config available: False
CUDA available: True
GPU free (GB): 14.46


In [15]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)
print("GPU memory used (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [16]:
import importlib.util

print("bitsandbytes package found:",
      importlib.util.find_spec("bitsandbytes") is not None)

try:
    import bitsandbytes as bnb
    print("bitsandbytes version:", bnb.__version__)
except Exception as error:
    print("Import failed:", type(error).__name__)
    print("Details:", str(error))

bitsandbytes package found: False
Import failed: ModuleNotFoundError
Details: No module named 'bitsandbytes'


In [17]:
%pip install -q bitsandbytes==0.50.2

import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00:00:0100:01
bitsandbytes version: 0.50.2
CUDA available: True


In [18]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)
print("GPU memory used (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [19]:
import transformers
from transformers.utils import is_bitsandbytes_available, is_accelerate_available

print("Transformers:", transformers.__version__)
print("Transformers detects bitsandbytes:", is_bitsandbytes_available())
print("Accelerate available:", is_accelerate_available())

Transformers: 5.18.0
Transformers detects bitsandbytes: False
Accelerate available: True


In [20]:
import json

with open("/content/training_examples_step7_backup.json", "w", encoding="utf-8") as f:
    json.dump(training_examples, f, ensure_ascii=False)

with open("/content/validation_examples_step7_backup.json", "w", encoding="utf-8") as f:
    json.dump(validation_examples, f, ensure_ascii=False)

print("Training examples saved:", len(training_examples))
print("Validation examples saved:", len(validation_examples))
print("Backup files saved in /content")

Training examples saved: 4919
Validation examples saved: 1053
Backup files saved in /content


In [21]:
import json
import torch
import bitsandbytes as bnb
import transformers
from transformers.utils import is_bitsandbytes_available

with open("/content/training_examples_step7_backup.json", encoding="utf-8") as f:
    training_examples = json.load(f)

with open("/content/validation_examples_step7_backup.json", encoding="utf-8") as f:
    validation_examples = json.load(f)

print("Training examples restored:", len(training_examples))
print("Validation examples restored:", len(validation_examples))
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("Transformers detects bitsandbytes:", is_bitsandbytes_available())
print("CUDA available:", torch.cuda.is_available())

Training examples restored: 4919
Validation examples restored: 1053
Transformers: 5.18.0
bitsandbytes: 0.50.2
Transformers detects bitsandbytes: False
CUDA available: True


In [22]:
import importlib.util
import importlib.metadata
import transformers.utils.import_utils as import_utils

spec = importlib.util.find_spec("bitsandbytes")

print("bitsandbytes module path:", spec.origin if spec else "not found")

try:
    print("bitsandbytes package version:",
          importlib.metadata.version("bitsandbytes"))
except Exception as error:
    print("Package metadata error:", type(error).__name__, str(error))

print("Transformers cached bitsandbytes flag:",
      getattr(import_utils, "_bitsandbytes_available", "not found"))
print("Transformers bitsandbytes check:",
      import_utils.is_bitsandbytes_available())

bitsandbytes module path: /usr/local/lib/python3.13/dist-packages/bitsandbytes/__init__.py
bitsandbytes package version: 0.50.2
Transformers cached bitsandbytes flag: not found
Transformers bitsandbytes check: False


In [23]:
import inspect
from transformers.utils import import_utils

print(inspect.getsource(import_utils.is_bitsandbytes_available))

@lru_cache
@_make_compile_constant
def is_bitsandbytes_available(min_version: str = BITSANDBYTES_MIN_VERSION) -> bool:
    is_available, bitsandbytes_version = _is_package_available("bitsandbytes", return_version=True)
    return is_available and version.parse(bitsandbytes_version) >= version.parse(min_version)



In [24]:
import transformers.utils.import_utils as import_utils

print("Required bitsandbytes version:",
      import_utils.BITSANDBYTES_MIN_VERSION)
print("Package detection:",
      import_utils._is_package_available("bitsandbytes", return_version=True))

import_utils.is_bitsandbytes_available.cache_clear()

print("Detection after clearing cache:",
      import_utils.is_bitsandbytes_available())

Required bitsandbytes version: 0.46.1
Package detection: (True, '0.50.2')
Detection after clearing cache: True


In [25]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)
print("GPU memory used (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model: Gemma4ForConditionalGeneration
Loaded in 4-bit: True
GPU memory used (GB): 6.28


In [26]:
from peft import LoraConfig, get_peft_model

target_suffixes = ("q_proj", "k_proj", "v_proj", "o_proj")

lora_targets = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and name.endswith(target_suffixes)
    and module.__class__.__name__ == "Linear4bit"
]

print("LoRA target count:", len(lora_targets))

for parameter in model.parameters():
    parameter.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=lora_targets,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

LoRA target count: 100
trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525


In [27]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Processor:", type(processor).__name__)
print("Training examples:", len(training_examples))
print("Validation examples:", len(validation_examples))

Processor: Gemma4Processor
Training examples: 4919
Validation examples: 1053


In [28]:
import torch

def collate_vqa(batch):
    if len(batch) != 1:
        raise ValueError("This collator expects batch size 1.")

    example = batch[0]

    user_message = {
        "role": "user",
        "content": [
            {"type": "image", "url": example["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {example['question']}"
                ),
            },
        ],
    }

    full = processor.apply_chat_template(
        [
            user_message,
            {"role": "assistant", "content": example["target_text"]},
        ],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=False,
    )

    prompt = processor.apply_chat_template(
        [user_message],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )

    prompt_ids = prompt["input_ids"][0]
    full_ids = full["input_ids"][0]

    if not torch.equal(full_ids[:len(prompt_ids)], prompt_ids):
        raise ValueError("Prompt does not match the start of the full example.")

    labels = full["input_ids"].clone()
    labels[:, :len(prompt_ids)] = -100
    full["labels"] = labels

    return dict(full)


test_batch = collate_vqa([training_examples[0]])

print("Batch keys:", list(test_batch.keys()))
for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))
print("Answer tokens used for training:",
      int((test_batch["labels"] != -100).sum()))

Batch keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'labels']
input_ids shape: (1, 301)
attention_mask shape: (1, 301)
mm_token_type_ids shape: (1, 301)
pixel_values shape: (1, 2520, 768)
image_position_ids shape: (1, 2520, 2)
labels shape: (1, 301)
Answer tokens used for training: 11


In [29]:
from transformers import Trainer, TrainingArguments

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/gemma4_slake_checkpoints",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="no",
    save_strategy="no",
    logging_steps=10,
    optim="adamw_torch",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_examples,
    eval_dataset=validation_examples,
    data_collator=collate_vqa,
    processing_class=processor,
)

print("Trainer created successfully.")
print("Training has not started.")

Trainer created successfully.
Training has not started.


In [ ]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

account = HfApi().whoami()
print("Signed in as:", account["name"])

In [ ]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

account = HfApi().whoami()
print("Signed in as:", account["name"])

In [ ]:
from huggingface_hub import HfApi
print("Signed in as:", HfApi().whoami()["name"])

In [ ]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

account = HfApi().whoami()
print("Signed in as:", account["name"])

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
print("Notebook is responding")

In [ ]:
print("Notebook is responding")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [1]:
print("Kernel connected")

Kernel connected


In [2]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

account = HfApi().whoami()
print("Signed in as:", account["name"])

ModuleNotFoundError: No module named 'huggingface_hub'

In [3]:
%pip install -q huggingface_hub

import huggingface_hub
print("huggingface_hub version:", huggingface_hub.__version__)

Note: you may need to restart the kernel to use updated packages.
huggingface_hub version: 2.2.0


In [4]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

account = HfApi().whoami()
print("Signed in as:", account["name"])

Signed in as: Thushi2003


In [5]:
from pathlib import Path

for filename in [
    "training_examples_step7_backup.json",
    "validation_examples_step7_backup.json",
]:
    path = Path("/content") / filename
    print(filename, "exists:", path.exists())

training_examples_step7_backup.json exists: False
validation_examples_step7_backup.json exists: False


In [6]:
from pathlib import Path

print("Files currently in /content:")
for path in Path("/content").iterdir():
    print(path.name)

Files currently in /content:


FileNotFoundError: [WinError 3] The system cannot find the path specified: '\\content'

In [1]:
import sys
import torch

print("Python:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: /usr/bin/python3
CUDA available: False


In [2]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Colab marker:", os.environ.get("COLAB_RELEASE_TAG", "not set"))
print("/content exists:", Path("/content").exists())

try:
    import google.colab
    print("google.colab package: available")
except ImportError:
    print("google.colab package: not available")

Python: /usr/bin/python3
Colab marker: release-colab-external-images_20261012-060052_RC00
/content exists: True
google.colab package: available


In [3]:
try:
    import huggingface_hub
    from huggingface_hub import get_token

    print("huggingface_hub version:", huggingface_hub.__version__)
    print("Token saved in this Colab runtime:", bool(get_token()))
except ModuleNotFoundError:
    print("huggingface_hub is not installed in this Colab runtime.")

huggingface_hub version: 1.33.0
Token saved in this Colab runtime: False


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [4]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

account = HfApi().whoami()
print("Signed in to this Colab runtime as:", account["name"])

Signed in to this Colab runtime as: Thushi2003


In [5]:
from pathlib import Path

print("Files in /content:")
for path in Path("/content").iterdir():
    print(path.name)

print("training_examples in memory:", "training_examples" in globals())
print("validation_examples in memory:", "validation_examples" in globals())

Files in /content:
.config
sample_data
training_examples in memory: False
validation_examples in memory: False


In [6]:
from huggingface_hub import HfApi

files = HfApi().list_repo_files(
    repo_id="BoKelvin/SLAKE",
    repo_type="dataset"
)

for name in files:
    print(name)

.gitattributes
KG.zip
README.md
imgs.zip
mask.txt
test.json
train.json
validation.json


In [7]:
from pathlib import Path
from huggingface_hub import hf_hub_download

repo_id = "BoKelvin/SLAKE"
data_dir = Path("/content/slake_source")
data_dir.mkdir(parents=True, exist_ok=True)

for filename in ["train.json", "validation.json", "imgs.zip"]:
    path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=filename,
        local_dir=str(data_dir),
    )
    print(f"{filename}: downloaded ({Path(path).stat().st_size / 1024**2:.1f} MB)")

train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

train.json: downloaded (2.8 MB)


validation.json:   0%|          | 0.00/639k [00:00<?, ?B/s]

validation.json: downloaded (0.6 MB)


imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

imgs.zip: downloaded (202.5 MB)


In [8]:
from pathlib import Path
from zipfile import ZipFile

archive = Path("/content/slake_source/imgs.zip")

with ZipFile(archive) as zf:
    names = zf.namelist()
    print("Files in archive:", len(names))
    print("First 10 paths:")
    for name in names[:10]:
        print(name)

Files in archive: 3235
First 10 paths:
imgs/
__MACOSX/._imgs
imgs/xmlab29/
imgs/xmlab198/
imgs/xmlab508/
imgs/xmlab395/
imgs/xmlab16/
imgs/xmlab537/
imgs/xmlab361/
imgs/xmlab153/


In [9]:
from pathlib import Path
from zipfile import ZipFile

archive = Path("/content/slake_source/imgs.zip")
extract_dir = Path("/content/slake_source")

with ZipFile(archive) as zf:
    image_files = [
        name for name in zf.namelist()
        if name.startswith("imgs/") and not name.endswith("/")
    ]
    zf.extractall(extract_dir, members=image_files)

image_root = extract_dir / "imgs"
jpg_files = list(image_root.rglob("*.jpg"))

print("Extracted image files:", len(jpg_files))
print("Image folder exists:", image_root.exists())
print("Example image:", jpg_files[0] if jpg_files else "None")
print("Expected sample exists:", (image_root / "xmlab1" / "source.jpg").exists())

Extracted image files: 642
Image folder exists: True
Example image: /content/slake_source/imgs/xmlab45/source.jpg
Expected sample exists: True


In [10]:
from pathlib import Path
from zipfile import ZipFile

archive = Path("/content/slake_source/imgs.zip")
extract_dir = Path("/content/slake_source")

with ZipFile(archive) as zf:
    image_files = [
        name for name in zf.namelist()
        if name.startswith("imgs/") and not name.endswith("/")
    ]
    zf.extractall(extract_dir, members=image_files)

image_root = extract_dir / "imgs"
jpg_files = list(image_root.rglob("*.jpg"))

print("Extracted image files:", len(jpg_files))
print("Image folder exists:", image_root.exists())
print("Example image:", jpg_files[0] if jpg_files else "None")
print("Expected sample exists:", (image_root / "xmlab1" / "source.jpg").exists())

Extracted image files: 642
Image folder exists: True
Example image: /content/slake_source/imgs/xmlab45/source.jpg
Expected sample exists: True


In [11]:
import json
from pathlib import Path

data_dir = Path("/content/slake_source")

with open(data_dir / "train.json", encoding="utf-8") as f:
    train_data = json.load(f)

with open(data_dir / "validation.json", encoding="utf-8") as f:
    validation_data = json.load(f)

train_english = [x for x in train_data if x.get("q_lang") == "en"]
validation_english = [x for x in validation_data if x.get("q_lang") == "en"]

print("Training examples:", len(train_data))
print("English training questions:", len(train_english))
print("Validation examples:", len(validation_data))
print("English validation questions:", len(validation_english))
print("\nExample English question:")
print(json.dumps(train_english[0], indent=2, ensure_ascii=False))

Training examples: 9835
English training questions: 4919
Validation examples: 2099
English validation questions: 1053

Example English question:
{
  "img_id": 1,
  "img_name": "xmlab1/source.jpg",
  "question": "What modality is used to take this image?",
  "answer": "MRI",
  "q_lang": "en",
  "location": "Abdomen",
  "modality": "MRI",
  "answer_type": "OPEN",
  "base_type": "vqa",
  "content_type": "Modality",
  "triple": [
    "vhead",
    "_",
    "_"
  ],
  "qid": 0
}


In [12]:
import json
from pathlib import Path

image_folder = Path("/content/slake_source/imgs/xmlab1")

for filename in ["question.json", "detection.json"]:
    path = image_folder / filename
    print(f"\n--- {filename} ---")
    with open(path, encoding="utf-8") as f:
        content = json.load(f)
    print(json.dumps(content, indent=2, ensure_ascii=False)[:5000])


--- question.json ---
[
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "question": "What modality is used to take this image?",
    "answer": "MRI",
    "q_lang": "en",
    "location": "Abdomen",
    "modality": "MRI",
    "answer_type": "OPEN",
    "base_type": "vqa",
    "triple": [
      "vhead",
      "_",
      "_"
    ]
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "question": "Which part of the body does this image belong to?",
    "answer": "Abdomen",
    "q_lang": "en",
    "location": "Abdomen",
    "modality": "MRI",
    "answer_type": "OPEN",
    "base_type": "vqa",
    "triple": [
      "vhead",
      "_",
      "_"
    ]
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "question": "What is the mr weighting in this image?",
    "answer": "T2",
    "q_lang": "en",
    "location": "Abdomen",
    "modality": "MRI",
    "answer_type": "OPEN",
    "base_type": "vqa",
    "triple": [
      "vhead",
      "_",
      "_"
    ]

In [13]:
import json
from pathlib import Path

folder = Path("/content/slake_source/imgs/xmlab1")

with open(folder / "detection.json", encoding="utf-8") as f:
    print("Detection annotations:")
    print(json.dumps(json.load(f), indent=2))

with open(folder / "question.json", encoding="utf-8") as f:
    questions = json.load(f)

print("\nEnglish questions for this image:")
for i, item in enumerate(questions):
    if item.get("q_lang") == "en":
        print(
            f"QID {i}: {item['question']} -> {item['answer']} "
            f"[{item.get('content_type', 'no content_type')}]"
        )

Detection annotations:
[
  {
    "Liver": [
      54.0,
      106.0,
      30.0,
      31.0
    ]
  }
]

English questions for this image:
QID 0: What modality is used to take this image? -> MRI [no content_type]
QID 1: Which part of the body does this image belong to? -> Abdomen [no content_type]
QID 2: What is the mr weighting in this image? -> T2 [no content_type]
QID 3: Does the picture contain liver? -> Yes [no content_type]
QID 4: Does the picture contain kidney? -> No [no content_type]
QID 5: Does the picture contain spleen? -> No [no content_type]
QID 6: Where is the liver? -> Right [no content_type]


In [14]:
import json
import re
from pathlib import Path
from PIL import Image

data_dir = Path("/content/slake_source")
image_root = data_dir / "imgs"

with open(data_dir / "train.json", encoding="utf-8") as f:
    train_data = json.load(f)

with open(data_dir / "validation.json", encoding="utf-8") as f:
    validation_data = json.load(f)

def build_records(source_data):
    records = []
    detection_cache = {}
    image_size_cache = {}

    for item in source_data:
        if item.get("q_lang") != "en":
            continue

        img_name = item["img_name"]
        image_path = image_root / img_name
        image_folder = image_path.parent

        if img_name not in detection_cache:
            detection_path = image_folder / "detection.json"
            if detection_path.exists():
                with open(detection_path, encoding="utf-8") as f:
                    annotations = json.load(f)
                detection_cache[img_name] = {
                    organ: box
                    for annotation in annotations
                    for organ, box in annotation.items()
                }
            else:
                detection_cache[img_name] = {}

        if img_name not in image_size_cache and image_path.exists():
            with Image.open(image_path) as image:
                image_size_cache[img_name] = image.size

        question = item["question"]
        answer = item["answer"]
        question_lower = question.lower()
        answer_lower = str(answer).strip().lower()
        boxes = detection_cache[img_name]

        matched_organ = None
        matched_box = None

        # Link a box only when the question names that annotated organ
        # and the answer indicates that the organ is present or located.
        for organ in sorted(boxes, key=len, reverse=True):
            pattern = rf"(?<!\w){re.escape(organ.lower())}(?!\w)"
            if re.search(pattern, question_lower):
                is_positive = (
                    answer_lower == "yes"
                    or question_lower.startswith(("where is", "where are"))
                )
                if is_positive:
                    matched_organ = organ
                    matched_box = [float(v) for v in boxes[organ]]
                    break

        grounding_status = "not_required"
        region = None

        if matched_box is not None:
            width, height = image_size_cache[img_name]
            x, y, box_width, box_height = matched_box
            bbox_xyxy_normalized = [
                round(x / width, 4),
                round(y / height, 4),
                round((x + box_width) / width, 4),
                round((y + box_height) / height, 4),
            ]
            region = {
                "organ": matched_organ,
                "bbox_xyxy_normalized": bbox_xyxy_normalized,
            }
            grounding_status = "grounded"
        elif answer_lower == "no" and "contain" in question_lower:
            grounding_status = "no_positive_region"

        target_text = json.dumps(
            {"answer": answer, "region": region},
            ensure_ascii=False,
            separators=(",", ":"),
        )

        records.append({
            "img_id": item["img_id"],
            "img_name": img_name,  # relative path; no runtime-specific path
            "qid": item["qid"],
            "question": question,
            "answer": answer,
            "bbox_xywh_pixels": matched_box,
            "grounded_organ": matched_organ,
            "grounding_status": grounding_status,
            "target_text": target_text,
        })

    return records

training_records = build_records(train_data)
validation_records = build_records(validation_data)

print("Training records:", len(training_records))
print("Training records with boxes:", sum(r["bbox_xywh_pixels"] is not None for r in training_records))
print("Validation records:", len(validation_records))
print("Validation records with boxes:", sum(r["bbox_xywh_pixels"] is not None for r in validation_records))
print("Example without a box:", next(r["target_text"] for r in training_records if r["bbox_xywh_pixels"] is None))
print("Example with a box:", next(r["target_text"] for r in training_records if r["bbox_xywh_pixels"] is not None))

Training records: 4919
Training records with boxes: 370
Validation records: 1053
Validation records with boxes: 69
Example without a box: {"answer":"MRI","region":null}
Example with a box: {"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.2109,0.4141,0.3281,0.5352]}}


In [15]:
from pathlib import Path
from PIL import Image

def check_boxes(records, split_name):
    invalid = []
    missing_images = []

    for record in records:
        box = record["bbox_xywh_pixels"]
        if box is None:
            continue

        image_path = Path("/content/slake_source/imgs") / record["img_name"]
        if not image_path.exists():
            missing_images.append(record["img_name"])
            continue

        with Image.open(image_path) as image:
            width, height = image.size

        x, y, box_width, box_height = box
        if (
            x < 0 or y < 0 or box_width <= 0 or box_height <= 0
            or x + box_width > width or y + box_height > height
        ):
            invalid.append((record["qid"], box, (width, height)))

    matched = sum(r["bbox_xywh_pixels"] is not None for r in records)
    print(f"{split_name}: {matched} matched boxes")
    print(f"{split_name}: invalid boxes =", len(invalid))
    print(f"{split_name}: missing images =", len(missing_images))
    if invalid:
        print("First invalid example:", invalid[0])
    if missing_images:
        print("First missing image:", missing_images[0])

check_boxes(training_records, "Training")
check_boxes(validation_records, "Validation")

Training: 370 matched boxes
Training: invalid boxes = 0
Training: missing images = 0
Validation: 69 matched boxes
Validation: invalid boxes = 0
Validation: missing images = 0


In [16]:
import json
from pathlib import Path

output_dir = Path("/content/slake_grounded_vqa_upload")
output_dir.mkdir(parents=True, exist_ok=True)

metadata_fields = [
    "q_lang",
    "location",
    "modality",
    "answer_type",
    "base_type",
    "content_type",
    "triple",
]

def save_jsonl(records, source_data, filename):
    source_by_qid = {item["qid"]: item for item in source_data}
    output_path = output_dir / filename

    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            row = dict(record)
            original = source_by_qid[record["qid"]]

            for field in metadata_fields:
                if field in original:
                    row[field] = original[field]

            # Keep only the relative filename; do not save Colab paths.
            row.pop("image_path", None)
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"{filename}: {len(records)} rows, {output_path.stat().st_size / 1024:.1f} KB")
    print("First row:", json.dumps(records[0], ensure_ascii=False))

save_jsonl(training_records, train_data, "train.jsonl")
save_jsonl(validation_records, validation_data, "validation.jsonl")

train.jsonl: 4919 rows, 2193.2 KB
First row: {"img_id": 1, "img_name": "xmlab1/source.jpg", "qid": 0, "question": "What modality is used to take this image?", "answer": "MRI", "bbox_xywh_pixels": null, "grounded_organ": null, "grounding_status": "not_required", "target_text": "{\"answer\":\"MRI\",\"region\":null}"}
validation.jsonl: 1053 rows, 473.1 KB
First row: {"img_id": 0, "img_name": "xmlab0/source.jpg", "qid": 9835, "question": "What modality is used to take this image?", "answer": "MRI", "bbox_xywh_pixels": null, "grounded_organ": null, "grounding_status": "not_required", "target_text": "{\"answer\":\"MRI\",\"region\":null}"}


In [1]:
from huggingface_hub import HfApi

api = HfApi()
username = api.whoami()["name"]
repo_id = f"{username}/slake-grounded-vqa-gemma4-e2b"

api.create_repo(
    repo_id=repo_id,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

print("Dataset repository created:")
print(f"https://huggingface.co/datasets/{repo_id}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


LocalTokenNotFoundError: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [2]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

print("Signed in as:", HfApi().whoami()["name"])

Signed in as: Thushi2003


In [3]:
from huggingface_hub import HfApi

api = HfApi()
username = api.whoami()["name"]
repo_id = f"{username}/slake-grounded-vqa-gemma4-e2b"

api.create_repo(
    repo_id=repo_id,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

print("Dataset repository:")
print(f"https://huggingface.co/datasets/{repo_id}")

Dataset repository:
https://huggingface.co/datasets/Thushi2003/slake-grounded-vqa-gemma4-e2b


In [4]:
from pathlib import Path
from huggingface_hub import HfApi

output_dir = Path("/content/slake_grounded_vqa_upload")

readme = """---
license: cc-by-4.0
task_categories:
- visual-question-answering
language:
- en
tags:
- medical
- visual-question-answering
pretty_name: SLAKE English Grounded VQA Annotations
---

# SLAKE English Grounded VQA Annotations

Derived English annotations for the SLAKE medical visual question-answering dataset.

## Files

- `train.jsonl`: 4,919 question records
- `validation.jsonl`: 1,053 question records

Each row contains the question and answer, the relative source image filename, SLAKE metadata and knowledge-graph triple, grounding status, and a structured answer-plus-region target. Bounding boxes are stored as pixel `xywh` values and normalized `xyxy` values in `target_text`.

A box is included when the question names an organ with a source detection annotation and asks whether it is present or where it is located. Image files are not included. Use `img_name` to locate each image in the original [BoKelvin/SLAKE dataset](https://huggingface.co/datasets/BoKelvin/SLAKE), and follow its license and attribution terms.

## Source and citation

Source dataset: [BoKelvin/SLAKE](https://huggingface.co/datasets/BoKelvin/SLAKE)  
Project page: https://www.med-vqa.com/slake/  
Paper: [SLAKE: A Semantically-Labeled Knowledge-Enhanced Dataset for Medical Visual Question Answering](https://arxiv.org/abs/2102.09542)

These files are prepared for research and educational use. They are not for clinical diagnosis.
"""

(output_dir / "README.md").write_text(readme, encoding="utf-8")

api = HfApi()
username = api.whoami()["name"]
repo_id = f"{username}/slake-grounded-vqa-gemma4-e2b"

result = api.upload_folder(
    repo_id=repo_id,
    repo_type="dataset",
    folder_path=str(output_dir),
    commit_message="Upload SLAKE English grounded VQA annotations",
)

print("Upload completed.")
print("Dataset page:", f"https://huggingface.co/datasets/{repo_id}")
print("Upload result:", result)

FileNotFoundError: [Errno 2] No such file or directory: '/content/slake_grounded_vqa_upload/README.md'

In [5]:
from pathlib import Path

upload_dir = Path("/content/slake_grounded_vqa_upload")

print("Upload folder exists:", upload_dir.exists())
if upload_dir.exists():
    print("Files in upload folder:", [p.name for p in upload_dir.iterdir()])

print("Training records in memory:", "training_records" in globals())
print("Validation records in memory:", "validation_records" in globals())
print("SLAKE source folder exists:", Path("/content/slake_source").exists())

Upload folder exists: False
Training records in memory: False
Validation records in memory: False
SLAKE source folder exists: False


In [6]:
from pathlib import Path
from zipfile import ZipFile
from huggingface_hub import hf_hub_download

data_dir = Path("/content/slake_source")
data_dir.mkdir(parents=True, exist_ok=True)

for filename in ["train.json", "validation.json", "imgs.zip"]:
    hf_hub_download(
        repo_id="BoKelvin/SLAKE",
        repo_type="dataset",
        filename=filename,
        local_dir=str(data_dir),
    )

archive = data_dir / "imgs.zip"
with ZipFile(archive) as zf:
    image_files = [
        name for name in zf.namelist()
        if name.startswith("imgs/") and not name.endswith("/")
    ]
    zf.extractall(data_dir, members=image_files)

print("Train JSON exists:", (data_dir / "train.json").exists())
print("Validation JSON exists:", (data_dir / "validation.json").exists())
print("Image files extracted:", len(list((data_dir / "imgs").rglob("*.jpg"))))

train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

validation.json:   0%|          | 0.00/639k [00:00<?, ?B/s]

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

Train JSON exists: True
Validation JSON exists: True
Image files extracted: 642


In [7]:
import json
import re
from pathlib import Path
from PIL import Image

data_dir = Path("/content/slake_source")
image_root = data_dir / "imgs"
output_dir = Path("/content/slake_grounded_vqa_upload")
output_dir.mkdir(parents=True, exist_ok=True)

with open(data_dir / "train.json", encoding="utf-8") as f:
    train_data = json.load(f)

with open(data_dir / "validation.json", encoding="utf-8") as f:
    validation_data = json.load(f)

def build_records(source_data):
    records = []
    detection_cache = {}
    size_cache = {}

    for item in source_data:
        if item.get("q_lang") != "en":
            continue

        img_name = item["img_name"]
        image_path = image_root / img_name

        if img_name not in detection_cache:
            detection_path = image_path.parent / "detection.json"
            with open(detection_path, encoding="utf-8") as f:
                annotations = json.load(f)
            detection_cache[img_name] = {
                organ: box
                for annotation in annotations
                for organ, box in annotation.items()
            }

        question = item["question"]
        answer = item["answer"]
        question_lower = question.lower()
        answer_lower = str(answer).strip().lower()

        matched_organ = None
        matched_box = None

        for organ in sorted(detection_cache[img_name], key=len, reverse=True):
            pattern = rf"(?<!\w){re.escape(organ.lower())}(?!\w)"
            if re.search(pattern, question_lower):
                if (
                    answer_lower == "yes"
                    or question_lower.startswith(("where is", "where are"))
                ):
                    matched_organ = organ
                    matched_box = [float(v) for v in detection_cache[img_name][organ]]
                    break

        region = None
        grounding_status = "not_required"

        if matched_box is not None:
            if img_name not in size_cache:
                with Image.open(image_path) as image:
                    size_cache[img_name] = image.size

            width, height = size_cache[img_name]
            x, y, box_width, box_height = matched_box
            region = {
                "organ": matched_organ,
                "bbox_xyxy_normalized": [
                    round(x / width, 4),
                    round(y / height, 4),
                    round((x + box_width) / width, 4),
                    round((y + box_height) / height, 4),
                ],
            }
            grounding_status = "grounded"
        elif answer_lower == "no" and "contain" in question_lower:
            grounding_status = "no_positive_region"

        target_text = json.dumps(
            {"answer": answer, "region": region},
            ensure_ascii=False,
            separators=(",", ":"),
        )

        row = {
            "img_id": item["img_id"],
            "img_name": img_name,
            "qid": item["qid"],
            "question": question,
            "answer": answer,
            "bbox_xywh_pixels": matched_box,
            "grounded_organ": matched_organ,
            "grounding_status": grounding_status,
            "target_text": target_text,
        }

        for field in [
            "q_lang", "location", "modality", "answer_type",
            "base_type", "content_type", "triple",
        ]:
            if field in item:
                row[field] = item[field]

        records.append(row)

    return records

training_records = build_records(train_data)
validation_records = build_records(validation_data)

for filename, records in [
    ("train.jsonl", training_records),
    ("validation.jsonl", validation_records),
]:
    with open(output_dir / filename, "w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Training rows:", len(training_records))
print("Training rows with boxes:", sum(r["bbox_xywh_pixels"] is not None for r in training_records))
print("Validation rows:", len(validation_records))
print("Validation rows with boxes:", sum(r["bbox_xywh_pixels"] is not None for r in validation_records))
print("Files saved:", [p.name for p in output_dir.iterdir()])

Training rows: 4919
Training rows with boxes: 370
Validation rows: 1053
Validation rows with boxes: 69
Files saved: ['train.jsonl', 'validation.jsonl']


In [8]:
from pathlib import Path
from getpass import getpass
from huggingface_hub import HfApi, get_token, login

output_dir = Path("/content/slake_grounded_vqa_upload")
repo_id = "Thushi2003/slake-grounded-vqa-gemma4-e2b"

if get_token() is None:
    hf_token = getpass("Paste your Hugging Face write token: ")
    login(token=hf_token, add_to_git_credential=False)
    del hf_token

readme = """---
license: cc-by-4.0
task_categories:
- visual-question-answering
language:
- en
tags:
- medical
- visual-question-answering
pretty_name: SLAKE English Grounded VQA Annotations
---

# SLAKE English Grounded VQA Annotations

Derived English annotations for the SLAKE medical visual question-answering dataset.

## Files

- `train.jsonl`: 4,919 question records
- `validation.jsonl`: 1,053 question records

Each row includes the question and answer, relative source image filename, SLAKE metadata and knowledge-graph triple, grounding status, and a structured answer-plus-region target. Bounding boxes are stored as pixel `xywh` values and normalized `xyxy` values in `target_text`.

A box is included when the question names an organ with a source detection annotation and asks whether it is present or where it is located. Image files are not included. Use `img_name` to locate each image in the original [BoKelvin/SLAKE dataset](https://huggingface.co/datasets/BoKelvin/SLAKE), and follow its license and attribution terms.

## Source and citation

Source dataset: [BoKelvin/SLAKE](https://huggingface.co/datasets/BoKelvin/SLAKE)  
Project page: https://www.med-vqa.com/slake/  
Paper: [SLAKE: A Semantically-Labeled Knowledge-Enhanced Dataset for Medical Visual Question Answering](https://arxiv.org/abs/2102.09542)

These files are prepared for research and educational use. They are not for clinical diagnosis.
"""

(output_dir / "README.md").write_text(readme, encoding="utf-8")

api = HfApi()
api.upload_folder(
    repo_id=repo_id,
    repo_type="dataset",
    folder_path=str(output_dir),
    commit_message="Upload SLAKE English grounded VQA annotations",
)

print("Upload completed.")
print("Dataset page:", f"https://huggingface.co/datasets/{repo_id}")
print("Uploaded files:", [p.name for p in output_dir.iterdir()])

Upload completed.
Dataset page: https://huggingface.co/datasets/Thushi2003/slake-grounded-vqa-gemma4-e2b
Uploaded files: ['train.jsonl', 'README.md', 'validation.jsonl']


In [9]:
from huggingface_hub import HfApi

repo_id = "Thushi2003/slake-grounded-vqa-gemma4-e2b"
files = HfApi().list_repo_files(repo_id=repo_id, repo_type="dataset")

print("Files on Hugging Face:")
for filename in files:
    print("-", filename)

Files on Hugging Face:
- .gitattributes
- README.md
- train.jsonl
- validation.jsonl


In [10]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for name in ["model", "processor", "trainer", "collate_vqa", "training_examples"]:
    print(f"{name} available:", name in globals())

CUDA available: True
GPU: Tesla T4
model available: False
processor available: False
trainer available: False
collate_vqa available: False
training_examples available: False


In [11]:
from pathlib import Path

paths = [
    Path("/content/slake_source/imgs/xmlab1/source.jpg"),
    Path("/content/slake_grounded_vqa_upload/train.jsonl"),
    Path("/content/slake_grounded_vqa_upload/validation.jsonl"),
]

for path in paths:
    print(f"{path}: {path.exists()}")

/content/slake_source/imgs/xmlab1/source.jpg: True
/content/slake_grounded_vqa_upload/train.jsonl: True
/content/slake_grounded_vqa_upload/validation.jsonl: True


In [12]:
import json
from pathlib import Path

data_dir = Path("/content/slake_grounded_vqa_upload")
image_root = Path("/content/slake_source/imgs")

def load_examples(filename):
    examples = []
    with open(data_dir / filename, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            row["image_path"] = str(image_root / row["img_name"])
            examples.append(row)
    return examples

training_examples = load_examples("train.jsonl")
validation_examples = load_examples("validation.jsonl")

missing = [
    row["image_path"]
    for row in training_examples + validation_examples
    if not Path(row["image_path"]).exists()
]

print("Training examples:", len(training_examples))
print("Validation examples:", len(validation_examples))
print("Missing image files:", len(missing))
print("Example:", training_examples[0])

Training examples: 4919
Validation examples: 1053
Missing image files: 0
Example: {'img_id': 1, 'img_name': 'xmlab1/source.jpg', 'qid': 0, 'question': 'What modality is used to take this image?', 'answer': 'MRI', 'bbox_xywh_pixels': None, 'grounded_organ': None, 'grounding_status': 'not_required', 'target_text': '{"answer":"MRI","region":null}', 'q_lang': 'en', 'location': 'Abdomen', 'modality': 'MRI', 'answer_type': 'OPEN', 'base_type': 'vqa', 'content_type': 'Modality', 'triple': ['vhead', '_', '_'], 'image_path': '/content/slake_source/imgs/xmlab1/source.jpg'}


In [13]:
import torch
from getpass import getpass
from huggingface_hub import get_token, login
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

if get_token() is None:
    hf_token = getpass("Paste your Hugging Face token: ")
    login(token=hf_token, add_to_git_credential=False)
    del hf_token

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("CUDA:", torch.cuda.is_available())
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [14]:
import bitsandbytes
import transformers
from transformers.utils.import_utils import is_bitsandbytes_available

print("bitsandbytes version:", bitsandbytes.__version__)
print("Transformers version:", transformers.__version__)
print("bitsandbytes detected before clearing cache:", is_bitsandbytes_available())

is_bitsandbytes_available.cache_clear()

print("bitsandbytes detected after clearing cache:", is_bitsandbytes_available())

ModuleNotFoundError: No module named 'bitsandbytes'

In [15]:
%pip install -q bitsandbytes==0.50.2

import bitsandbytes
print("bitsandbytes version:", bitsandbytes.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00:00:0100:01
bitsandbytes version: 0.50.2


In [16]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from transformers.utils.import_utils import is_bitsandbytes_available

is_bitsandbytes_available.cache_clear()
print("Transformers detects bitsandbytes:", is_bitsandbytes_available())

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

if "processor" not in globals():
    processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

Transformers detects bitsandbytes: True


model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model: Gemma4ForConditionalGeneration
Processor: Gemma4Processor
GPU memory used (GB): 6.28


In [17]:
import torch
from peft import LoraConfig, get_peft_model

target_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj"}

target_modules = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and name.split(".")[-1] in target_suffixes
]

print("LoRA targets found:", len(target_modules))

if len(target_modules) != 100:
    raise ValueError(
        f"Expected the previous 100 language-model targets, found {len(target_modules)}."
    )

# Freeze the quantized base model; train only the LoRA adapters.
for parameter in model.parameters():
    parameter.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

LoRA targets found: 100
trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525


In [18]:
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

data_dir = Path("/content/slake_source")
data_dir.mkdir(parents=True, exist_ok=True)

test_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    repo_type="dataset",
    filename="test.json",
    local_dir=str(data_dir),
)

with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)

test_english = [item for item in test_data if item.get("q_lang") == "en"]

print("All test questions:", len(test_data))
print("English test questions:", len(test_english))
print("Example:", test_english[0])

test.json:   0%|          | 0.00/636k [00:00<?, ?B/s]

All test questions: 2094
English test questions: 1061
Example: {'img_id': 102, 'img_name': 'xmlab102/source.jpg', 'question': 'What modality is used to take this image?', 'answer': 'CT', 'q_lang': 'en', 'location': 'Lung', 'modality': 'CT', 'answer_type': 'OPEN', 'base_type': 'vqa', 'triple': ['vhead', '_', '_'], 'qid': 11934, 'content_type': 'Modality'}


In [19]:
import json
import re
from pathlib import Path
from PIL import Image

image_root = Path("/content/slake_source/imgs")
output_dir = Path("/content/slake_grounded_vqa_upload")
test_records = []
detection_cache = {}
size_cache = {}
missing_images = []
invalid_boxes = []

for item in test_english:
    img_name = item["img_name"]
    image_path = image_root / img_name

    if not image_path.exists():
        missing_images.append(img_name)
        continue

    if img_name not in detection_cache:
        detection_path = image_path.parent / "detection.json"
        with open(detection_path, encoding="utf-8") as f:
            annotations = json.load(f)
        detection_cache[img_name] = {
            organ: box
            for annotation in annotations
            for organ, box in annotation.items()
        }

    question = item["question"]
    answer = item["answer"]
    question_lower = question.lower()
    answer_lower = str(answer).strip().lower()

    matched_organ = None
    matched_box = None

    for organ in sorted(detection_cache[img_name], key=len, reverse=True):
        pattern = rf"(?<!\w){re.escape(organ.lower())}(?!\w)"
        if re.search(pattern, question_lower) and (
            answer_lower == "yes"
            or question_lower.startswith(("where is", "where are"))
        ):
            matched_organ = organ
            matched_box = [float(v) for v in detection_cache[img_name][organ]]
            break

    region = None
    grounding_status = "not_required"

    if matched_box is not None:
        if img_name not in size_cache:
            with Image.open(image_path) as image:
                size_cache[img_name] = image.size

        width, height = size_cache[img_name]
        x, y, box_width, box_height = matched_box

        if (
            x < 0 or y < 0 or box_width <= 0 or box_height <= 0
            or x + box_width > width or y + box_height > height
        ):
            invalid_boxes.append((item["qid"], matched_box, (width, height)))
            matched_box = None
            matched_organ = None
        else:
            region = {
                "organ": matched_organ,
                "bbox_xyxy_normalized": [
                    round(x / width, 4),
                    round(y / height, 4),
                    round((x + box_width) / width, 4),
                    round((y + box_height) / height, 4),
                ],
            }
            grounding_status = "grounded"

    if matched_box is None and answer_lower == "no" and "contain" in question_lower:
        grounding_status = "no_positive_region"

    row = {
        "img_id": item["img_id"],
        "img_name": img_name,
        "qid": item["qid"],
        "question": question,
        "answer": answer,
        "bbox_xywh_pixels": matched_box,
        "grounded_organ": matched_organ,
        "grounding_status": grounding_status,
        "target_text": json.dumps(
            {"answer": answer, "region": region},
            ensure_ascii=False,
            separators=(",", ":"),
        ),
    }

    for field in [
        "q_lang", "location", "modality", "answer_type",
        "base_type", "content_type", "triple",
    ]:
        if field in item:
            row[field] = item[field]

    test_records.append(row)

test_path = output_dir / "test.jsonl"
with open(test_path, "w", encoding="utf-8") as f:
    for row in test_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Test records saved:", len(test_records))
print("Test records with boxes:", sum(r["bbox_xywh_pixels"] is not None for r in test_records))
print("Invalid boxes:", len(invalid_boxes))
print("Missing images:", len(missing_images))
print("Saved file:", test_path)

Test records saved: 1061
Test records with boxes: 78
Invalid boxes: 0
Missing images: 0
Saved file: /content/slake_grounded_vqa_upload/test.jsonl


In [20]:
from pathlib import Path
from getpass import getpass
from huggingface_hub import HfApi, get_token, login

output_dir = Path("/content/slake_grounded_vqa_upload")
repo_id = "Thushi2003/slake-grounded-vqa-gemma4-e2b"

readme = """---
license: cc-by-4.0
task_categories:
- visual-question-answering
language:
- en
tags:
- medical
- visual-question-answering
pretty_name: SLAKE English Grounded VQA Annotations
configs:
- config_name: default
  data_files:
  - split: train
    path: train.jsonl
  - split: validation
    path: validation.jsonl
  - split: test
    path: test.jsonl
---

# SLAKE English Grounded VQA Annotations

Derived English annotations for the SLAKE medical visual question-answering dataset.

## Splits

- `train.jsonl`: 4,919 records; 370 with boxes
- `validation.jsonl`: 1,053 records; 69 with boxes
- `test.jsonl`: 1,061 records; 78 with boxes

Each row contains a question, answer, relative image filename, SLAKE metadata and knowledge-graph triple, grounding status, and structured answer-plus-region target. Images are not included. Use `img_name` to locate each image in the original [BoKelvin/SLAKE dataset](https://huggingface.co/datasets/BoKelvin/SLAKE).

## Source and citation

Project page: https://www.med-vqa.com/slake/  
Paper: [SLAKE: A Semantically-Labeled Knowledge-Enhanced Dataset for Medical Visual Question Answering](https://arxiv.org/abs/2102.09542)

These files are prepared for research and educational use. They are not for clinical diagnosis.
"""

(output_dir / "README.md").write_text(readme, encoding="utf-8")

if get_token() is None:
    hf_token = getpass("Paste your Hugging Face write token: ")
    login(token=hf_token, add_to_git_credential=False)
    del hf_token

api = HfApi()
api.upload_folder(
    repo_id=repo_id,
    repo_type="dataset",
    folder_path=str(output_dir),
    commit_message="Add English test split and configure all dataset splits",
)

print("Upload completed.")
print("Dataset page:", f"https://huggingface.co/datasets/{repo_id}")

Upload completed.
Dataset page: https://huggingface.co/datasets/Thushi2003/slake-grounded-vqa-gemma4-e2b


In [21]:
from huggingface_hub import HfApi

repo_id = "Thushi2003/slake-grounded-vqa-gemma4-e2b"
files = HfApi().list_repo_files(repo_id=repo_id, repo_type="dataset")

for filename in ["README.md", "train.jsonl", "validation.jsonl", "test.jsonl"]:
    print(f"{filename} uploaded:", filename in files)

README.md uploaded: True
train.jsonl uploaded: True
validation.jsonl uploaded: True
test.jsonl uploaded: True


In [22]:
import torch
from PIL import Image

def collate_one_vqa(example):
    with Image.open(example["image_path"]) as source_image:
        image = source_image.convert("RGB")

    user_message = {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": example["question"]},
        ],
    }

    full_messages = [
        user_message,
        {
            "role": "assistant",
            "content": [{"type": "text", "text": example["target_text"]}],
        },
    ]

    prompt = processor.apply_chat_template(
        [user_message],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    full = processor.apply_chat_template(
        full_messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    prompt_ids = prompt["input_ids"][0]
    full_ids = full["input_ids"][0]

    if not torch.equal(full_ids[:len(prompt_ids)], prompt_ids):
        raise ValueError("Prompt tokens do not match the start of the full example.")

    labels = full["input_ids"].clone()
    labels[:, :len(prompt_ids)] = -100

    batch = {
        key: full[key].to("cuda")
        for key in [
            "input_ids",
            "attention_mask",
            "mm_token_type_ids",
            "pixel_values",
            "image_position_ids",
        ]
        if key in full
    }
    batch["labels"] = labels.to("cuda")

    return batch

test_batch = collate_one_vqa(training_examples[0])

print("Batch keys:", list(test_batch.keys()))
for key, value in test_batch.items():
    print(f"{key} shape:", tuple(value.shape))

model.train()
model.zero_grad(set_to_none=True)

with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = model(**test_batch)

print("Loss:", outputs.loss.item())
outputs.loss.backward()

gradient_count = sum(
    parameter.grad is not None
    for parameter in model.parameters()
    if parameter.requires_grad
)
print("LoRA parameters with gradients:", gradient_count)

model.zero_grad(set_to_none=True)

Batch keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'labels']
input_ids shape: (1, 287)
attention_mask shape: (1, 287)
mm_token_type_ids shape: (1, 287)
pixel_values shape: (1, 2520, 768)
image_position_ids shape: (1, 2520, 2)
labels shape: (1, 287)
Loss: 6.628909587860107
LoRA parameters with gradients: 200


In [23]:
from transformers import Trainer, TrainingArguments

def collate_vqa(examples):
    if len(examples) != 1:
        raise ValueError("This setup currently expects batch size 1.")
    return collate_one_vqa(examples[0])

training_args = TrainingArguments(
    output_dir="/content/gemma4_slake_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_examples,
    data_collator=collate_vqa,
)

print("Trainer recreated successfully.")
print("Training rows:", len(training_examples))
print("Training has not started.")

Trainer recreated successfully.
Training rows: 4919
Training has not started.


In [24]:
trainer.args.max_steps = 5
trainer.args.logging_steps = 1

train_result = trainer.train()

print("Pilot completed.")
print("Optimizer steps:", trainer.state.global_step)
print("Training loss:", train_result.training_loss)

Step,Training Loss
1,5.674556
2,4.954288
3,5.346941
4,5.193663
5,5.771874


Pilot completed.
Optimizer steps: 5
Training loss: 5.388264465332031


In [25]:
from transformers import Trainer, TrainingArguments

model = trainer.model
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/gemma4_slake_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=-1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_examples,
    data_collator=collate_vqa,
)

print("Starting full training...")
train_result = trainer.train()

print("Training finished.")
print("Optimizer steps:", trainer.state.global_step)
print("Training loss:", train_result.training_loss)

Starting full training...


Step,Training Loss
20,2.845422
40,0.497825
60,0.462760
80,0.397221
100,0.288471
120,0.273597
140,0.242787
160,0.211365
180,0.214468
200,0.282207


Training finished.
Optimizer steps: 1230
Training loss: 0.2203147779150707


In [1]:
from PIL import Image
import torch

model = trainer.model
model.eval()

sample = validation_examples[0]
image = Image.open(sample["image_path"]).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": sample["question"]},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

# Keep only inputs the model uses; omit training-only assistant masks.
model_input_keys = {
    "input_ids",
    "attention_mask",
    "mm_token_type_ids",
    "pixel_values",
    "image_position_ids",
}
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
    if key in model_input_keys
}

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
prediction = processor.decode(new_tokens, skip_special_tokens=True)

print("Question:", sample["question"])
print("Expected answer:", sample["answer"])
print("Model output:", prediction)

ModuleNotFoundError: No module named 'PIL'

In [2]:
%pip install -q Pillow

from PIL import Image
print("Pillow installed successfully.")

Note: you may need to restart the kernel to use updated packages.
Pillow installed successfully.


In [3]:
from PIL import Image
import torch

model = trainer.model
model.eval()

sample = validation_examples[0]
image = Image.open(sample["image_path"]).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": sample["question"]},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

# Keep only inputs the model uses; omit training-only assistant masks.
model_input_keys = {
    "input_ids",
    "attention_mask",
    "mm_token_type_ids",
    "pixel_values",
    "image_position_ids",
}
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
    if key in model_input_keys
}

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
prediction = processor.decode(new_tokens, skip_special_tokens=True)

print("Question:", sample["question"])
print("Expected answer:", sample["answer"])
print("Model output:", prediction)

ModuleNotFoundError: No module named 'torch'

In [4]:
import sys
import importlib.util

print("Python:", sys.executable)
print("PyTorch installed:", importlib.util.find_spec("torch") is not None)
print("Trainer still in memory:", "trainer" in globals())

Python: c:\Users\ThushiharanPirabahar\AppData\Local\Programs\Python\Python314\python.exe
PyTorch installed: False
Trainer still in memory: False


In [1]:
import torch

print("Python:", __import__("sys").executable)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Trainer still in memory:", "trainer" in globals())

Python: /usr/bin/python3
CUDA available: True
GPU: Tesla T4
Trainer still in memory: False


In [2]:
from pathlib import Path

checkpoint_dir = Path("/content/gemma4_slake_checkpoints")
print("Checkpoint folder exists:", checkpoint_dir.exists())

if checkpoint_dir.exists():
    checkpoints = sorted(
        checkpoint_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1])
    )
    print("Checkpoint folders:", [p.name for p in checkpoints])

    for folder in checkpoints:
        print(f"\n{folder.name} files:")
        print([p.name for p in folder.iterdir()])

Checkpoint folder exists: False


In [3]:
from google.colab import drive
drive.mount("/content/drive")

MessageError: User cancelled dfs_ephemeral authorization

In [4]:
from getpass import getpass
from huggingface_hub import login, HfApi

hf_token = getpass("Paste your Hugging Face write token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

print("Signed in as:", HfApi().whoami()["name"])

Signed in as: Thushi2003


In [5]:
from huggingface_hub import HfApi

api = HfApi()
model_repo_id = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

repo_url = api.create_repo(
    repo_id=model_repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
)

print("Model repository:", repo_url)

Model repository: https://huggingface.co/Thushi2003/slake-grounded-vqa-gemma4-e2b-lora


In [6]:
from pathlib import Path

print("training_examples in memory:", "training_examples" in globals())
print("validation_examples in memory:", "validation_examples" in globals())
print("model in memory:", "model" in globals())
print("processor in memory:", "processor" in globals())

print("\nFiles in /content:")
for path in Path("/content").iterdir():
    print(path.name)

print("\nTraining JSONL exists:",
      Path("/content/slake_grounded_vqa_upload/train.jsonl").exists())
print("Validation JSONL exists:",
      Path("/content/slake_grounded_vqa_upload/validation.jsonl").exists())

training_examples in memory: False
validation_examples in memory: False
model in memory: False
processor in memory: False

Files in /content:
.config
sample_data

Training JSONL exists: False
Validation JSONL exists: False


In [7]:
import sys
import torch
import transformers
import peft
import importlib.util

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes installed:", importlib.util.find_spec("bitsandbytes") is not None)

Python: /usr/bin/python3
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Transformers: 5.18.0
PEFT: 0.21.1
bitsandbytes installed: False


In [8]:
%pip install -q bitsandbytes==0.50.2

import bitsandbytes
from transformers.utils.import_utils import is_bitsandbytes_available

is_bitsandbytes_available.cache_clear()

print("bitsandbytes:", bitsandbytes.__version__)
print("Transformers detects bitsandbytes:", is_bitsandbytes_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.2 MB/s eta 0:00:0000:01:00:01
bitsandbytes: 0.50.2
Transformers detects bitsandbytes: True


In [9]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import json
import zipfile

# Download and extract SLAKE images
images_zip = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    repo_type="dataset",
    filename="imgs.zip",
)

image_extract_dir = Path("/content/slake_source")
image_extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(images_zip, "r") as archive:
    archive.extractall(image_extract_dir)

image_root = image_extract_dir / "imgs"

# Download your prepared annotation splits and rebuild image paths
dataset_repo = "Thushi2003/slake-grounded-vqa-gemma4-e2b"
restored_splits = {}

for split in ["train", "validation", "test"]:
    jsonl_path = hf_hub_download(
        repo_id=dataset_repo,
        repo_type="dataset",
        filename=f"{split}.jsonl",
    )

    with open(jsonl_path, "r", encoding="utf-8") as file:
        records = [json.loads(line) for line in file if line.strip()]

    for record in records:
        record["image_path"] = str(image_root / record["img_name"])

    restored_splits[split] = records

    missing = sum(
        not Path(record["image_path"]).is_file()
        for record in records
    )
    print(f"{split}: {len(records)} records; missing images: {missing}")

print("Example training record:", restored_splits["train"][0])

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

train.jsonl:   0%|          | 0.00/2.25M [00:00<?, ?B/s]

train: 4919 records; missing images: 0


validation.jsonl:   0%|          | 0.00/484k [00:00<?, ?B/s]

validation: 1053 records; missing images: 0


test.jsonl:   0%|          | 0.00/488k [00:00<?, ?B/s]

test: 1061 records; missing images: 0
Example training record: {'img_id': 1, 'img_name': 'xmlab1/source.jpg', 'qid': 0, 'question': 'What modality is used to take this image?', 'answer': 'MRI', 'bbox_xywh_pixels': None, 'grounded_organ': None, 'grounding_status': 'not_required', 'target_text': '{"answer":"MRI","region":null}', 'q_lang': 'en', 'location': 'Abdomen', 'modality': 'MRI', 'answer_type': 'OPEN', 'base_type': 'vqa', 'content_type': 'Modality', 'triple': ['vhead', '_', '_'], 'image_path': '/content/slake_source/imgs/xmlab1/source.jpg'}


In [10]:
training_examples = restored_splits["train"]
validation_examples = restored_splits["validation"]
test_examples = restored_splits["test"]

for name, records in [
    ("Training", training_examples),
    ("Validation", validation_examples),
    ("Test", test_examples),
]:
    grounded = sum(record.get("bbox_xywh_pixels") is not None for record in records)
    print(f"{name}: {len(records)} records; grounded boxes: {grounded}")

Training: 4919 records; grounded boxes: 370
Validation: 1053 records; grounded boxes: 69
Test: 1061 records; grounded boxes: 78


In [11]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model: Gemma4ForConditionalGeneration
Processor: Gemma4Processor
GPU memory used (GB): 6.28


In [12]:
from collections import Counter
from peft import LoraConfig, get_peft_model

suffixes = (
    "self_attn.q_proj",
    "self_attn.k_proj",
    "self_attn.v_proj",
    "self_attn.o_proj",
)

target_modules = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and name.endswith(suffixes)
]

print("LoRA targets found:", len(target_modules))
print("Targets by type:", Counter(name.rsplit(".", 1)[-1] for name in target_modules))

# Freeze the quantized base model; train only the LoRA adapter.
for parameter in model.parameters():
    parameter.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

LoRA targets found: 100
Targets by type: Counter({'q_proj': 35, 'o_proj': 35, 'k_proj': 15, 'v_proj': 15})
trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525
GPU memory used (GB): 6.29


In [13]:
from PIL import Image
import torch

def collate_one_vqa(example):
    image = Image.open(example["image_path"]).convert("RGB")

    user_messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": example["question"]},
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        user_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    full_messages = user_messages + [
        {
            "role": "assistant",
            "content": [{"type": "text", "text": example["target_text"]}],
        }
    ]

    full = processor.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
    )

    prompt_ids = prompt["input_ids"][0]
    full_ids = full["input_ids"][0]

    if not torch.equal(full_ids[:len(prompt_ids)], prompt_ids):
        raise ValueError("Prompt tokens do not match the start of the full example.")

    labels = full["input_ids"].clone()
    labels[:, :len(prompt_ids)] = -100

    model_input_keys = {
        "input_ids",
        "attention_mask",
        "mm_token_type_ids",
        "pixel_values",
        "image_position_ids",
    }

    batch = {key: value for key, value in full.items() if key in model_input_keys}
    batch["labels"] = labels
    return batch


grounded_example = next(
    record for record in training_examples
    if record.get("bbox_xywh_pixels") is not None
)

test_batch = collate_one_vqa(grounded_example)
test_batch = {
    key: value.to(model.device)
    for key, value in test_batch.items()
}

model.eval()
with torch.inference_mode():
    test_output = model(**test_batch)

print("Question:", grounded_example["question"])
print("Target:", grounded_example["target_text"])
print("Batch keys:", list(test_batch.keys()))
print("Loss:", float(test_output.loss))

Question: Does the picture contain liver?
Target: {"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.2109,0.4141,0.3281,0.5352]}}
Batch keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'labels']
Loss: 4.1455206871032715


In [14]:
from transformers import Trainer, TrainingArguments

def collate_vqa(batch):
    if len(batch) != 1:
        raise ValueError("This setup requires batch size 1.")
    return collate_one_vqa(batch[0])

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/gemma4_slake_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=-1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    push_to_hub=True,
    hub_model_id="Thushi2003/slake-grounded-vqa-gemma4-e2b-lora",
    hub_strategy="checkpoint",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_examples,
    data_collator=collate_vqa,
)

print("Trainer created:", trainer is not None)
print("Training rows:", len(training_examples))
print("Hub uploads enabled:", trainer.args.push_to_hub)
print("Hub strategy:", trainer.args.hub_strategy)
print("Repository:", trainer.args.hub_model_id)
print("Training has not started.")

Trainer created: True
Training rows: 4919
Hub uploads enabled: True
Hub strategy: HubStrategy.CHECKPOINT
Repository: Thushi2003/slake-grounded-vqa-gemma4-e2b-lora
Training has not started.


In [15]:
train_result = trainer.train()

# Save and push the final adapter after the last training step.
trainer.save_model()
hub_result = trainer.push_to_hub(
    commit_message="Save final Gemma 4 E2B SLAKE LoRA adapter"
)

print("Training finished.")
print("Optimizer steps:", trainer.state.global_step)
print("Training loss:", train_result.training_loss)
print("Hugging Face model:", hub_result)

Step,Training Loss
20,3.076431
40,0.498300
60,0.445467
80,0.389115
100,0.280777
120,0.271948
140,0.243003
160,0.209485
180,0.212977
200,0.284434


No files have been modified since last commit. Skipping to prevent empty commit.


Training finished.
Optimizer steps: 1230
Training loss: 0.22407870597955656
Hugging Face model: https://huggingface.co/Thushi2003/slake-grounded-vqa-gemma4-e2b-lora/commit/9921696f1c12ca42e21f3fa5befb7599dea4fba7


In [16]:
from huggingface_hub import HfApi

repo_id = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"
files = HfApi().list_repo_files(repo_id=repo_id, repo_type="model")

print("Files in model repository:")
for filename in files:
    print("-", filename)

print("Adapter weights found:",
      any("adapter_model" in filename for filename in files))
print("Adapter config found:",
      any("adapter_config.json" in filename for filename in files))

Files in model repository:
- .gitattributes
- README.md
- adapter_config.json
- adapter_model.safetensors
- last-checkpoint/README.md
- last-checkpoint/adapter_config.json
- last-checkpoint/adapter_model.safetensors
- last-checkpoint/optimizer.pt
- last-checkpoint/rng_state.pth
- last-checkpoint/scaler.pt
- last-checkpoint/scheduler.pt
- last-checkpoint/trainer_state.json
- last-checkpoint/training_args.bin
- training_args.bin
Adapter weights found: True
Adapter config found: True


In [17]:
from PIL import Image
import torch

model = trainer.model
model.eval()

sample = validation_examples[0]
image = Image.open(sample["image_path"]).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": sample["question"]},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

model_input_keys = {
    "input_ids",
    "attention_mask",
    "mm_token_type_ids",
    "pixel_values",
    "image_position_ids",
}
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
    if key in model_input_keys
}

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
prediction = processor.decode(new_tokens, skip_special_tokens=True)

print("Question:", sample["question"])
print("Expected answer:", sample["answer"])
print("Model output:", prediction)

Question: What modality is used to take this image?
Expected answer: MRI
Model output: {"answer":"MRI","region":null}


In [18]:
from PIL import Image
from tqdm.auto import tqdm
import json
import torch

model = trainer.model
model.eval()

prediction_path = "/content/validation_predictions.jsonl"

with open(prediction_path, "w", encoding="utf-8") as output_file:
    for sample in tqdm(validation_examples, desc="Evaluating validation split"):
        image = Image.open(sample["image_path"]).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": sample["question"]},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )

        model_input_keys = {
            "input_ids",
            "attention_mask",
            "mm_token_type_ids",
            "pixel_values",
            "image_position_ids",
        }
        inputs = {
            key: value.to(model.device)
            for key, value in inputs.items()
            if key in model_input_keys
        }

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
            )

        new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
        prediction_text = processor.decode(
            new_tokens,
            skip_special_tokens=True,
        )

        result = {
            "qid": sample.get("qid"),
            "question": sample["question"],
            "expected_answer": sample["answer"],
            "target_text": sample["target_text"],
            "prediction_text": prediction_text,
            "bbox_xywh_pixels": sample.get("bbox_xywh_pixels"),
            "grounded_organ": sample.get("grounded_organ"),
            "image_path": sample["image_path"],
        }

        output_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        output_file.flush()

print("Validation predictions:", len(validation_examples))
print("Saved to:", prediction_path)

Evaluating validation split:   0%|          | 0/1053 [00:00<?, ?it/s]

Validation predictions: 1053
Saved to: /content/validation_predictions.jsonl


In [19]:
from pathlib import Path
from PIL import Image
import json
import re
import numpy as np

prediction_path = Path("/content/validation_predictions.jsonl")

def normalize_answer(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()

def parse_prediction(text):
    text = text.strip()

    candidates = [text]
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        candidates.append(text[start:end + 1])

    for candidate in candidates:
        try:
            value = json.loads(candidate)
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            pass
    return None

def wilson_ci(successes, total, z=1.96):
    if total == 0:
        return None
    p = successes / total
    denom = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denom
    margin = (
        z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2))
        / denom
    )
    return [float(center - margin), float(center + margin)]

def box_iou(box_a, box_b):
    # Boxes are normalized [x1, y1, x2, y2].
    if any(value < 0 or value > 1 for value in box_a + box_b):
        return 0.0
    if box_a[2] <= box_a[0] or box_a[3] <= box_a[1]:
        return 0.0
    if box_b[2] <= box_b[0] or box_b[3] <= box_b[1]:
        return 0.0

    ix1, iy1 = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    ix2, iy2 = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


evaluated = []
answer_failures = []
format_failures = []
localization_failures = []

for line in prediction_path.open(encoding="utf-8"):
    row = json.loads(line)
    parsed = parse_prediction(row["prediction_text"])

    predicted_answer = parsed.get("answer", "") if parsed else ""
    answer_correct = (
        normalize_answer(predicted_answer)
        == normalize_answer(row["expected_answer"])
    )

    item = {
        "qid": row["qid"],
        "question": row["question"],
        "expected_answer": row["expected_answer"],
        "prediction_text": row["prediction_text"],
        "json_valid": parsed is not None,
        "answer_correct": answer_correct,
    }

    if parsed is None:
        format_failures.append(item)
    if not answer_correct:
        answer_failures.append(item)

    gt_box = row.get("bbox_xywh_pixels")
    if gt_box is not None:
        pred_region = parsed.get("region") if parsed else None
        pred_box = (
            pred_region.get("bbox_xyxy_normalized")
            if isinstance(pred_region, dict)
            else None
        )

        if pred_box and len(pred_box) == 4:
            pred_box = [float(value) for value in pred_box]
        else:
            pred_box = None

        with Image.open(row["image_path"]) as image:
            width, height = image.size

        x, y, box_width, box_height = [float(value) for value in gt_box]
        gt_normalized = [
            x / width,
            y / height,
            (x + box_width) / width,
            (y + box_height) / height,
        ]

        iou = box_iou(pred_box, gt_normalized) if pred_box else 0.0
        item["ground_truth_box"] = gt_normalized
        item["predicted_box"] = pred_box
        item["iou"] = iou

        if iou < 0.5:
            localization_failures.append(item)

    evaluated.append(item)

total = len(evaluated)
correct = sum(item["answer_correct"] for item in evaluated)
valid_json = sum(item["json_valid"] for item in evaluated)
grounded = [item for item in evaluated if "iou" in item]
localized = sum(item["iou"] >= 0.5 for item in grounded)
ious = [item["iou"] for item in grounded]

metrics = {
    "validation_questions": total,
    "valid_json": {
        "count": valid_json,
        "rate": valid_json / total,
        "ci95": wilson_ci(valid_json, total),
    },
    "normalized_exact_answer_match": {
        "correct": correct,
        "rate": correct / total,
        "ci95": wilson_ci(correct, total),
    },
    "grounded_questions": len(grounded),
    "localization_iou_at_0_5": {
        "correct": localized,
        "rate": localized / len(grounded) if grounded else None,
        "ci95": wilson_ci(localized, len(grounded)),
    },
    "mean_grounded_box_iou": float(np.mean(ious)) if ious else None,
}

if ious:
    rng = np.random.default_rng(42)
    boot_means = [
        rng.choice(ious, size=len(ious), replace=True).mean()
        for _ in range(2000)
    ]
    metrics["mean_grounded_box_iou_ci95"] = [
        float(np.quantile(boot_means, 0.025)),
        float(np.quantile(boot_means, 0.975)),
    ]

Path("/content/validation_metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
Path("/content/validation_failures.json").write_text(
    json.dumps({
        "answer_failures": answer_failures[:20],
        "format_failures": format_failures[:20],
        "localization_failures": localization_failures[:20],
    }, indent=2),
    encoding="utf-8",
)

print(json.dumps(metrics, indent=2))
print("Failure examples saved to /content/validation_failures.json")

{
  "validation_questions": 1053,
  "valid_json": {
    "count": 1053,
    "rate": 1.0,
    "ci95": [
      0.9963650181824787,
      0.9999999999999999
    ]
  },
  "normalized_exact_answer_match": {
    "correct": 782,
    "rate": 0.7426400759734093,
    "ci95": [
      0.7153854332032427,
      0.7681307342148452
    ]
  },
  "grounded_questions": 69,
  "localization_iou_at_0_5": {
    "correct": 4,
    "rate": 0.057971014492753624,
    "ci95": [
      0.022772020303367495,
      0.13979442524423166
    ]
  },
  "mean_grounded_box_iou": 0.09702100080541774,
  "mean_grounded_box_iou_ci95": [
    0.05424611985624341,
    0.14244178091412343
  ]
}
Failure examples saved to /content/validation_failures.json


In [20]:
import json

with open("/content/validation_metrics.json", encoding="utf-8") as file:
    metrics = json.load(file)

print("Full metrics:")
print(json.dumps(metrics, indent=2))

with open("/content/validation_failures.json", encoding="utf-8") as file:
    failures = json.load(file)

print("\nFirst 5 localization failures:")
for item in failures["localization_failures"][:5]:
    print("\nQID:", item["qid"])
    print("Question:", item["question"])
    print("Expected answer:", item["expected_answer"])
    print("Ground-truth box:", item["ground_truth_box"])
    print("Predicted box:", item["predicted_box"])
    print("IoU:", item["iou"])
    print("Raw model output:", item["prediction_text"])

Full metrics:
{
  "validation_questions": 1053,
  "valid_json": {
    "count": 1053,
    "rate": 1.0,
    "ci95": [
      0.9963650181824787,
      0.9999999999999999
    ]
  },
  "normalized_exact_answer_match": {
    "correct": 782,
    "rate": 0.7426400759734093,
    "ci95": [
      0.7153854332032427,
      0.7681307342148452
    ]
  },
  "grounded_questions": 69,
  "localization_iou_at_0_5": {
    "correct": 4,
    "rate": 0.057971014492753624,
    "ci95": [
      0.022772020303367495,
      0.13979442524423166
    ]
  },
  "mean_grounded_box_iou": 0.09702100080541774,
  "mean_grounded_box_iou_ci95": [
    0.05424611985624341,
    0.14244178091412343
  ]
}

First 5 localization failures:

QID: 9842
Question: Does the picture contain spleen?
Expected answer: Yes
Ground-truth box: [0.76953125, 0.4609375, 0.8515625, 0.64453125]
Predicted box: None
IoU: 0.0
Raw model output: {"answer":"Yes","region":null}

QID: 9850
Question: Does the picture contain liver?
Expected answer: Yes
Ground

In [21]:
import json

with open("/content/validation_metrics.json", encoding="utf-8") as file:
    metrics = json.load(file)

print("Mean box IoU:", metrics["mean_grounded_box_iou"])
print("Mean IoU 95% CI:", metrics.get("mean_grounded_box_iou_ci95"))

with open("/content/validation_failures.json", encoding="utf-8") as file:
    failures = json.load(file)

for item in failures["localization_failures"][:3]:
    print(
        f'\nQID {item["qid"]} | IoU {item["iou"]:.3f}'
        f'\nQuestion: {item["question"]}'
        f'\nExpected: {item["expected_answer"]}'
        f'\nGT box: {item["ground_truth_box"]}'
        f'\nPredicted box: {item["predicted_box"]}'
    )

Mean box IoU: 0.09702100080541774
Mean IoU 95% CI: [0.05424611985624341, 0.14244178091412343]

QID 9842 | IoU 0.000
Question: Does the picture contain spleen?
Expected: Yes
GT box: [0.76953125, 0.4609375, 0.8515625, 0.64453125]
Predicted box: None

QID 9850 | IoU 0.177
Question: Does the picture contain liver?
Expected: Yes
GT box: [0.153125, 0.4875, 0.28125, 0.725]
Predicted box: [0.152, 0.3516, 0.4453, 0.6133]

QID 9953 | IoU 0.000
Question: Does the picture contain liver?
Expected: Yes
GT box: [0.138671875, 0.314453125, 0.205078125, 0.537109375]
Predicted box: [0.2891, 0.3516, 0.414, 0.4844]


In [ ]:
from PIL import Image
from tqdm.auto import tqdm
import json
import torch

model = trainer.model
model.eval()

def generate_text(sample):
    image = Image.open(sample["image_path"]).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": sample["question"]},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    model_input_keys = {
        "input_ids",
        "attention_mask",
        "mm_token_type_ids",
        "pixel_values",
        "image_position_ids",
    }
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
        if key in model_input_keys
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True)


zero_shot_path = "/content/zero_shot_validation_predictions.jsonl"

with model.disable_adapter():
    with open(zero_shot_path, "w", encoding="utf-8") as output_file:
        for sample in tqdm(validation_examples, desc="Zero-shot validation"):
            result = {
                "qid": sample.get("qid"),
                "question": sample["question"],
                "expected_answer": sample["answer"],
                "prediction_text": generate_text(sample),
            }
            output_file.write(json.dumps(result, ensure_ascii=False) + "\n")
            output_file.flush()

print("Zero-shot predictions:", len(validation_examples))
print("Saved to:", zero_shot_path)

: 

In [1]:
import sys
import torch
from pathlib import Path

print("Python:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("Trainer in memory:", "trainer" in globals())

for filename in [
    "validation_predictions.jsonl",
    "validation_metrics.json",
    "zero_shot_validation_predictions.jsonl",
]:
    path = Path("/content") / filename
    if path.exists():
        if path.suffix == ".jsonl":
            print(f"{filename}: {sum(1 for _ in path.open(encoding='utf-8'))} rows")
        else:
            print(f"{filename}: exists")
    else:
        print(f"{filename}: missing")

Python: /usr/bin/python3
CUDA available: False
Trainer in memory: False
validation_predictions.jsonl: missing
validation_metrics.json: missing
zero_shot_validation_predictions.jsonl: missing


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: False
GPU: No GPU


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

ModuleNotFoundError: No module named 'torch'

In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Python: /usr/bin/python3
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import json
import zipfile

dataset_repo = "Thushi2003/slake-grounded-vqa-gemma4-e2b"
source_repo = "BoKelvin/SLAKE"

data_dir = Path("/content/project_data")
source_dir = Path("/content/slake_source")
data_dir.mkdir(parents=True, exist_ok=True)
source_dir.mkdir(parents=True, exist_ok=True)

# Restore validation and test records
for filename in ["validation.jsonl", "test.jsonl"]:
    hf_hub_download(
        repo_id=dataset_repo,
        filename=filename,
        repo_type="dataset",
        local_dir=str(data_dir),
    )

# Restore SLAKE images from the original dataset
images_zip = hf_hub_download(
    repo_id=source_repo,
    filename="imgs.zip",
    repo_type="dataset",
    local_dir=str(source_dir),
)

with zipfile.ZipFile(images_zip) as archive:
    archive.extractall(source_dir)

# Verify records and their image files
for split in ["validation", "test"]:
    records = [
        json.loads(line)
        for line in (data_dir / f"{split}.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    missing = sum(
        not (source_dir / "imgs" / record["img_name"]).exists()
        for record in records
    )
    print(f"{split}: {len(records)} records; missing images: {missing}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


validation.jsonl:   0%|          | 0.00/484k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/488k [00:00<?, ?B/s]

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

validation: 1053 records; missing images: 0
test: 1061 records; missing images: 0


In [3]:
import json
from pathlib import Path

data_dir = Path("/content/project_data")
image_root = Path("/content/slake_source/imgs")

def load_split(split):
    path = data_dir / f"{split}.jsonl"
    records = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    for record in records:
        record["image_path"] = str(image_root / record["img_name"])

    return records

validation_records = load_split("validation")
test_records = load_split("test")

print("Validation records:", len(validation_records))
print("Test records:", len(test_records))
print("Validation boxes:", sum(r["bbox_xywh_pixels"] is not None for r in validation_records))
print("Test boxes:", sum(r["bbox_xywh_pixels"] is not None for r in test_records))
print("Example:", validation_records[0])

Validation records: 1053
Test records: 1061
Validation boxes: 69
Test boxes: 78
Example: {'img_id': 0, 'img_name': 'xmlab0/source.jpg', 'qid': 9835, 'question': 'What modality is used to take this image?', 'answer': 'MRI', 'bbox_xywh_pixels': None, 'grounded_organ': None, 'grounding_status': 'not_required', 'target_text': '{"answer":"MRI","region":null}', 'q_lang': 'en', 'location': 'Abdomen', 'modality': 'MRI', 'answer_type': 'OPEN', 'base_type': 'vqa', 'content_type': 'Modality', 'triple': ['vhead', '_', '_'], 'image_path': '/content/slake_source/imgs/xmlab0/source.jpg'}


In [4]:
import importlib.util
import transformers
import peft
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("bitsandbytes installed:", importlib.util.find_spec("bitsandbytes") is not None)

Transformers: 5.18.0
PEFT: 0.21.1
CUDA: True
GPU: Tesla T4
bitsandbytes installed: False


In [5]:
%pip install -q bitsandbytes==0.50.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.2 MB/s eta 0:00:00:00:0100:01


In [6]:
import bitsandbytes
import torch

print("bitsandbytes:", bitsandbytes.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

bitsandbytes: 0.50.2
CUDA: True
GPU: Tesla T4


In [7]:
import json
from pathlib import Path

data_dir = Path("/content/project_data")
image_root = Path("/content/slake_source/imgs")

def load_split(split):
    path = data_dir / f"{split}.jsonl"
    records = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    for record in records:
        record["image_path"] = str(image_root / record["img_name"])
    return records

validation_records = load_split("validation")
test_records = load_split("test")

print("Validation records:", len(validation_records))
print("Test records:", len(test_records))
print("Validation images found:", sum(Path(r["image_path"]).is_file() for r in validation_records))
print("Test images found:", sum(Path(r["image_path"]).is_file() for r in test_records))

Validation records: 1053
Test records: 1061
Validation images found: 1053
Test images found: 1061


In [8]:
import json
from huggingface_hub import hf_hub_download

ADAPTER_REPO_ID = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

config_path = hf_hub_download(
    repo_id=ADAPTER_REPO_ID,
    filename="adapter_config.json"
)

with open(config_path, encoding="utf-8") as f:
    adapter_config = json.load(f)

print("Base model:", adapter_config.get("base_model_name_or_path"))
print("LoRA rank:", adapter_config.get("r"))
print("LoRA targets:", adapter_config.get("target_modules"))

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6ad5ac1e-65e3d25a06a09f663337d250;b9b54420-644a-44b1-b576-a5a42a40a9d0)

Repository Not Found for url: https://huggingface.co/Thushi2003/slake-grounded-vqa-gemma4-e2b-lora/resolve/main/adapter_config.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [9]:
from getpass import getpass
from huggingface_hub import login, whoami

hf_token = getpass("Paste your Hugging Face token: ")
login(token=hf_token)
print("Signed in as:", whoami()["name"])

Signed in as: Thushi2003


In [10]:
import json
from huggingface_hub import hf_hub_download

ADAPTER_REPO_ID = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

config_path = hf_hub_download(
    repo_id=ADAPTER_REPO_ID,
    filename="adapter_config.json"
)

with open(config_path, encoding="utf-8") as f:
    adapter_config = json.load(f)

print("Base model:", adapter_config.get("base_model_name_or_path"))
print("LoRA rank:", adapter_config.get("r"))
print("LoRA targets:", adapter_config.get("target_modules"))

adapter_config.json:   0%|          | 0.00/6.53k [00:00<?, ?B/s]

Base model: google/gemma-4-E2B-it
LoRA rank: 8
LoRA targets: ['model.language_model.layers.14.self_attn.o_proj', 'model.language_model.layers.14.self_attn.v_proj', 'model.language_model.layers.28.self_attn.q_proj', 'model.language_model.layers.24.self_attn.q_proj', 'model.language_model.layers.23.self_attn.q_proj', 'model.language_model.layers.33.self_attn.q_proj', 'model.language_model.layers.11.self_attn.k_proj', 'model.language_model.layers.10.self_attn.q_proj', 'model.language_model.layers.3.self_attn.q_proj', 'model.language_model.layers.6.self_attn.v_proj', 'model.language_model.layers.14.self_attn.k_proj', 'model.language_model.layers.11.self_attn.v_proj', 'model.language_model.layers.3.self_attn.k_proj', 'model.language_model.layers.18.self_attn.q_proj', 'model.language_model.layers.17.self_attn.o_proj', 'model.language_model.layers.2.self_attn.q_proj', 'model.language_model.layers.5.self_attn.v_proj', 'model.language_model.layers.32.self_attn.o_proj', 'model.language_model.lay

In [11]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "google/gemma-4-E2B-it"
ADAPTER_REPO_ID = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
model.eval()

print("Base model:", MODEL_ID)
print("Model class:", type(model).__name__)
print("Processor class:", type(processor).__name__)
print("Adapter attached:", True)
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 10.7MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Base model: google/gemma-4-E2B-it
Model class: PeftModelForCausalLM
Processor class: Gemma4Processor
Adapter attached: True
GPU memory used (GB): 6.29


In [12]:
from PIL import Image
import torch

record = validation_records[0]
image = Image.open(record["image_path"]).convert("RGB")

prompt = (
    "Answer the question about this medical image. "
    'Return only a JSON object with keys "answer" and "region". '
    'Use "region": null when no image region applies. '
    f"Question: {record['question']}"
)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

device = next(model.parameters()).device
inputs = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

new_tokens = output[0, inputs["input_ids"].shape[1]:]
prediction = processor.decode(new_tokens, skip_special_tokens=True)

print("Question:", record["question"])
print("Expected:", record["target_text"])
print("Prediction:", prediction)

Question: What modality is used to take this image?
Expected: {"answer":"MRI","region":null}
Prediction: {"answer":"MRI","region":null}


In [13]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import torch

predictions_path = Path("/content/test_predictions.jsonl")

# Keep completed predictions so a rerun can continue after an interruption.
completed = {}
if predictions_path.exists():
    with predictions_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                completed[str(item["qid"])] = item

remaining = [r for r in test_records if str(r["qid"]) not in completed]
device = next(model.parameters()).device

with predictions_path.open("a", encoding="utf-8") as f:
    for record in tqdm(remaining, desc="Fine-tuned test predictions"):
        image = Image.open(record["image_path"]).convert("RGB")
        prompt = (
            "Answer the question about this medical image. "
            'Return only a JSON object with keys "answer" and "region". '
            'Use "region": null when no image region applies. '
            f"Question: {record['question']}"
        )
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

        new_tokens = output[0, inputs["input_ids"].shape[1]:]
        prediction = processor.decode(new_tokens, skip_special_tokens=True).strip()

        item = {
            "qid": record["qid"],
            "question": record["question"],
            "expected": record["target_text"],
            "prediction": prediction,
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()

print("Test records:", len(test_records))
print("Predictions saved:", len(completed) + len(remaining))
print("File:", predictions_path)

Fine-tuned test predictions:   0%|          | 0/1061 [00:00<?, ?it/s]

Test records: 1061
Predictions saved: 1061
File: /content/test_predictions.jsonl


In [14]:
import json
import re
import string
import random
from pathlib import Path
from PIL import Image

predictions_path = Path("/content/test_predictions.jsonl")
test_by_qid = {str(r["qid"]): r for r in test_records}

predictions = {}
with predictions_path.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            predictions[str(item["qid"])] = item

def parse_json_object(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def normalize_answer(text):
    text = str(text).lower().strip()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def wilson_ci(successes, total, z=1.96):
    if total == 0:
        return [None, None]
    p = successes / total
    denom = 1 + z*z/total
    center = (p + z*z/(2*total)) / denom
    margin = z * ((p*(1-p)/total + z*z/(4*total*total)) ** 0.5) / denom
    return [max(0, center-margin), min(1, center+margin)]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = iw * ih
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

answer_correct = 0
valid_json_count = 0
grounded_ious = []
answer_failures = []
localization_failures = []

for qid, item in predictions.items():
    record = test_by_qid[qid]
    pred = parse_json_object(item["prediction"])
    expected = json.loads(record["target_text"])

    if pred is not None and "answer" in pred:
        valid_json_count += 1
        is_correct = normalize_answer(pred["answer"]) == normalize_answer(expected["answer"])
    else:
        is_correct = False

    if is_correct:
        answer_correct += 1
    else:
        answer_failures.append({
            "qid": record["qid"],
            "question": record["question"],
            "expected": expected["answer"],
            "prediction": item["prediction"],
        })

    gt = record.get("bbox_xywh_pixels")
    if gt is not None:
        if pred is None:
            iou = 0.0
        else:
            region = pred.get("region")
            pred_box = region.get("bbox_xyxy_normalized") if isinstance(region, dict) else None

            if not isinstance(pred_box, list) or len(pred_box) != 4:
                iou = 0.0
            else:
                with Image.open(record["image_path"]) as im:
                    width, height = im.size
                x, y, w, h = gt
                gt_box = [x/width, y/height, (x+w)/width, (y+h)/height]
                iou = box_iou(gt_box, pred_box)

        grounded_ious.append(iou)
        if iou < 0.5:
            localization_failures.append({
                "qid": record["qid"],
                "question": record["question"],
                "expected_answer": record["answer"],
                "ground_truth_box_xywh_pixels": gt,
                "predicted_region": pred.get("region") if pred else None,
                "iou": iou,
            })

n = len(test_records)
grounded_n = len(grounded_ious)
iou_successes = sum(iou >= 0.5 for iou in grounded_ious)
mean_iou = sum(grounded_ious) / grounded_n if grounded_n else None

# Bootstrap CI for mean IoU over grounded test questions
random.seed(42)
bootstrap_means = []
if grounded_ious:
    for _ in range(2000):
        sample = random.choices(grounded_ious, k=grounded_n)
        bootstrap_means.append(sum(sample) / grounded_n)
    bootstrap_means.sort()
    mean_iou_ci = [
        bootstrap_means[int(0.025 * len(bootstrap_means))],
        bootstrap_means[int(0.975 * len(bootstrap_means))],
    ]
else:
    mean_iou_ci = [None, None]

metrics = {
    "test_questions": n,
    "predictions_found": len(predictions),
    "valid_json": {
        "count": valid_json_count,
        "rate": valid_json_count / n,
        "wilson_95_ci": wilson_ci(valid_json_count, n),
    },
    "normalized_exact_answer_match": {
        "correct": answer_correct,
        "rate": answer_correct / n,
        "wilson_95_ci": wilson_ci(answer_correct, n),
    },
    "grounded_questions": grounded_n,
    "localization_iou_at_0_5": {
        "correct": iou_successes,
        "rate": iou_successes / grounded_n if grounded_n else None,
        "wilson_95_ci": wilson_ci(iou_successes, grounded_n),
    },
    "mean_grounding_iou": {
        "value": mean_iou,
        "bootstrap_95_ci": mean_iou_ci,
    },
}

Path("/content/test_metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
Path("/content/test_failures.json").write_text(
    json.dumps({
        "answer_failures": answer_failures[:20],
        "localization_failures": localization_failures[:20],
    }, indent=2),
    encoding="utf-8",
)

print(json.dumps(metrics, indent=2))
print("Failure examples saved: /content/test_failures.json")

{
  "test_questions": 1061,
  "predictions_found": 1061,
  "valid_json": {
    "count": 1061,
    "rate": 1.0,
    "wilson_95_ci": [
      0.9963923272719621,
      0.9999999999999998
    ]
  },
  "normalized_exact_answer_match": {
    "correct": 741,
    "rate": 0.6983977379830348,
    "wilson_95_ci": [
      0.6701061522486897,
      0.7252578155001284
    ]
  },
  "grounded_questions": 78,
  "localization_iou_at_0_5": {
    "correct": 0,
    "rate": 0.0,
    "wilson_95_ci": [
      0,
      0.04693945377411976
    ]
  },
  "mean_grounding_iou": {
    "value": 0.0,
    "bootstrap_95_ci": [
      0.0,
      0.0
    ]
  }
}
Failure examples saved: /content/test_failures.json


In [15]:
record = test_records[0]
image = Image.open(record["image_path"]).convert("RGB")

prompt = (
    "Answer the question about this medical image. "
    'Return only a JSON object with keys "answer" and "region". '
    'Use "region": null when no image region applies. '
    f"Question: {record['question']}"
)
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
device = next(model.parameters()).device
inputs = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

with model.disable_adapter():
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

new_tokens = output[0, inputs["input_ids"].shape[1]:]
prediction = processor.decode(new_tokens, skip_special_tokens=True)

print("Question:", record["question"])
print("Expected:", record["target_text"])
print("Zero-shot prediction:", prediction)

Question: What modality is used to take this image?
Expected: {"answer":"CT","region":null}
Zero-shot prediction: ```json
{
  "answer": "Computed Tomography (CT)",
  "region": null
}
```


In [1]:
import sys
from pathlib import Path
import torch

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Kaggle working folder exists:", Path("/kaggle/working").is_dir())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {gpu.name}, {gpu.total_memory / 1024**3:.1f} GiB")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Executable: /usr/bin/python3
Kaggle working folder exists: True
PyTorch: 2.11.0+cu128
PyTorch CUDA version: 12.8
CUDA available: True
GPU count: 2
GPU 0: Tesla T4, 14.6 GiB
GPU 1: Tesla T4, 14.6 GiB


In [2]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/content"),
]

matches = []
for root in roots:
    if root.is_dir():
        for path in root.rglob("*"):
            if (
                path.is_file()
                and path.suffix.lower() in {".json", ".jsonl", ".csv", ".zip"}
                and any(word in path.name.lower()
                        for word in ("prediction", "metric", "failure", "zero_shot"))
            ):
                matches.append(path)
                print(f"{path} | {path.stat().st_size:,} bytes")

if not matches:
    print("No matching evaluation files found in this runtime.")

No matching evaluation files found in this runtime.


In [3]:
import importlib.util
from pathlib import Path

for package in ["transformers", "peft", "bitsandbytes", "huggingface_hub", "PIL"]:
    print(f"{package} installed:", importlib.util.find_spec(package) is not None)

print("\nFiles in /kaggle/working:")
print([p.name for p in Path("/kaggle/working").iterdir()][:30])

print("\nFiles in /kaggle/input:")
input_dir = Path("/kaggle/input")
print([p.name for p in input_dir.iterdir()][:30] if input_dir.exists() else "Not found")

transformers installed: True
peft installed: True
bitsandbytes installed: False
huggingface_hub installed: True
PIL installed: True

Files in /kaggle/working:
['.virtual_documents']

Files in /kaggle/input:
[]


In [4]:
%pip install -q bitsandbytes==0.50.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 51.3 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
import bitsandbytes
import torch
import transformers

print("bitsandbytes:", bitsandbytes.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU 0:", torch.cuda.get_device_name(0))

bitsandbytes: 0.50.2
Transformers: 5.16.1
CUDA available: True
GPU 0: Tesla T4


In [6]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

print("Signed in to Hugging Face as:", whoami(token=hf_token)["name"])

BackendError: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 139575812 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.

In [7]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

print("Signed in to Hugging Face as:", whoami(token=hf_token)["name"])

Signed in to Hugging Face as: Thushi2003


In [8]:
import json
import shutil
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA_REPO = "Thushi2003/slake-grounded-vqa-gemma4-e2b"
SOURCE_REPO = "BoKelvin/SLAKE"

work_dir = Path("/kaggle/working")
data_dir = work_dir / "project_data"
image_root = work_dir / "slake_source" / "imgs"
data_dir.mkdir(parents=True, exist_ok=True)
image_root.parent.mkdir(parents=True, exist_ok=True)

# Download validation and test records from your prepared dataset repository.
for split in ["validation", "test"]:
    downloaded = hf_hub_download(
        repo_id=DATA_REPO,
        repo_type="dataset",
        filename=f"{split}.jsonl",
        token=True,
    )
    shutil.copy2(downloaded, data_dir / f"{split}.jsonl")

# Download and extract the SLAKE image archive.
archive_path = Path(hf_hub_download(
    repo_id=SOURCE_REPO,
    repo_type="dataset",
    filename="imgs.zip",
    token=True,
))
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(work_dir / "slake_source")

# Check records and image paths.
for split in ["validation", "test"]:
    records = [
        json.loads(line)
        for line in (data_dir / f"{split}.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    found = sum((image_root / r["img_name"]).is_file() for r in records)
    print(f"{split}: {len(records)} records; images found: {found}")

validation.jsonl:   0%|          | 0.00/484k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/488k [00:00<?, ?B/s]

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

validation: 1053 records; images found: 1053
test: 1061 records; images found: 1061


In [9]:
import json
from pathlib import Path

data_dir = Path("/kaggle/working/project_data")
image_root = Path("/kaggle/working/slake_source/imgs")

def load_split(split):
    path = data_dir / f"{split}.jsonl"
    records = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    for record in records:
        record["image_path"] = str(image_root / record["img_name"])
    return records

validation_records = load_split("validation")
test_records = load_split("test")

print("Validation records:", len(validation_records))
print("Validation boxes:", sum(r["bbox_xywh_pixels"] is not None for r in validation_records))
print("Test records:", len(test_records))
print("Test boxes:", sum(r["bbox_xywh_pixels"] is not None for r in test_records))
print("Example:", validation_records[0]["img_name"], "->", validation_records[0]["image_path"])

Validation records: 1053
Validation boxes: 69
Test records: 1061
Test boxes: 78
Example: xmlab0/source.jpg -> /kaggle/working/slake_source/imgs/xmlab0/source.jpg


In [10]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "google/gemma-4-E2B-it"
ADAPTER_REPO_ID = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
model.eval()

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("Adapter attached:", True)
print("GPU 0 memory used (GB):", round(torch.cuda.memory_allocated(0) / 1024**3, 2))

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

adapter_config.json:   0%|          | 0.00/6.53k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 10.7MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Model: PeftModelForCausalLM
Processor: Gemma4Processor
Adapter attached: True
GPU 0 memory used (GB): 6.29


In [11]:
from PIL import Image
import torch

record = test_records[0]
image = Image.open(record["image_path"]).convert("RGB")

prompt = (
    "Answer the question about this medical image. "
    'Return only a JSON object with keys "answer" and "region". '
    'Use "region": null when no image region applies. '
    f"Question: {record['question']}"
)
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
device = next(model.parameters()).device
inputs = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

def generate_answer():
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

fine_tuned_prediction = generate_answer()

with model.disable_adapter():
    zero_shot_prediction = generate_answer()

print("Question:", record["question"])
print("Expected:", record["target_text"])
print("Fine-tuned:", fine_tuned_prediction)
print("Zero-shot:", zero_shot_prediction)

Question: What modality is used to take this image?
Expected: {"answer":"CT","region":null}
Fine-tuned: {"answer":"CT","region":null}
Zero-shot: ```json
{
  "answer": "Computed Tomography (CT)",
  "region": null
}
```


In [12]:
import random

random.seed(42)

answer_eval_records = random.sample(validation_records, 100)
grounding_eval_records = [
    r for r in validation_records
    if r["bbox_xywh_pixels"] is not None
]

# Combine them without duplicate questions.
eval_by_qid = {str(r["qid"]): r for r in answer_eval_records}
eval_by_qid.update({str(r["qid"]): r for r in grounding_eval_records})
paired_eval_records = list(eval_by_qid.values())

print("Random answer-evaluation questions:", len(answer_eval_records))
print("Grounded localization questions:", len(grounding_eval_records))
print("Unique questions to run in both modes:", len(paired_eval_records))
print("Grounded questions in random sample:",
      sum(r["bbox_xywh_pixels"] is not None for r in answer_eval_records))

Random answer-evaluation questions: 100
Grounded localization questions: 69
Unique questions to run in both modes: 163
Grounded questions in random sample: 6


In [14]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import torch

paired_path = Path("/kaggle/working/paired_eval_predictions.jsonl")
metadata_path = Path("/kaggle/working/paired_eval_metadata.json")

answer_eval_qids = {str(r["qid"]) for r in answer_eval_records}
grounding_eval_qids = {str(r["qid"]) for r in grounding_eval_records}

# Save which questions belong to each metric group.
metadata = {
    "answer_eval_qids": sorted(answer_eval_qids),
    "grounding_eval_qids": sorted(grounding_eval_qids),
    "seed": 42,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

completed = set()
if paired_path.exists():
    with paired_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                completed.add(str(json.loads(line)["qid"]))

remaining = [r for r in paired_eval_records if str(r["qid"]) not in completed]
device = next(model.parameters()).device

def generate_text(inputs):
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

with paired_path.open("a", encoding="utf-8") as f:
    for record in tqdm(remaining, desc="Paired validation evaluation"):
        image = Image.open(record["image_path"]).convert("RGB")
        prompt = (
            "Answer the question about this medical image. "
            'Return only a JSON object with keys "answer" and "region". '
            'Use "region": null when no image region applies. '
            f"Question: {record['question']}"
        )
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in inputs.items()
        }

        fine_tuned_text = generate_text(inputs)
        with model.disable_adapter():
            zero_shot_text = generate_text(inputs)

        item = {
            "qid": record["qid"],
            "question": record["question"],
            "expected": record["target_text"],
            "grounded": record["bbox_xywh_pixels"] is not None,
            "in_answer_sample": str(record["qid"]) in answer_eval_qids,
            "fine_tuned": fine_tuned_text,
            "zero_shot": zero_shot_text,
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()

print("Paired questions saved:", len(completed) + len(remaining))
print("Predictions per question: 2 (fine-tuned and zero-shot)")
print("Prediction file:", paired_path)
print("Subset metadata:", metadata_path)

Paired validation evaluation:   0%|          | 0/110 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import torch

paired_path = Path("/kaggle/working/paired_eval_predictions.jsonl")
metadata_path = Path("/kaggle/working/paired_eval_metadata.json")

answer_eval_qids = {str(r["qid"]) for r in answer_eval_records}
grounding_eval_qids = {str(r["qid"]) for r in grounding_eval_records}

# Save which questions belong to each metric group.
metadata = {
    "answer_eval_qids": sorted(answer_eval_qids),
    "grounding_eval_qids": sorted(grounding_eval_qids),
    "seed": 42,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

completed = set()
if paired_path.exists():
    with paired_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                completed.add(str(json.loads(line)["qid"]))

remaining = [r for r in paired_eval_records if str(r["qid"]) not in completed]
device = next(model.parameters()).device

def generate_text(inputs):
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

with paired_path.open("a", encoding="utf-8") as f:
    for record in tqdm(remaining, desc="Paired validation evaluation"):
        image = Image.open(record["image_path"]).convert("RGB")
        prompt = (
            "Answer the question about this medical image. "
            'Return only a JSON object with keys "answer" and "region". '
            'Use "region": null when no image region applies. '
            f"Question: {record['question']}"
        )
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in inputs.items()
        }

        fine_tuned_text = generate_text(inputs)
        with model.disable_adapter():
            zero_shot_text = generate_text(inputs)

        item = {
            "qid": record["qid"],
            "question": record["question"],
            "expected": record["target_text"],
            "grounded": record["bbox_xywh_pixels"] is not None,
            "in_answer_sample": str(record["qid"]) in answer_eval_qids,
            "fine_tuned": fine_tuned_text,
            "zero_shot": zero_shot_text,
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()

print("Paired questions saved:", len(completed) + len(remaining))
print("Predictions per question: 2 (fine-tuned and zero-shot)")
print("Prediction file:", paired_path)
print("Subset metadata:", metadata_path)

Paired validation evaluation:   0%|          | 0/53 [00:00<?, ?it/s]

Paired questions saved: 163
Predictions per question: 2 (fine-tuned and zero-shot)
Prediction file: /kaggle/working/paired_eval_predictions.jsonl
Subset metadata: /kaggle/working/paired_eval_metadata.json


In [16]:
import json
import re
import string
import random
from pathlib import Path
from PIL import Image

paired_path = Path("/kaggle/working/paired_eval_predictions.jsonl")
metadata_path = Path("/kaggle/working/paired_eval_metadata.json")

with metadata_path.open(encoding="utf-8") as f:
    metadata = json.load(f)

with paired_path.open(encoding="utf-8") as f:
    predictions = {
        str(item["qid"]): item
        for item in (json.loads(line) for line in f if line.strip())
    }

records_by_qid = {str(r["qid"]): r for r in validation_records}
answer_qids = set(metadata["answer_eval_qids"])
grounding_qids = set(metadata["grounding_eval_qids"])

def parse_json_object(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def normalize_answer(text):
    text = str(text).lower().strip()
    text = text.replace("computed tomography", "ct")
    text = text.replace("magnetic resonance imaging", "mri")
    text = text.replace("x-ray", "xray").replace("x ray", "xray")
    text = text.replace("sonography", "ultrasound")
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def wilson_ci(successes, total, z=1.96):
    if total == 0:
        return [None, None]
    p = successes / total
    denom = 1 + z*z/total
    center = (p + z*z/(2*total)) / denom
    margin = z * ((p*(1-p)/total + z*z/(4*total*total)) ** 0.5) / denom
    return [max(0, center-margin), min(1, center+margin)]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = iw * ih
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

def evaluate_mode(field):
    answer_correct = 0
    valid_json = 0
    answer_failures = []

    for qid in answer_qids:
        item = predictions[qid]
        record = records_by_qid[qid]
        expected = json.loads(record["target_text"])
        pred = parse_json_object(item[field])

        if pred is not None and "answer" in pred:
            valid_json += 1
            correct = (
                normalize_answer(pred["answer"])
                == normalize_answer(expected["answer"])
            )
        else:
            correct = False

        if correct:
            answer_correct += 1
        else:
            answer_failures.append({
                "qid": record["qid"],
                "question": record["question"],
                "expected": expected["answer"],
                "prediction": item[field],
            })

    ious = []
    localization_failures = []
    for qid in grounding_qids:
        item = predictions[qid]
        record = records_by_qid[qid]
        pred = parse_json_object(item[field])
        gt = record["bbox_xywh_pixels"]

        if pred is None:
            pred_box = None
        else:
            region = pred.get("region")
            pred_box = (
                region.get("bbox_xyxy_normalized")
                if isinstance(region, dict) else None
            )

        if not isinstance(pred_box, list) or len(pred_box) != 4:
            iou = 0.0
        else:
            with Image.open(record["image_path"]) as im:
                width, height = im.size
            x, y, w, h = gt
            gt_box = [x/width, y/height, (x+w)/width, (y+h)/height]
            iou = box_iou(gt_box, pred_box)

        ious.append(iou)
        if iou < 0.5:
            localization_failures.append({
                "qid": record["qid"],
                "question": record["question"],
                "ground_truth_box_xywh_pixels": gt,
                "predicted_region": pred.get("region") if pred else None,
                "iou": iou,
            })

    hits = sum(iou >= 0.5 for iou in ious)
    mean_iou = sum(ious) / len(ious) if ious else None

    random.seed(42)
    boot_means = [
        sum(random.choices(ious, k=len(ious))) / len(ious)
        for _ in range(2000)
    ] if ious else []
    boot_means.sort()
    mean_iou_ci = (
        [boot_means[50], boot_means[1949]]
        if boot_means else [None, None]
    )

    n_answers = len(answer_qids)
    n_grounded = len(ious)
    return {
        "answer_sample_questions": n_answers,
        "valid_json": {
            "count": valid_json,
            "rate": valid_json / n_answers,
            "wilson_95_ci": wilson_ci(valid_json, n_answers),
        },
        "normalized_exact_answer_match": {
            "correct": answer_correct,
            "rate": answer_correct / n_answers,
            "wilson_95_ci": wilson_ci(answer_correct, n_answers),
        },
        "grounded_questions": n_grounded,
        "localization_iou_at_0_5": {
            "correct": hits,
            "rate": hits / n_grounded,
            "wilson_95_ci": wilson_ci(hits, n_grounded),
        },
        "mean_iou": {
            "value": mean_iou,
            "bootstrap_95_ci": mean_iou_ci,
        },
        "answer_failures": answer_failures[:10],
        "localization_failures": localization_failures[:10],
    }

results = {
    "evaluation_note": (
        "Answer metrics use the seeded random 100-question validation sample; "
        "localization metrics use all 69 grounded validation questions. "
        "Confidence intervals are question-level."
    ),
    "fine_tuned": evaluate_mode("fine_tuned"),
    "zero_shot": evaluate_mode("zero_shot"),
}

Path("/kaggle/working/paired_eval_metrics.json").write_text(
    json.dumps(results, indent=2), encoding="utf-8"
)

print(json.dumps({
    "evaluation_note": results["evaluation_note"],
    "fine_tuned": {
        k: v for k, v in results["fine_tuned"].items()
        if k not in ("answer_failures", "localization_failures")
    },
    "zero_shot": {
        k: v for k, v in results["zero_shot"].items()
        if k not in ("answer_failures", "localization_failures")
    },
}, indent=2))
print("Full metrics and failure examples saved to:")
print("/kaggle/working/paired_eval_metrics.json")

{
  "evaluation_note": "Answer metrics use the seeded random 100-question validation sample; localization metrics use all 69 grounded validation questions. Confidence intervals are question-level.",
  "fine_tuned": {
    "answer_sample_questions": 100,
    "valid_json": {
      "count": 100,
      "rate": 1.0,
      "wilson_95_ci": [
        0.9630051925239981,
        0.9999999999999999
      ]
    },
    "normalized_exact_answer_match": {
      "correct": 74,
      "rate": 0.74,
      "wilson_95_ci": [
        0.6462882736813838,
        0.8159542187301354
      ]
    },
    "grounded_questions": 69,
    "localization_iou_at_0_5": {
      "correct": 0,
      "rate": 0.0,
      "wilson_95_ci": [
        0,
        0.05273909414400562
      ]
    },
    "mean_iou": {
      "value": 0.0,
      "bootstrap_95_ci": [
        0.0,
        0.0
      ]
    }
  },
  "zero_shot": {
    "answer_sample_questions": 100,
    "valid_json": {
      "count": 84,
      "rate": 0.84,
      "wilson_95_ci

In [17]:
import json
from pathlib import Path

with open("/kaggle/working/paired_eval_metrics.json", encoding="utf-8") as f:
    results = json.load(f)

for mode in ["fine_tuned", "zero_shot"]:
    m = results[mode]
    print(f"\n{mode.upper()}")
    print("Answer accuracy:", m["normalized_exact_answer_match"])
    print("Valid JSON:", m["valid_json"])
    print("Localization IoU ≥ 0.5:", m["localization_iou_at_0_5"])
    print("Mean IoU:", m["mean_iou"])


FINE_TUNED
Answer accuracy: {'correct': 74, 'rate': 0.74, 'wilson_95_ci': [0.6462882736813838, 0.8159542187301354]}
Valid JSON: {'count': 100, 'rate': 1.0, 'wilson_95_ci': [0.9630051925239981, 0.9999999999999999]}
Localization IoU ≥ 0.5: {'correct': 0, 'rate': 0.0, 'wilson_95_ci': [0, 0.05273909414400562]}
Mean IoU: {'value': 0.0, 'bootstrap_95_ci': [0.0, 0.0]}

ZERO_SHOT
Answer accuracy: {'correct': 29, 'rate': 0.29, 'wilson_95_ci': [0.21014706973984146, 0.38539074940007934]}
Valid JSON: {'count': 84, 'rate': 0.84, 'wilson_95_ci': [0.7557955544110557, 0.8990479765052628]}
Localization IoU ≥ 0.5: {'correct': 0, 'rate': 0.0, 'wilson_95_ci': [0, 0.05273909414400562]}
Mean IoU: {'value': 0.0, 'bootstrap_95_ci': [0.0, 0.0]}


In [18]:
import json

with open("/kaggle/working/paired_eval_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

grounded_qids = set(metadata["grounding_eval_qids"])
shown = 0

for line in open(
    "/kaggle/working/paired_eval_predictions.jsonl",
    encoding="utf-8"
):
    item = json.loads(line)
    if str(item["qid"]) in grounded_qids:
        record = next(r for r in validation_records if str(r["qid"]) == str(item["qid"]))
        print("\nQID:", item["qid"])
        print("Question:", item["question"])
        print("Expected target:", record["target_text"])
        print("Fine-tuned output:", item["fine_tuned"])
        print("Zero-shot output:", item["zero_shot"])
        shown += 1
        if shown == 5:
            break


QID: 10292
Question: Can cardiomegaly be observed in this picture?
Expected target: {"answer":"Yes","region":{"organ":"Cardiomegaly","bbox_xyxy_normalized":[0.3447,0.4424,0.7891,0.8604]}}
Fine-tuned output: {"answer":"Yes","region":null}
Zero-shot output: ```json
{
  "answer": "Based on the provided chest X-ray, it is difficult to definitively confirm or rule out cardiomegaly without a clinical context and comparison to prior images. However, the cardiac silhouette appears within normal limits for a standard projection, and there is no obvious gross cardiomegaly (enlargement of the heart silhouette).",
  "region": null
}
```

QID: 10120
Question: Does the picture contain liver?
Expected target: {"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1074,0.3008,0.5039,0.7578]}}
Fine-tuned output: {"answer":"Yes","region":null}
Zero-shot output: ```json
{
  "answer": "Yes",
  "region": "liver"
}
```

QID: 10242
Question: Where is the cardiomegaly in this image?
Expected tar

In [19]:
from PIL import Image
import torch

record = next(r for r in validation_records if str(r["qid"]) == "10120")
image = Image.open(record["image_path"]).convert("RGB")

prompt = (
    "Answer the question using the medical image. Return JSON only with keys "
    '"answer" and "region". For a question asking whether an organ is present, '
    "answer Yes or No. If the organ is visible and the answer is Yes, region "
    "must be an object with its organ name and a tight bounding box in "
    "normalized [x1, y1, x2, y2] coordinates from 0 to 1. Do not use null for "
    "a visible organ. If the organ is absent, use region null. For questions "
    "that do not ask about a visible region, use region null. "
    f"Question: {record['question']}"
)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
device = next(model.parameters()).device
inputs = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=120, do_sample=False)

new_tokens = output[0, inputs["input_ids"].shape[1]:]
prediction = processor.decode(new_tokens, skip_special_tokens=True).strip()

print("Question:", record["question"])
print("Expected:", record["target_text"])
print("Prompted prediction:", prediction)

Question: Does the picture contain liver?
Expected: {"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1074,0.3008,0.5039,0.7578]}}
Prompted prediction: {"answer":"Yes","region":{"Liver":{"x1":0.01171875, "y1":0.1171875, "x2":0.5078125, "y2":0.5078125}}}


In [20]:
record = next(r for r in validation_records if str(r["qid"]) == "10120")
image = Image.open(record["image_path"]).convert("RGB")

prompt = (
    "Answer the question using the medical image. Return JSON only, with no "
    "markdown. Follow this exact structure: "
    '{"answer":"Yes","region":{"organ":"Liver",'
    '"bbox_xyxy_normalized":[0.1,0.2,0.5,0.7]}}. '
    "For this question, if the liver is visible, answer Yes and estimate a "
    "tight box around the liver. The box must use the exact keys organ and "
    "bbox_xyxy_normalized. Give normalized coordinates [x1,y1,x2,y2] from 0 "
    "to 1. Do not put the organ name as a key inside region. "
    f"Question: {record['question']}"
)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
device = next(model.parameters()).device
inputs = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=120, do_sample=False)

new_tokens = output[0, inputs["input_ids"].shape[1]:]
print(processor.decode(new_tokens, skip_special_tokens=True).strip())

{"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1,0.2,0.5,0.7]}}


In [21]:
prompt = (
    "Answer the question using the medical image. Return only a JSON object "
    'with keys "answer" and "region"; no markdown. For a question about an '
    "organ, if the organ is visible, answer Yes and set region to an object "
    'with keys "organ" and "bbox_xyxy_normalized". The box must contain four '
    "numbers estimated from the image: normalized x1, y1, x2, y2 coordinates "
    "between 0 and 1, with x1 < x2 and y1 < y2. Estimate the box from the "
    "image itself; do not copy coordinates from this instruction. If the "
    "organ is absent, answer No and set region to null. "
    f"Question: {record['question']}"
)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=120, do_sample=False)

new_tokens = output[0, inputs["input_ids"].shape[1]:]
print(processor.decode(new_tokens, skip_special_tokens=True).strip())

{"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1328,0.2188,0.586,0.65625]}}


In [22]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import torch

prompted_path = Path(
    "/kaggle/working/prompted_grounding_validation.jsonl"
)

completed = set()
if prompted_path.exists():
    with prompted_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                completed.add(str(json.loads(line)["qid"]))

remaining = [
    r for r in grounding_eval_records
    if str(r["qid"]) not in completed
]
device = next(model.parameters()).device

with prompted_path.open("a", encoding="utf-8") as f:
    for record in tqdm(remaining, desc="Prompted grounding validation"):
        image = Image.open(record["image_path"]).convert("RGB")
        prompt = (
            "Answer the radiology question using the image. Return exactly "
            'one JSON object with keys "answer" and "region"; no markdown. '
            "When the answer refers to a positive, visible organ, structure, "
            "or finding, region must be an object with keys "
            '"organ" and "bbox_xyxy_normalized". Estimate a tight box around '
            "the relevant visible region. Use four estimated numbers in "
            "x1,y1,x2,y2 order, normalized from 0 to 1. Estimate the numbers "
            "from the image itself. Use region null only when the question "
            "does not depend on a visible region or the answer is no. "
            f"Question: {record['question']}"
        )
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            output = model.generate(
                **inputs, max_new_tokens=120, do_sample=False
            )

        new_tokens = output[0, inputs["input_ids"].shape[1]:]
        prediction = processor.decode(
            new_tokens, skip_special_tokens=True
        ).strip()

        f.write(json.dumps({
            "qid": record["qid"],
            "question": record["question"],
            "expected": record["target_text"],
            "prediction": prediction,
        }, ensure_ascii=False) + "\n")
        f.flush()

print("Grounded validation records:", len(grounding_eval_records))
print("Prompted predictions saved:", len(completed) + len(remaining))
print("File:", prompted_path)

Prompted grounding validation:   0%|          | 0/69 [00:00<?, ?it/s]

Grounded validation records: 69
Prompted predictions saved: 69
File: /kaggle/working/prompted_grounding_validation.jsonl


In [23]:
import json
import re
import string
import random
from pathlib import Path
from PIL import Image

pred_path = Path("/kaggle/working/prompted_grounding_validation.jsonl")
records_by_qid = {str(r["qid"]): r for r in grounding_eval_records}

with pred_path.open(encoding="utf-8") as f:
    prompted_predictions = [
        json.loads(line) for line in f if line.strip()
    ]

def parse_json_object(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def normalize_answer(text):
    text = str(text).lower().strip()
    text = text.replace("computed tomography", "ct")
    text = text.replace("magnetic resonance imaging", "mri")
    text = text.replace("x-ray", "xray").replace("x ray", "xray")
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def wilson_ci(successes, total, z=1.96):
    if total == 0:
        return [None, None]
    p = successes / total
    denom = 1 + z*z/total
    center = (p + z*z/(2*total)) / denom
    margin = z * ((p*(1-p)/total + z*z/(4*total*total)) ** 0.5) / denom
    return [max(0, center-margin), min(1, center+margin)]

def extract_box(region):
    if not isinstance(region, dict):
        return None

    box = region.get("bbox_xyxy_normalized")
    if isinstance(box, list) and len(box) == 4:
        return [float(v) for v in box]

    # Also handle a nested organ format, if the model uses one.
    for value in region.values():
        if isinstance(value, dict) and all(
            k in value for k in ("x1", "y1", "x2", "y2")
        ):
            return [float(value[k]) for k in ("x1", "y1", "x2", "y2")]
    return None

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

valid_json = 0
answer_correct = 0
ious = []
failures = []

for item in prompted_predictions:
    record = records_by_qid[str(item["qid"])]
    expected = json.loads(record["target_text"])
    pred = parse_json_object(item["prediction"])

    if pred is not None and "answer" in pred:
        valid_json += 1
        correct = normalize_answer(pred["answer"]) == normalize_answer(expected["answer"])
    else:
        correct = False

    answer_correct += int(correct)

    pred_box = extract_box(pred.get("region") if pred else None)
    if pred_box is None:
        iou = 0.0
    else:
        with Image.open(record["image_path"]) as im:
            width, height = im.size
        x, y, w, h = record["bbox_xywh_pixels"]
        gt_box = [x/width, y/height, (x+w)/width, (y+h)/height]
        iou = box_iou(gt_box, pred_box)

    ious.append(iou)
    if iou < 0.5:
        failures.append({
            "qid": record["qid"],
            "question": record["question"],
            "expected_box_xywh_pixels": record["bbox_xywh_pixels"],
            "prediction": item["prediction"],
            "iou": iou,
        })

n = len(prompted_predictions)
hits = sum(v >= 0.5 for v in ious)
random.seed(42)
boot = sorted(
    sum(random.choices(ious, k=n)) / n
    for _ in range(2000)
)
mean_iou = sum(ious) / n

metrics = {
    "grounded_validation_questions": n,
    "valid_json": {
        "count": valid_json,
        "rate": valid_json / n,
        "wilson_95_ci": wilson_ci(valid_json, n),
    },
    "normalized_exact_answer_match": {
        "correct": answer_correct,
        "rate": answer_correct / n,
        "wilson_95_ci": wilson_ci(answer_correct, n),
    },
    "localization_iou_at_0_5": {
        "correct": hits,
        "rate": hits / n,
        "wilson_95_ci": wilson_ci(hits, n),
    },
    "mean_iou": {
        "value": mean_iou,
        "bootstrap_95_ci": [boot[50], boot[1949]],
    },
}

Path("/kaggle/working/prompted_grounding_metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
Path("/kaggle/working/prompted_grounding_failures.json").write_text(
    json.dumps(failures[:20], indent=2), encoding="utf-8"
)

print(json.dumps(metrics, indent=2))
print("Metrics and failure examples saved in /kaggle/working.")

{
  "grounded_validation_questions": 69,
  "valid_json": {
    "count": 69,
    "rate": 1.0,
    "wilson_95_ci": [
      0.9472609058559944,
      1
    ]
  },
  "normalized_exact_answer_match": {
    "correct": 18,
    "rate": 0.2608695652173913,
    "wilson_95_ci": [
      0.17185419574593983,
      0.3751079797142367
    ]
  },
  "localization_iou_at_0_5": {
    "correct": 0,
    "rate": 0.0,
    "wilson_95_ci": [
      0,
      0.05273909414400562
    ]
  },
  "mean_iou": {
    "value": 0.0,
    "bootstrap_95_ci": [
      0.0,
      0.0
    ]
  }
}
Metrics and failure examples saved in /kaggle/working.


In [24]:
import json

with open("/kaggle/working/prompted_grounding_metrics.json", encoding="utf-8") as f:
    metrics = json.load(f)

with open("/kaggle/working/prompted_grounding_validation.jsonl", encoding="utf-8") as f:
    items = [json.loads(line) for line in f if line.strip()]

print("Mean IoU:", metrics["mean_iou"])

for item in items[:5]:
    record = next(r for r in grounding_eval_records if str(r["qid"]) == str(item["qid"]))
    print("\nQID:", item["qid"])
    print("Expected:", record["target_text"])
    print("Prediction:", item["prediction"])

Mean IoU: {'value': 0.0, 'bootstrap_95_ci': [0.0, 0.0]}

QID: 9840
Expected: {"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1289,0.2617,0.5117,0.6836]}}
Prediction: {"answer":"Yes","region":null}

QID: 9842
Expected: {"answer":"Yes","region":{"organ":"Spleen","bbox_xyxy_normalized":[0.7695,0.4609,0.8516,0.6445]}}
Prediction: {"answer":"Yes","region":{"organ":"Spleen","bbox_xyxy_normalized":[0.1531,0.2539,0.3516,0.4453]}}

QID: 9844
Expected: {"answer":"Right","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1289,0.2617,0.5117,0.6836]}}
Prediction: {"answer":"Left","region":null}

QID: 9850
Expected: {"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1531,0.4875,0.2812,0.725]}}
Prediction: {"answer":"Yes","region":null}

QID: 9953
Expected: {"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.1387,0.3145,0.2051,0.5371]}}
Prediction: {"answer":"No","region":null}


In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

SOURCE_REPO = "BoKelvin/SLAKE"
kg_zip = hf_hub_download(
    repo_id=SOURCE_REPO,
    repo_type="dataset",
    filename="KG.zip",
    token=True,
)

kg_dir = Path("/kaggle/working/slake_kg")
kg_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(kg_zip) as archive:
    archive.extractall(kg_dir)
    print("Files in KG.zip:")
    for name in archive.namelist()[:30]:
        print(name)

print("\nExtracted files:")
for path in list(kg_dir.rglob("*"))[:30]:
    if path.is_file():
        print(path.relative_to(kg_dir), "-", path.stat().st_size, "bytes")

In [1]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

SOURCE_REPO = "BoKelvin/SLAKE"
kg_zip = hf_hub_download(
    repo_id=SOURCE_REPO,
    repo_type="dataset",
    filename="KG.zip",
    token=True,
)

kg_dir = Path("/kaggle/working/slake_kg")
kg_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(kg_zip) as archive:
    archive.extractall(kg_dir)
    print("Files in KG.zip:")
    for name in archive.namelist()[:30]:
        print(name)

print("\nExtracted files:")
for path in list(kg_dir.rglob("*"))[:30]:
    if path.is_file():
        print(path.relative_to(kg_dir), "-", path.stat().st_size, "bytes")

c:\Users\ThushiharanPirabahar\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ThushiharanPirabahar\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ThushiharanPirabahar\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run P

Files in KG.zip:
KG/
KG/en_organ_rel.csv
KG/en_organ.csv
KG/organ_rel.csv
KG/organ.csv
KG/en_disease.csv
KG/disease.csv

Extracted files:
KG\disease.csv - 128790 bytes
KG\en_disease.csv - 161528 bytes
KG\en_organ.csv - 14485 bytes
KG\en_organ_rel.csv - 3645 bytes
KG\organ.csv - 11770 bytes
KG\organ_rel.csv - 2796 bytes


c:\Users\ThushiharanPirabahar\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ThushiharanPirabahar\.cache\huggingface\hub\datasets--BoKelvin--SLAKE. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [2]:
import sys
import importlib.util
from pathlib import Path

print("Python:", sys.executable)
print("/kaggle/working exists:", Path("/kaggle/working").exists())
print("PyTorch installed:", importlib.util.find_spec("torch") is not None)

if importlib.util.find_spec("torch"):
    import torch
    print("CUDA available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Python: c:\Users\ThushiharanPirabahar\AppData\Local\Programs\Python\Python314\python.exe
/kaggle/working exists: True
PyTorch installed: False


In [1]:
import sys
import importlib.util
from pathlib import Path

print("Python:", sys.executable)
print("Kaggle working folder exists:", Path("/kaggle/working").exists())
print("PyTorch installed:", importlib.util.find_spec("torch") is not None)

if importlib.util.find_spec("torch") is not None:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "No GPU",
    )

Python: /usr/bin/python3
Kaggle working folder exists: True
PyTorch installed: True
CUDA available: True
GPU: Tesla T4


In [2]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

kg_zip = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    repo_type="dataset",
    filename="KG.zip",
    token=True,
)

kg_dir = Path("/kaggle/working/slake_kg")
kg_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(kg_zip) as archive:
    archive.extractall(kg_dir)
    print("Files in KG.zip:")
    for name in archive.namelist():
        print(name)

print("\nExtracted CSV files:")
for path in kg_dir.rglob("*.csv"):
    print(path.relative_to(kg_dir), "-", path.stat().st_size, "bytes")

LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [3]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

kg_zip = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    repo_type="dataset",
    filename="KG.zip",
)

kg_dir = Path("/kaggle/working/slake_kg")
kg_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(kg_zip) as archive:
    archive.extractall(kg_dir)
    print("Files in KG.zip:")
    for name in archive.namelist():
        print(name)

print("\nExtracted CSV files:")
for path in kg_dir.rglob("*.csv"):
    print(path.relative_to(kg_dir), "-", path.stat().st_size, "bytes")

KG.zip: reconstructing file:   0%|          |  0.00B / 81.7kB            

KG.zip: downloading bytes:           |  0.00B            

Files in KG.zip:
KG/
KG/en_organ_rel.csv
KG/en_organ.csv
KG/organ_rel.csv
KG/organ.csv
KG/en_disease.csv
KG/disease.csv

Extracted CSV files:
KG/organ.csv - 11770 bytes
KG/disease.csv - 128790 bytes
KG/en_organ.csv - 14485 bytes
KG/en_disease.csv - 161528 bytes
KG/organ_rel.csv - 2796 bytes
KG/en_organ_rel.csv - 3645 bytes


In [4]:
import pandas as pd
from collections import Counter
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")

for filename in ["en_organ.csv", "en_disease.csv", "en_organ_rel.csv"]:
    df = pd.read_csv(kg_root / filename)
    print(f"\n--- {filename} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

for split_name, records in [
    ("validation", validation_records),
    ("test", test_records),
]:
    print(f"\n--- {split_name} question metadata ---")
    print("base_type counts:", Counter(str(r.get("base_type")) for r in records))
    print("content_type counts:", Counter(str(r.get("content_type")) for r in records))

    meaningful = [
        r for r in records
        if isinstance(r.get("triple"), list)
        and any(str(value) not in {"_", "vhead", "None"} for value in r["triple"])
    ]
    print("Questions with non-placeholder triples:", len(meaningful))
    for record in meaningful[:5]:
        print({
            "question": record["question"],
            "answer": record["answer"],
            "base_type": record.get("base_type"),
            "content_type": record.get("content_type"),
            "triple": record.get("triple"),
        })

ParserError: Error tokenizing data. C error: Expected 3 fields in line 13, saw 6


In [5]:
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")

for filename in ["en_organ.csv", "en_disease.csv", "en_organ_rel.csv"]:
    path = kg_root / filename
    lines = path.read_text(encoding="utf-8-sig").splitlines()

    print(f"\n--- {filename}: first 15 raw lines ---")
    for line in lines[:15]:
        print(repr(line))


--- en_organ.csv: first 15 raw lines ---
'organ#attribute#value'
'Motor System#component#bone, bone junctions, skeletal muscles'
'Motor System#function#movement and support'
'Skeleton#definition#a structure used to support something'
'Joint#definition#connecting points between bone fragments of human bones'
'Joint#component#joint capsule, articular surface, articular cavity'
'Ligament#definition#ligaments belong to dense connective tissue'
'Ligament#classification#elastic connective tissue, dense connective tissue'
'Ligament#nature#flexible, fibrous dense connective tissue'
'Ligament#effect#limit the range of motion of bones to avoid injury'
'Spine#position#the median of back and the upper end is connected with the skull'
'Spine#characteristic#gradual widening from top to bottom'
'Digestive System#component#oral cavity, pharynx, esophagus, stomach, small intestine, large intestine'
'Digestive System#status#one of the nine systems of the human body'
'Digestive System#component#digestiv

In [6]:
import pandas as pd
from collections import Counter
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")

for filename in ["en_organ.csv", "en_disease.csv", "en_organ_rel.csv"]:
    df = pd.read_csv(kg_root / filename, sep="#")
    print(f"\n--- {filename} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

for split_name, records in [
    ("validation", validation_records),
    ("test", test_records),
]:
    print(f"\n--- {split_name} question metadata ---")
    print("base_type counts:", Counter(str(r.get("base_type")) for r in records))
    print("content_type counts:", Counter(str(r.get("content_type")) for r in records))

    meaningful = [
        r for r in records
        if isinstance(r.get("triple"), list)
        and any(str(value) not in {"_", "vhead", "None"} for value in r["triple"])
    ]
    print("Questions with non-placeholder triples:", len(meaningful))
    for record in meaningful[:5]:
        print({
            "question": record["question"],
            "answer": record["answer"],
            "base_type": record.get("base_type"),
            "content_type": record.get("content_type"),
            "triple": record.get("triple"),
        })


--- en_organ.csv ---
Rows: 281
Columns: ['organ', 'attribute', 'value']
       organ  attribute                                  value
Motor System  component bone, bone junctions, skeletal muscles
Motor System   function                   movement and support
    Skeleton definition  a structure used to support something

--- en_disease.csv ---
Rows: 2216
Columns: ['disease', 'attribute', 'value']
             disease   attribute                                                                                                                                                                                                                                value
Alveolar Proteinosis    location                                                                                                                                                                                                                                 Lung
Alveolar Proteinosis description a rare disease characterized by positive

NameError: name 'validation_records' is not defined

In [7]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

data_dir = Path("/kaggle/working/project_data")
image_root = Path("/kaggle/working/slake_source/imgs")
kg_root = Path("/kaggle/working/slake_kg/KG")

def load_split(split):
    records = [
        json.loads(line)
        for line in (data_dir / f"{split}.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    for record in records:
        record["image_path"] = str(image_root / record["img_name"])
    return records

validation_records = load_split("validation")
test_records = load_split("test")

for filename in ["en_organ.csv", "en_disease.csv", "en_organ_rel.csv"]:
    df = pd.read_csv(kg_root / filename, sep="#")
    print(f"\n--- {filename} ---")
    print("Rows:", len(df), "| Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

for split_name, records in [("validation", validation_records), ("test", test_records)]:
    print(f"\n--- {split_name} questions ---")
    print("base_type counts:", Counter(str(r.get("base_type")) for r in records))
    print("content_type counts:", Counter(str(r.get("content_type")) for r in records))

    meaningful = [
        r for r in records
        if isinstance(r.get("triple"), list)
        and any(str(v) not in {"_", "vhead", "None"} for v in r["triple"])
    ]
    print("Non-placeholder triples:", len(meaningful))
    for r in meaningful[:5]:
        print(r["question"], "->", r["answer"], "|", r.get("triple"))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/project_data/validation.jsonl'

In [8]:
from pathlib import Path

for folder in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n{folder} exists:", folder.exists())
    if folder.exists():
        for item in list(folder.iterdir())[:40]:
            print(item.name, "(folder)" if item.is_dir() else "(file)")


/kaggle/working exists: True
.virtual_documents (folder)
slake_kg (folder)

/kaggle/input exists: True


In [9]:
import json
import shutil
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA_REPO = "Thushi2003/slake-grounded-vqa-gemma4-e2b"
SOURCE_REPO = "BoKelvin/SLAKE"

work_dir = Path("/kaggle/working")
data_dir = work_dir / "project_data"
image_root = work_dir / "slake_source" / "imgs"
data_dir.mkdir(parents=True, exist_ok=True)
image_root.parent.mkdir(parents=True, exist_ok=True)

for split in ["validation", "test"]:
    file_path = hf_hub_download(
        repo_id=DATA_REPO,
        repo_type="dataset",
        filename=f"{split}.jsonl",
    )
    shutil.copy2(file_path, data_dir / f"{split}.jsonl")

images_zip = hf_hub_download(
    repo_id=SOURCE_REPO,
    repo_type="dataset",
    filename="imgs.zip",
)

with zipfile.ZipFile(images_zip) as archive:
    archive.extractall(work_dir / "slake_source")

for split in ["validation", "test"]:
    records = [
        json.loads(line)
        for line in (data_dir / f"{split}.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    found = sum((image_root / r["img_name"]).is_file() for r in records)
    print(f"{split}: {len(records)} records; images found: {found}")

validation.jsonl:   0%|          | 0.00/484k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/488k [00:00<?, ?B/s]

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

validation: 1053 records; images found: 1053
test: 1061 records; images found: 1061


In [10]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

data_dir = Path("/kaggle/working/project_data")
image_root = Path("/kaggle/working/slake_source/imgs")
kg_root = Path("/kaggle/working/slake_kg/KG")

def load_split(split):
    records = [
        json.loads(line)
        for line in (data_dir / f"{split}.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    for record in records:
        record["image_path"] = str(image_root / record["img_name"])
    return records

validation_records = load_split("validation")
test_records = load_split("test")

for filename in ["en_organ.csv", "en_disease.csv", "en_organ_rel.csv"]:
    df = pd.read_csv(kg_root / filename, sep="#")
    print(f"\n--- {filename} ---")
    print("Rows:", len(df), "| Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

for split_name, records in [("validation", validation_records), ("test", test_records)]:
    print(f"\n--- {split_name} question metadata ---")
    print("base_type counts:", Counter(str(r.get("base_type")) for r in records))
    print("content_type counts:", Counter(str(r.get("content_type")) for r in records))

    meaningful = [
        r for r in records
        if isinstance(r.get("triple"), list)
        and any(str(v) not in {"_", "vhead", "None"} for v in r["triple"])
    ]
    print("Questions with non-placeholder triples:", len(meaningful))
    for r in meaningful[:5]:
        print({
            "question": r["question"],
            "answer": r["answer"],
            "base_type": r.get("base_type"),
            "content_type": r.get("content_type"),
            "triple": r.get("triple"),
        })


--- en_organ.csv ---
Rows: 281 | Columns: ['organ', 'attribute', 'value']
       organ  attribute                                  value
Motor System  component bone, bone junctions, skeletal muscles
Motor System   function                   movement and support
    Skeleton definition  a structure used to support something

--- en_disease.csv ---
Rows: 2216 | Columns: ['disease', 'attribute', 'value']
             disease   attribute                                                                                                                                                                                                                                value
Alveolar Proteinosis    location                                                                                                                                                                                                                                 Lung
Alveolar Proteinosis description a rare disease characterized by posi

In [11]:
from collections import Counter

for split_name, records in [
    ("validation", validation_records),
    ("test", test_records),
]:
    kg_questions = [r for r in records if r.get("base_type") == "kvqa"]
    print(f"\n{split_name} KG questions:", len(kg_questions))
    print("Relations:", Counter(r["triple"][1] for r in kg_questions))

    for r in kg_questions[:10]:
        print({
            "image": r["img_name"],
            "question": r["question"],
            "answer": r["answer"],
            "location": r.get("location"),
            "triple": r.get("triple"),
        })


validation KG questions: 155
Relations: Counter({'belong to': 37, 'effect': 22, 'part of': 15, 'function': 12, 'damage': 11, 'be used for': 11, 'treat': 9, 'symptom': 8, 'prevent': 8, 'can': 8, 'could': 7, 'cause': 6, 'role': 1})
{'image': 'xmlab303/source.jpg', 'question': 'How much damage will be caused to the body taking such an image?', 'answer': 'A Little', 'location': 'Lung', 'triple': ['vhead', 'damage', 'ktail']}
{'image': 'xmlab303/source.jpg', 'question': 'How to treat the disease located on the left of lung in this image?', 'answer': 'Medical treatment, surgical treatment', 'location': 'Lung', 'triple': ['vhead', 'treat', 'ktail']}
{'image': 'xmlab303/source.jpg', 'question': 'Which organs/organ in the picture belong to the circulatory system?', 'answer': 'Heart', 'location': 'Lung', 'triple': ['vhead', 'belong to', 'circulatory system']}
{'image': 'xmlab307/source.jpg', 'question': 'What is the function of the main organ in this picture?', 'answer': 'Breathe', 'location': 

In [12]:
import pandas as pd
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")

for filename, entity_col in [
    ("en_organ.csv", "organ"),
    ("en_disease.csv", "disease"),
    ("en_organ_rel.csv", "organ"),
]:
    df = pd.read_csv(kg_root / filename, sep="#")
    print(f"\n--- {filename} ---")
    print("Relation counts:")
    print(df["attribute"].value_counts().to_string())

    print("\nExample rows:")
    print(df.head(5).to_string(index=False))


--- en_organ.csv ---
Relation counts:
attribute
function                         70
position                         46
definition                       33
component                        24
effect                           20
shape                            16
nature                           10
characteristic                    6
length                            5
classification                    4
alias                             3
diameter                          3
weight                            3
divided into                      2
long                              2
type                              2
size                              2
main ingredient                   1
secretion                         1
texture                           1
status                            1
tissue structure                  1
therapeutic method                1
uplink                            1
downlink                          1
artery                            1
vein           

In [17]:
import json
import re
import string
import random
from pathlib import Path
from PIL import Image

paired_path = Path("/kaggle/working/paired_eval_predictions.jsonl")
metadata_path = Path("/kaggle/working/paired_eval_metadata.json")

with metadata_path.open(encoding="utf-8") as f:
    metadata = json.load(f)

with paired_path.open(encoding="utf-8") as f:
    predictions = {
        str(item["qid"]): item
        for item in (json.loads(line) for line in f if line.strip())
    }

records_by_qid = {str(r["qid"]): r for r in validation_records}
answer_qids = set(metadata["answer_eval_qids"])
grounding_qids = set(metadata["grounding_eval_qids"])

def parse_json_object(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def normalize_answer(text):
    text = str(text).lower().strip()
    text = text.replace("computed tomography", "ct")
    text = text.replace("magnetic resonance imaging", "mri")
    text = text.replace("x-ray", "xray").replace("x ray", "xray")
    text = text.replace("sonography", "ultrasound")
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def wilson_ci(successes, total, z=1.96):
    if total == 0:
        return [None, None]
    p = successes / total
    denom = 1 + z*z/total
    center = (p + z*z/(2*total)) / denom
    margin = z * ((p*(1-p)/total + z*z/(4*total*total)) ** 0.5) / denom
    return [max(0, center-margin), min(1, center+margin)]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = iw * ih
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

def evaluate_mode(field):
    answer_correct = 0
    valid_json = 0
    answer_failures = []

    for qid in answer_qids:
        item = predictions[qid]
        record = records_by_qid[qid]
        expected = json.loads(record["target_text"])
        pred = parse_json_object(item[field])

        if pred is not None and "answer" in pred:
            valid_json += 1
            correct = (
                normalize_answer(pred["answer"])
                == normalize_answer(expected["answer"])
            )
        else:
            correct = False

        if correct:
            answer_correct += 1
        else:
            answer_failures.append({
                "qid": record["qid"],
                "question": record["question"],
                "expected": expected["answer"],
                "prediction": item[field],
            })

    ious = []
    localization_failures = []
    for qid in grounding_qids:
        item = predictions[qid]
        record = records_by_qid[qid]
        pred = parse_json_object(item[field])
        gt = record["bbox_xywh_pixels"]

        if pred is None:
            pred_box = None
        else:
            region = pred.get("region")
            pred_box = (
                region.get("bbox_xyxy_normalized")
                if isinstance(region, dict) else None
            )

        if not isinstance(pred_box, list) or len(pred_box) != 4:
            iou = 0.0
        else:
            with Image.open(record["image_path"]) as im:
                width, height = im.size
            x, y, w, h = gt
            gt_box = [x/width, y/height, (x+w)/width, (y+h)/height]
            iou = box_iou(gt_box, pred_box)

        ious.append(iou)
        if iou < 0.5:
            localization_failures.append({
                "qid": record["qid"],
                "question": record["question"],
                "ground_truth_box_xywh_pixels": gt,
                "predicted_region": pred.get("region") if pred else None,
                "iou": iou,
            })

    hits = sum(iou >= 0.5 for iou in ious)
    mean_iou = sum(ious) / len(ious) if ious else None

    random.seed(42)
    boot_means = [
        sum(random.choices(ious, k=len(ious))) / len(ious)
        for _ in range(2000)
    ] if ious else []
    boot_means.sort()
    mean_iou_ci = (
        [boot_means[50], boot_means[1949]]
        if boot_means else [None, None]
    )

    n_answers = len(answer_qids)
    n_grounded = len(ious)
    return {
        "answer_sample_questions": n_answers,
        "valid_json": {
            "count": valid_json,
            "rate": valid_json / n_answers,
            "wilson_95_ci": wilson_ci(valid_json, n_answers),
        },
        "normalized_exact_answer_match": {
            "correct": answer_correct,
            "rate": answer_correct / n_answers,
            "wilson_95_ci": wilson_ci(answer_correct, n_answers),
        },
        "grounded_questions": n_grounded,
        "localization_iou_at_0_5": {
            "correct": hits,
            "rate": hits / n_grounded,
            "wilson_95_ci": wilson_ci(hits, n_grounded),
        },
        "mean_iou": {
            "value": mean_iou,
            "bootstrap_95_ci": mean_iou_ci,
        },
        "answer_failures": answer_failures[:10],
        "localization_failures": localization_failures[:10],
    }

results = {
    "evaluation_note": (
        "Answer metrics use the seeded random 100-question validation sample; "
        "localization metrics use all 69 grounded validation questions. "
        "Confidence intervals are question-level."
    ),
    "fine_tuned": evaluate_mode("fine_tuned"),
    "zero_shot": evaluate_mode("zero_shot"),
}

Path("/kaggle/working/paired_eval_metrics.json").write_text(
    json.dumps(results, indent=2), encoding="utf-8"
)

print(json.dumps({
    "evaluation_note": results["evaluation_note"],
    "fine_tuned": {
        k: v for k, v in results["fine_tuned"].items()
        if k not in ("answer_failures", "localization_failures")
    },
    "zero_shot": {
        k: v for k, v in results["zero_shot"].items()
        if k not in ("answer_failures", "localization_failures")
    },
}, indent=2))
print("Full metrics and failure examples saved to:")
print("/kaggle/working/paired_eval_metrics.json")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/paired_eval_metadata.json'

In [18]:
import json
import re
from pathlib import Path
import pandas as pd

kg_root = Path("/kaggle/working/slake_kg/KG")
data_path = Path("/kaggle/working/project_data/validation.jsonl")

# Load facts from the English KG files
kg_files = [
    ("en_organ.csv", "organ"),
    ("en_organ_rel.csv", "organ"),
    ("en_disease.csv", "disease"),
]

facts = []
for filename, entity_column in kg_files:
    df = pd.read_csv(kg_root / filename, sep="#").fillna("")
    for _, row in df.iterrows():
        facts.append({
            "entity": str(row[entity_column]).strip(),
            "relation": str(row["attribute"]).strip().lower(),
            "value": str(row["value"]).strip(),
            "source": filename,
        })

# Load validation examples and keep KG questions
records = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
kg_records = [
    r for r in records
    if r.get("base_type") == "kvqa" or r.get("content_type") == "KG"
]

stop_words = {
    "what", "which", "where", "how", "will", "does", "the", "a", "an",
    "in", "on", "to", "of", "for", "is", "this", "picture", "image",
    "body", "patient", "main", "organ", "disease", "located", "such",
    "bring", "taking", "that", "can", "could"
}

def words(text):
    tokens = re.findall(r"[a-z]+", str(text).lower())
    return {w[:-1] if len(w) > 4 and w.endswith("s") else w for w in tokens} - stop_words

def retrieve_facts(record, top_k=3):
    triple = record.get("triple") or ["_", "_", "_"]
    relation = str(triple[1]).strip().lower()
    requested_tail = str(triple[2]).strip().lower()

    # Restrict candidates to the requested relation.
    candidates = [f for f in facts if f["relation"] == relation]

    # If the question specifies a system or other tail, use it as a retrieval constraint.
    if requested_tail not in {"_", "ktail", ""}:
        tail_words = words(requested_tail)
        matching = [
            f for f in candidates
            if tail_words and tail_words.issubset(words(f["value"]))
        ]
        if matching:
            candidates = matching

    query_words = words(record.get("question", "")) | words(record.get("location", ""))
    ranked = []
    for fact in candidates:
        fact_words = words(fact["entity"]) | words(fact["value"])
        overlap = len(query_words & fact_words)
        ranked.append((overlap, fact))

    ranked.sort(key=lambda item: item[0], reverse=True)
    return [fact for _, fact in ranked[:top_k]]

print("KG facts loaded:", len(facts))
print("Validation KG questions:", len(kg_records))
print("\nTop retrieval candidates for the first 10 KG questions:\n")

for record in kg_records[:10]:
    print("Question:", record["question"])
    print("Gold answer (for inspection only):", record["answer"])
    print("Triple:", record.get("triple"))
    candidates = retrieve_facts(record)
    for candidate in candidates:
        print("  ", candidate["entity"], "|", candidate["relation"], "|", candidate["value"])
    print()

KG facts loaded: 2600
Validation KG questions: 155

Top retrieval candidates for the first 10 KG questions:

Question: How much damage will be caused to the body taking such an image?
Gold answer (for inspection only): A Little
Triple: ['vhead', 'damage', 'ktail']

Question: How to treat the disease located on the left of lung in this image?
Gold answer (for inspection only): Medical treatment, surgical treatment
Triple: ['vhead', 'treat', 'ktail']

Question: Which organs/organ in the picture belong to the circulatory system?
Gold answer (for inspection only): Heart
Triple: ['vhead', 'belong to', 'circulatory system']
   Heart | belong to | Circulatory System
   Artery | belong to | Circulatory System
   Vein | belong to | Circulatory System

Question: What is the function of the main organ in this picture?
Gold answer (for inspection only): Breathe
Triple: ['vhead', 'function', 'ktail']
   Pharynx | function | respiratory function
   Gallbladder | function | bile can be directly disch

In [19]:
import pandas as pd
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")

files = [
    ("en_organ.csv", "organ"),
    ("en_organ_rel.csv", "organ"),
    ("en_disease.csv", "disease"),
]

for filename, entity_column in files:
    df = pd.read_csv(kg_root / filename, sep="#").fillna("")
    df["attribute_clean"] = df["attribute"].astype(str).str.strip().str.lower()

    print(f"\n--- {filename} ---")
    print("Rows:", len(df))
    print("Attributes:")
    print(df["attribute_clean"].value_counts().to_string())

    relevant = df[
        df["attribute_clean"].str.contains(
            r"damage|harm|treat|function|effect|symptom|cause|prevent|use|role|belong|part",
            regex=True,
            na=False
        )
    ]

    print("\nRelevant fact examples:")
    for _, row in relevant.head(15).iterrows():
        print(
            f'{row[entity_column]} | '
            f'{row["attribute_clean"]} | '
            f'{row["value"]}'
        )


--- en_organ.csv ---
Rows: 281
Attributes:
attribute_clean
function                         70
position                         46
definition                       33
component                        24
effect                           20
shape                            16
nature                           10
characteristic                    6
length                            5
classification                    4
alias                             3
diameter                          3
weight                            3
divided into                      2
long                              2
type                              2
size                              2
main ingredient                   1
secretion                         1
texture                           1
status                            1
tissue structure                  1
therapeutic method                1
uplink                            1
downlink                          1
artery                            1
vein

In [20]:
import pandas as pd
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")
files = [
    ("en_organ.csv", "organ"),
    ("en_organ_rel.csv", "organ"),
    ("en_disease.csv", "disease"),
]

wanted = {
    "function", "effect", "damage",
    "treatment", "prevention", "symptom", "cause"
}

for filename, entity_column in files:
    df = pd.read_csv(kg_root / filename, sep="#").fillna("")
    df["attribute_clean"] = df["attribute"].astype(str).str.strip().str.lower()

    print(f"\n--- {filename} ---")
    print("Relevant attribute counts:")
    counts = df[df["attribute_clean"].isin(wanted)]["attribute_clean"].value_counts()
    print(counts.to_dict())

    for attribute in sorted(wanted):
        rows = df[df["attribute_clean"] == attribute]
        if len(rows):
            print(f"\n{attribute} examples:")
            for _, row in rows.head(3).iterrows():
                print(f'  {row[entity_column]} | {row["value"]}')


--- en_organ.csv ---
Relevant attribute counts:
{'function': 70, 'effect': 20}

effect examples:
  Ligament | limit the range of motion of bones to avoid injury
  Tooth | cutting, chewing, maintaining facial contour and assisting pronunciation
  Esophagus | food delivery

function examples:
  Motor System | movement and support
  Tongue | perceived taste
  Pharynx | is the passage of gas

--- en_organ_rel.csv ---
Relevant attribute counts:
{}

--- en_disease.csv ---
Relevant attribute counts:
{'treatment': 304, 'symptom': 304, 'cause': 298, 'prevention': 277}

cause examples:
  Alveolar Proteinosis | decreased immune function, genetic factors, alcohol abuse, microbial infection
  Pertussis | bordetella pertussis
  Asthmatoid Bronchitis | often secondary to upper respiratory tract infection

prevention examples:
  Alveolar Proteinosis | avoid bacterial infection and improve immunity
  Pertussis | vaccination
  Asthmatoid Bronchitis | pay attention to family and child allergy history

s

In [21]:
import json
import re
import pandas as pd
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")
data_path = Path("/kaggle/working/project_data/validation.jsonl")

files = [
    ("en_organ.csv", "organ"),
    ("en_organ_rel.csv", "organ"),
    ("en_disease.csv", "disease"),
]

facts = []
disease_locations = {}

for filename, entity_column in files:
    df = pd.read_csv(kg_root / filename, sep="#").fillna("")
    df["attribute_clean"] = df["attribute"].astype(str).str.strip().str.lower()

    if filename == "en_disease.csv":
        for _, row in df[df["attribute_clean"] == "location"].iterrows():
            disease_locations[str(row["disease"]).strip().lower()] = (
                str(row["value"]).strip().lower()
            )

    for _, row in df.iterrows():
        facts.append({
            "entity": str(row[entity_column]).strip(),
            "relation": str(row["attribute"]).strip().lower(),
            "value": str(row["value"]).strip(),
            "source": filename,
        })

records = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
kg_records = [
    r for r in records
    if r.get("base_type") == "kvqa" or r.get("content_type") == "KG"
]

relation_aliases = {
    "treat": "treatment",
    "prevent": "prevention",
}

def retrieve_facts(record, top_k=3):
    triple = record.get("triple") or ["_", "_", "_"]
    question_relation = str(triple[1]).strip().lower()
    kg_relation = relation_aliases.get(question_relation, question_relation)

    candidates = [f for f in facts if f["relation"] == kg_relation]

    # For disease facts, restrict candidates to diseases associated with
    # the image's body location when matching location facts are available.
    location = str(record.get("location", "")).strip().lower()
    if candidates and location:
        disease_candidates = [
            f for f in candidates
            if f["source"] == "en_disease.csv"
            and disease_locations.get(f["entity"].lower(), "") == location
        ]
        if disease_candidates:
            candidates = disease_candidates

    # If the triple specifies a target system, restrict by that KG value.
    requested_tail = str(triple[2]).strip().lower()
    if requested_tail not in {"_", "ktail", ""}:
        tail = requested_tail.replace("_", " ")
        matching = [f for f in candidates if tail in f["value"].lower()]
        if matching:
            candidates = matching

    # Prefer facts whose entity matches the image location.
    candidates.sort(
        key=lambda f: (
            f["entity"].lower() != location,
            f["entity"].lower() not in location
        )
    )
    return candidates[:top_k], kg_relation

print("Validation KG questions:", len(kg_records))
print("Relation aliases:", relation_aliases)
print("Damage relation supported by KG:", any(f["relation"] == "damage" for f in facts))

for record in kg_records[:10]:
    retrieved, mapped_relation = retrieve_facts(record)
    print(f"\nQuestion: {record['question']}")
    print(f"KG relation used: {mapped_relation} | location: {record.get('location')}")
    print("Gold answer (inspection only):", record["answer"])
    if not retrieved:
        print("No matching KG facts.")
    else:
        for fact in retrieved:
            print(f'  {fact["entity"]} | {fact["relation"]} | {fact["value"]}')

Validation KG questions: 155
Relation aliases: {'treat': 'treatment', 'prevent': 'prevention'}
Damage relation supported by KG: False

Question: How much damage will be caused to the body taking such an image?
KG relation used: damage | location: Lung
Gold answer (inspection only): A Little
No matching KG facts.

Question: How to treat the disease located on the left of lung in this image?
KG relation used: treatment | location: Lung
Gold answer (inspection only): Medical treatment, surgical treatment
  Alveolar Proteinosis | treatment | bronchoalveolar lavage
  Lobar Pneumonia | treatment | antibiotic drug therapy
  Pulmonary Actinomycosis | treatment | high dose penicillin treatment, surgical treatment

Question: Which organs/organ in the picture belong to the circulatory system?
KG relation used: belong to | location: Lung
Gold answer (inspection only): Heart
  Heart | belong to | Circulatory System
  Artery | belong to | Circulatory System
  Vein | belong to | Circulatory System

Q

In [22]:
import json
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict

kg_root = Path("/kaggle/working/slake_kg/KG")
data_path = Path("/kaggle/working/project_data/validation.jsonl")

files = [
    ("en_organ.csv", "organ"),
    ("en_organ_rel.csv", "organ"),
    ("en_disease.csv", "disease"),
]

facts = []
disease_locations = {}

for filename, entity_column in files:
    df = pd.read_csv(kg_root / filename, sep="#").fillna("")
    df["attribute_clean"] = df["attribute"].astype(str).str.strip().str.lower()

    if filename == "en_disease.csv":
        for _, row in df[df["attribute_clean"] == "location"].iterrows():
            disease_locations[str(row["disease"]).strip().lower()] = (
                str(row["value"]).strip().lower()
            )

    for _, row in df.iterrows():
        facts.append({
            "entity": str(row[entity_column]).strip(),
            "relation": str(row["attribute"]).strip().lower(),
            "value": str(row["value"]).strip(),
            "source": filename,
        })

records = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
kg_records = [
    r for r in records
    if r.get("base_type") == "kvqa" or r.get("content_type") == "KG"
]

relation_aliases = {"treat": "treatment", "prevent": "prevention"}

def normalize(text):
    return " ".join(re.findall(r"[a-z0-9]+", str(text).lower()))

def get_candidates(record):
    triple = record.get("triple") or ["_", "_", "_"]
    relation = str(triple[1]).strip().lower()
    relation = relation_aliases.get(relation, relation)
    candidates = [f for f in facts if f["relation"] == relation]

    requested_tail = str(triple[2]).strip().lower()
    if requested_tail not in {"_", "ktail", ""}:
        tail = normalize(requested_tail)
        matching = [f for f in candidates if tail in normalize(f["value"])]
        if matching:
            candidates = matching

    location = str(record.get("location", "")).strip().lower()
    disease_candidates = [
        f for f in candidates
        if f["source"] == "en_disease.csv"
        and disease_locations.get(f["entity"].lower(), "") == location
    ]
    if disease_candidates:
        candidates = disease_candidates

    # Rank with question/location terms, never with the gold answer.
    query_terms = set(normalize(record.get("question", "")).split())
    query_terms |= set(normalize(location).split())

    def score(fact):
        fact_terms = set(normalize(fact["entity"] + " " + fact["value"]).split())
        return len(query_terms & fact_terms)

    return sorted(candidates, key=score, reverse=True), relation

def answer_supported(answer, fact):
    gold = normalize(answer)
    fact_text = normalize(fact["entity"] + " " + fact["value"])
    return bool(gold) and (gold in fact_text or fact_text in gold)

by_relation = defaultdict(lambda: {
    "questions": 0, "has_candidates": 0,
    "gold_supported_any": 0, "gold_supported_top1": 0, "gold_supported_top5": 0
})

for record in kg_records:
    candidates, relation = get_candidates(record)
    stats = by_relation[relation]
    stats["questions"] += 1
    stats["has_candidates"] += bool(candidates)
    stats["gold_supported_any"] += any(
        answer_supported(record["answer"], fact) for fact in candidates
    )
    stats["gold_supported_top1"] += bool(
        candidates and answer_supported(record["answer"], candidates[0])
    )
    stats["gold_supported_top5"] += any(
        answer_supported(record["answer"], fact) for fact in candidates[:5]
    )

print("Validation KG questions:", len(kg_records))
print("Per-relation retrieval results:")
for relation, stats in sorted(by_relation.items()):
    print(relation, stats)

totals = {
    key: sum(stats[key] for stats in by_relation.values())
    for key in ["questions", "has_candidates", "gold_supported_any",
                "gold_supported_top1", "gold_supported_top5"]
}
print("\nTotals:", totals)

Validation KG questions: 155
Per-relation retrieval results:
be used for {'questions': 11, 'has_candidates': 0, 'gold_supported_any': 0, 'gold_supported_top1': 0, 'gold_supported_top5': 0}
belong to {'questions': 37, 'has_candidates': 37, 'gold_supported_any': 22, 'gold_supported_top1': 12, 'gold_supported_top5': 16}
can {'questions': 8, 'has_candidates': 0, 'gold_supported_any': 0, 'gold_supported_top1': 0, 'gold_supported_top5': 0}
cause {'questions': 6, 'has_candidates': 6, 'gold_supported_any': 5, 'gold_supported_top1': 0, 'gold_supported_top5': 0}
could {'questions': 7, 'has_candidates': 0, 'gold_supported_any': 0, 'gold_supported_top1': 0, 'gold_supported_top5': 0}
damage {'questions': 11, 'has_candidates': 0, 'gold_supported_any': 0, 'gold_supported_top1': 0, 'gold_supported_top5': 0}
effect {'questions': 22, 'has_candidates': 22, 'gold_supported_any': 0, 'gold_supported_top1': 0, 'gold_supported_top5': 0}
function {'questions': 12, 'has_candidates': 12, 'gold_supported_any': 11

In [23]:
shown = 0

for record in kg_records:
    candidates, relation = get_candidates(record)

    matching_ranks = [
        i + 1 for i, fact in enumerate(candidates)
        if answer_supported(record["answer"], fact)
    ]

    if matching_ranks and matching_ranks[0] > 5:
        print(f"\nRelation: {relation}")
        print("Question:", record["question"])
        print("Gold answer:", record["answer"])
        print("First matching fact rank:", matching_ranks[0])
        print("Top 5 retrieved facts:")
        for fact in candidates[:5]:
            print(f'  {fact["entity"]} | {fact["relation"]} | {fact["value"]}')
        print("Matching fact:")
        print(" ", candidates[matching_ranks[0] - 1])
        shown += 1

        if shown == 5:
            break

if shown == 0:
    print("No cases found where a matching fact exists below rank 5.")


Relation: treatment
Question: How to treat the disease located on the left of lung in this image?
Gold answer: Medical treatment, surgical treatment
First matching fact rank: 8
Top 5 retrieved facts:
  Lung Cancer | treatment | pharmacotherapy, surgical therapy, radiotherapy, chemotherapy, targeted drug therapy
  Alveolar Proteinosis | treatment | bronchoalveolar lavage
  Lobar Pneumonia | treatment | antibiotic drug therapy
  Pulmonary Actinomycosis | treatment | high dose penicillin treatment, surgical treatment
  Radiation Pneumonia | treatment | drug symptomatic treatment mainly, early may need hormone treatment
Matching fact:
  {'entity': 'Emphysema', 'relation': 'treatment', 'value': 'medical treatment, surgical treatment', 'source': 'en_disease.csv'}

Relation: function
Question: What is the function of the main organ in this picture?
Gold answer: Breathe
First matching fact rank: 8
Top 5 retrieved facts:
  Pharynx | function | is the passage of gas
  Gallbladder | function | b

In [24]:
import math

def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return (0.0, 0.0)

    p = successes / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = (
        z
        * math.sqrt(p * (1 - p) / total + z**2 / (4 * total**2))
        / denominator
    )
    return (center - margin, center + margin)

ks = [1, 5, 10, 20]
hits = {k: 0 for k in ks}
covered = 0
no_candidates = 0

for record in kg_records:
    candidates, relation = get_candidates(record)

    if not candidates:
        no_candidates += 1
        continue

    covered += 1
    for k in ks:
        if any(
            answer_supported(record["answer"], fact)
            for fact in candidates[:k]
        ):
            hits[k] += 1

total = len(kg_records)
print("Validation KG questions:", total)
print("Questions with at least one candidate:", covered)
print("Questions with no candidate:", no_candidates)

for k in ks:
    low, high = wilson_interval(hits[k], total)
    print(
        f"Lexical answer support in top {k}: "
        f"{hits[k]}/{total} = {hits[k] / total:.1%}, "
        f"95% Wilson CI [{low:.1%}, {high:.1%}]"
    )

Validation KG questions: 155
Questions with at least one candidate: 102
Questions with no candidate: 53
Lexical answer support in top 1: 12/155 = 7.7%, 95% Wilson CI [4.5%, 13.0%]
Lexical answer support in top 5: 16/155 = 10.3%, 95% Wilson CI [6.5%, 16.1%]
Lexical answer support in top 10: 33/155 = 21.3%, 95% Wilson CI [15.6%, 28.4%]
Lexical answer support in top 20: 40/155 = 25.8%, 95% Wilson CI [19.6%, 33.2%]


In [25]:
relation_aliases["part of"] = "belong to"

hits = {k: 0 for k in [1, 5, 10, 20]}
covered = 0
no_candidates = 0

for record in kg_records:
    candidates, relation = get_candidates(record)

    if not candidates:
        no_candidates += 1
        continue

    covered += 1
    for k in hits:
        if any(
            answer_supported(record["answer"], fact)
            for fact in candidates[:k]
        ):
            hits[k] += 1

print("Questions with candidates:", covered)
print("Questions with no candidates:", no_candidates)

for k, count in hits.items():
    low, high = wilson_interval(count, len(kg_records))
    print(
        f"Top {k}: {count}/{len(kg_records)} = {count / len(kg_records):.1%}, "
        f"95% CI [{low:.1%}, {high:.1%}]"
    )

Questions with candidates: 117
Questions with no candidates: 38
Top 1: 24/155 = 15.5%, 95% CI [10.6%, 22.0%]
Top 5: 29/155 = 18.7%, 95% CI [13.4%, 25.6%]
Top 10: 47/155 = 30.3%, 95% CI [23.6%, 38.0%]
Top 20: 54/155 = 34.8%, 95% CI [27.8%, 42.6%]


In [26]:
import json
import re
import random
import math
from pathlib import Path
from collections import defaultdict
from PIL import Image
import torch

# Confirm the active Kaggle session has the required objects
for name in ["model", "processor", "kg_records", "get_candidates"]:
    if name not in globals():
        raise NameError(f"{name} is not loaded. Restore/reload it before this evaluation.")

image_root = Path("/kaggle/working/slake_source/imgs")
output_path = Path("/kaggle/working/rag_paired_eval.jsonl")
metrics_path = Path("/kaggle/working/rag_paired_metrics.json")

# Fixed, relation-balanced sample: up to four questions per relation
grouped = defaultdict(list)
for record in kg_records:
    triple = record.get("triple") or ["_", "_", "_"]
    grouped[str(triple[1]).strip().lower()].append(record)

rng = random.Random(42)
sample = []
for relation in sorted(grouped):
    rows = grouped[relation]
    sample.extend(rng.sample(rows, min(4, len(rows))))

print("Evaluation questions:", len(sample))
print("Relations represented:", len(grouped))
print("GPU available:", torch.cuda.is_available())

def generate_answer(record, use_retrieval):
    image_path = Path(record.get("image_path", ""))
    if not image_path.is_file():
        image_path = image_root / record["img_name"]
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    retrieved, mapped_relation = get_candidates(record)
    retrieved = retrieved[:5]

    if use_retrieval and retrieved:
        facts_text = "\n".join(
            f'- {f["entity"]} | {f["relation"]} | {f["value"]}'
            for f in retrieved
        )
        knowledge_instruction = (
            "Use these retrieved KG facts when they are relevant. "
            "Do not claim a fact that the evidence does not support.\n"
            f"KG facts:\n{facts_text}"
        )
    elif use_retrieval:
        knowledge_instruction = (
            "No matching KG facts were retrieved. Do not invent KG evidence."
        )
    else:
        knowledge_instruction = "Answer without using external KG facts."

    prompt = f"""
Answer this SLAKE medical visual question.
{knowledge_instruction}

Return only a JSON object in this format:
{{"answer":"short answer","region":null}}

Question: {record["question"]}
""".strip()

    with Image.open(image_path) as im:
        image = im.convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
        )

    prompt_length = inputs["input_ids"].shape[1]
    text = processor.decode(
        output_ids[0][prompt_length:],
        skip_special_tokens=True,
    ).strip()

    # Parse JSON even if the model adds a code fence or extra text
    parsed = None
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    return {
        "raw_output": text,
        "parsed": parsed,
        "retrieved_facts": retrieved if use_retrieval else [],
        "mapped_relation": mapped_relation,
    }

def normalize_answer(text):
    return " ".join(re.findall(r"[a-z0-9]+", str(text).lower()))

results = []
for i, record in enumerate(sample, start=1):
    no_rag = generate_answer(record, use_retrieval=False)
    with_rag = generate_answer(record, use_retrieval=True)

    gold = normalize_answer(record["answer"])
    results.append({
        "qid": record.get("qid"),
        "question": record["question"],
        "gold_answer": record["answer"],
        "without_retrieval": no_rag,
        "with_retrieval": with_rag,
        "correct_without_retrieval": (
            no_rag["parsed"] is not None
            and normalize_answer(no_rag["parsed"].get("answer", "")) == gold
        ),
        "correct_with_retrieval": (
            with_rag["parsed"] is not None
            and normalize_answer(with_rag["parsed"].get("answer", "")) == gold
        ),
    })

    if i % 5 == 0 or i == len(sample):
        print(f"Completed {i}/{len(sample)} questions")

with output_path.open("w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def wilson(successes, total, z=1.96):
    if total == 0:
        return [0.0, 0.0]
    p = successes / total
    d = 1 + z*z/total
    center = (p + z*z/(2*total)) / d
    margin = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / d
    return [center - margin, center + margin]

n = len(results)
without_correct = sum(r["correct_without_retrieval"] for r in results)
with_correct = sum(r["correct_with_retrieval"] for r in results)
without_json = sum(r["without_retrieval"]["parsed"] is not None for r in results)
with_json = sum(r["with_retrieval"]["parsed"] is not None for r in results)

metrics = {
    "note": "Seeded relation-balanced validation sample; exact normalized answer match.",
    "questions": n,
    "without_retrieval": {
        "exact_answer_correct": without_correct,
        "accuracy": without_correct / n,
        "wilson_95_ci": wilson(without_correct, n),
        "valid_json": without_json / n,
    },
    "with_retrieval": {
        "exact_answer_correct": with_correct,
        "accuracy": with_correct / n,
        "wilson_95_ci": wilson(with_correct, n),
        "valid_json": with_json / n,
    },
    "paired_changes": {
        "improved": sum(
            r["correct_with_retrieval"] and not r["correct_without_retrieval"]
            for r in results
        ),
        "worsened": sum(
            r["correct_without_retrieval"] and not r["correct_with_retrieval"]
            for r in results
        ),
        "unchanged": sum(
            r["correct_with_retrieval"] == r["correct_without_retrieval"]
            for r in results
        ),
    },
}

metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("\nMetrics:")
print(json.dumps(metrics, indent=2))
print("\nSaved predictions:", output_path)
print("Saved metrics:", metrics_path)

NameError: model is not loaded. Restore/reload it before this evaluation.

In [27]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, AutoModelForMultimodalLM
from peft import PeftModel

MODEL_ID = "google/gemma-4-E2B-it"
ADAPTER_REPO_ID = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

assert torch.cuda.is_available(), "Reconnect to a Kaggle GPU first."
print("GPU:", torch.cuda.get_device_name(0))

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=True)

base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
    token=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO_ID,
    token=True,
)
model.eval()

free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("Adapter loaded:", model is not None)
print("GPU free (GB):", round(free_bytes / 1024**3, 2))

GPU: Tesla T4


LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [28]:
from getpass import getpass
from huggingface_hub import login, whoami

hf_token = getpass("Paste your Hugging Face read token (input is hidden): ")
login(token=hf_token, add_to_git_credential=False)

print("Signed in as:", whoami(token=hf_token)["name"])
del hf_token

HfHubHTTPError: Invalid user token.

In [29]:
from getpass import getpass
from huggingface_hub import login, whoami

hf_token = getpass("Paste your Hugging Face read token (input is hidden): ")
login(token=hf_token, add_to_git_credential=False)

print("Signed in as:", whoami(token=hf_token)["name"])
del hf_token

Signed in as: Thushi2003


In [30]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, AutoModelForMultimodalLM
from peft import PeftModel

MODEL_ID = "google/gemma-4-E2B-it"
ADAPTER_REPO_ID = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=True)

base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
    token=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO_ID,
    token=True,
)
model.eval()

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("Adapter loaded:", model is not None)
print("GPU:", torch.cuda.get_device_name(0))

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [31]:
%pip install -q bitsandbytes==0.50.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.8 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import bitsandbytes
import transformers
from huggingface_hub import get_token

print("bitsandbytes:", bitsandbytes.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Hugging Face token cached:", bool(get_token()))

bitsandbytes: 0.50.2
Transformers: 5.16.1
CUDA available: True
GPU: Tesla T4
Hugging Face token cached: True


In [2]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, AutoModelForMultimodalLM
from peft import PeftModel

MODEL_ID = "google/gemma-4-E2B-it"
ADAPTER_REPO_ID = "Thushi2003/slake-grounded-vqa-gemma4-e2b-lora"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=True)

base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
    token=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO_ID,
    token=True,
)
model.eval()

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("Adapter loaded:", True)
print("GPU:", torch.cuda.get_device_name(0))

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/6.53k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 10.7MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Model: PeftModelForCausalLM
Processor: Gemma4Processor
Adapter loaded: True
GPU: Tesla T4


In [3]:
import json
import re
import pandas as pd
from pathlib import Path

kg_root = Path("/kaggle/working/slake_kg/KG")
data_path = Path("/kaggle/working/project_data/validation.jsonl")
image_root = Path("/kaggle/working/slake_source/imgs")

assert data_path.is_file(), f"Missing validation file: {data_path}"
assert kg_root.is_dir(), f"Missing KG folder: {kg_root}"
assert image_root.is_dir(), f"Missing images folder: {image_root}"

records = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

for record in records:
    record["image_path"] = str(image_root / record["img_name"])

kg_records = [
    r for r in records
    if r.get("base_type") == "kvqa" or r.get("content_type") == "KG"
]

facts = []
disease_locations = {}

for filename, entity_column in [
    ("en_organ.csv", "organ"),
    ("en_organ_rel.csv", "organ"),
    ("en_disease.csv", "disease"),
]:
    df = pd.read_csv(kg_root / filename, sep="#").fillna("")
    df["attribute_clean"] = df["attribute"].astype(str).str.strip().str.lower()

    if filename == "en_disease.csv":
        for _, row in df[df["attribute_clean"] == "location"].iterrows():
            disease_locations[str(row["disease"]).strip().lower()] = (
                str(row["value"]).strip().lower()
            )

    for _, row in df.iterrows():
        facts.append({
            "entity": str(row[entity_column]).strip(),
            "relation": str(row["attribute"]).strip().lower(),
            "value": str(row["value"]).strip(),
            "source": filename,
        })

relation_aliases = {
    "treat": "treatment",
    "prevent": "prevention",
    "part of": "belong to",
}

def normalize(text):
    return " ".join(re.findall(r"[a-z0-9]+", str(text).lower()))

def get_candidates(record):
    triple = record.get("triple") or ["_", "_", "_"]
    question_relation = str(triple[1]).strip().lower()
    kg_relation = relation_aliases.get(question_relation, question_relation)

    candidates = [f for f in facts if f["relation"] == kg_relation]

    requested_tail = str(triple[2]).strip().lower()
    if requested_tail not in {"_", "ktail", ""}:
        tail = normalize(requested_tail)
        matching = [f for f in candidates if tail in normalize(f["value"])]
        if matching:
            candidates = matching

    location = str(record.get("location", "")).strip().lower()
    disease_candidates = [
        f for f in candidates
        if f["source"] == "en_disease.csv"
        and disease_locations.get(f["entity"].lower(), "") == location
    ]
    if disease_candidates:
        candidates = disease_candidates

    query_terms = set(normalize(record.get("question", "")).split())
    query_terms |= set(normalize(location).split())

    def score(fact):
        fact_terms = set(
            normalize(fact["entity"] + " " + fact["value"]).split()
        )
        return len(query_terms & fact_terms)

    return sorted(candidates, key=score, reverse=True), kg_relation

def answer_supported(answer, fact):
    gold = normalize(answer)
    fact_text = normalize(fact["entity"] + " " + fact["value"])
    return bool(gold) and (gold in fact_text or fact_text in gold)

missing_images = sum(
    not Path(r["image_path"]).is_file() for r in kg_records
)

print("Validation KG questions:", len(kg_records))
print("KG facts loaded:", len(facts))
print("Missing images:", missing_images)
print("Retriever ready:", callable(get_candidates))

Validation KG questions: 155
KG facts loaded: 2600
Missing images: 0
Retriever ready: True


In [4]:
import json, random, re, math
from collections import defaultdict
from pathlib import Path
from PIL import Image
import torch

# Fixed relation-balanced sample for a quicker comparison
groups = defaultdict(list)
for r in kg_records:
    relation = str((r.get("triple") or ["_", "_", "_"])[1]).lower()
    groups[relation].append(r)

rng = random.Random(42)
sample = []
for relation in sorted(groups):
    sample.extend(rng.sample(groups[relation], min(4, len(groups[relation]))))

print("Questions to evaluate:", len(sample))

def norm(text):
    return " ".join(re.findall(r"[a-z0-9]+", str(text).lower()))

def predict_one(record, use_rag):
    candidates, _ = get_candidates(record)
    candidates = candidates[:5] if use_rag else []

    facts_text = "\n".join(
        f'- {f["entity"]} | {f["relation"]} | {f["value"]}'
        for f in candidates
    )
    context = (
        f"Relevant medical KG facts:\n{facts_text}"
        if facts_text else
        "No KG facts are provided."
    )

    prompt = f"""
Answer the medical visual question. Use KG facts only if provided and relevant.
{context}
Return only JSON: {{"answer":"short answer","region":null}}
Question: {record["question"]}
""".strip()

    with Image.open(record["image_path"]) as im:
        image = im.convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=96, do_sample=False)

    text = processor.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    parsed = None
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return text, parsed

results = []
for i, r in enumerate(sample, start=1):
    plain_text, plain_json = predict_one(r, use_rag=False)
    rag_text, rag_json = predict_one(r, use_rag=True)
    gold = norm(r["answer"])

    results.append({
        "qid": r.get("qid"),
        "question": r["question"],
        "gold": r["answer"],
        "without_retrieval": plain_text,
        "with_retrieval": rag_text,
        "correct_without_retrieval": (
            plain_json is not None and norm(plain_json.get("answer", "")) == gold
        ),
        "correct_with_retrieval": (
            rag_json is not None and norm(rag_json.get("answer", "")) == gold
        ),
        "valid_json_without_retrieval": plain_json is not None,
        "valid_json_with_retrieval": rag_json is not None,
    })

    if i % 5 == 0 or i == len(sample):
        print(f"Completed {i}/{len(sample)}")

def wilson(k, n):
    if not n:
        return [0.0, 0.0]
    z = 1.96
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    m = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return [c-m, c+m]

n = len(results)
plain = sum(x["correct_without_retrieval"] for x in results)
rag = sum(x["correct_with_retrieval"] for x in results)

metrics = {
    "sample_questions": n,
    "without_retrieval": {
        "correct": plain, "accuracy": plain/n, "wilson_95_ci": wilson(plain, n)
    },
    "with_retrieval": {
        "correct": rag, "accuracy": rag/n, "wilson_95_ci": wilson(rag, n)
    },
    "improved_by_retrieval": sum(
        x["correct_with_retrieval"] and not x["correct_without_retrieval"]
        for x in results
    ),
    "worsened_by_retrieval": sum(
        x["correct_without_retrieval"] and not x["correct_with_retrieval"]
        for x in results
    ),
}

out = Path("/kaggle/working/rag_paired_eval.jsonl")
out.write_text(
    "".join(json.dumps(x, ensure_ascii=False) + "\n" for x in results),
    encoding="utf-8",
)
Path("/kaggle/working/rag_paired_metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)

print("\nMetrics:")
print(json.dumps(metrics, indent=2))
print("Saved predictions:", out)

Questions to evaluate: 49
Completed 5/49
Completed 10/49
Completed 15/49
Completed 20/49
Completed 25/49
Completed 30/49
Completed 35/49
Completed 40/49
Completed 45/49
Completed 49/49

Metrics:
{
  "sample_questions": 49,
  "without_retrieval": {
    "correct": 22,
    "accuracy": 0.4489795918367347,
    "wilson_95_ci": [
      0.3185262492963633,
      0.5868513320032188
    ]
  },
  "with_retrieval": {
    "correct": 18,
    "accuracy": 0.3673469387755102,
    "wilson_95_ci": [
      0.24664982524781226,
      0.5073318861311014
    ]
  },
  "improved_by_retrieval": 1,
  "worsened_by_retrieval": 5
}
Saved predictions: /kaggle/working/rag_paired_eval.jsonl


In [5]:
record_by_qid = {str(r.get("qid")): r for r in kg_records}

for result in results:
    plain = result["correct_without_retrieval"]
    rag = result["correct_with_retrieval"]

    if plain != rag:
        record = record_by_qid.get(str(result["qid"]))
        candidates, _ = get_candidates(record) if record else ([], None)

        print("\nQID:", result["qid"])
        print("Question:", result["question"])
        print("Gold:", result["gold"])
        print("Without retrieval:", result["without_retrieval"])
        print("With retrieval:", result["with_retrieval"])
        print("Retrieved facts:")
        for fact in candidates[:5]:
            print(f'  {fact["entity"]} | {fact["relation"]} | {fact["value"]}')


QID: 10501
Question: Do any of the organs in the picture belong to the respiratory system?
Gold: Yes
Without retrieval: {"answer":"No","region":null}
With retrieval: {"answer":"Yes","region":null}
Retrieved facts:
  Nasal Cavity | belong to | Respiratory System
  Pharynx | belong to | Respiratory System
  Larynx | belong to | Respiratory System
  Trachea | belong to | Respiratory System
  Bronchial Tube | belong to | Respiratory System

QID: 10415
Question: What is the effect of the center organ in this picture?
Gold: Promote blood flow
Without retrieval: {"answer":"Promote blood flow","region":null}
With retrieval: {"answer":"Promote reproduction","region":null}
Retrieved facts:
  Ligament | effect | limit the range of motion of bones to avoid injury
  Ciliary Body | effect | generation of aqueous humor and adjustment of refractive capacity of the eye
  Gallbladder | effect | concentration and storage of bile
  Testis | effect | reproduction, secretion of hormones
  Vas Deferens | ef

In [6]:
import json
import random
from pathlib import Path
from collections import defaultdict

test_path = Path("/kaggle/working/project_data/test.jsonl")
image_root = Path("/kaggle/working/slake_source/imgs")
assert test_path.is_file(), f"Missing test file: {test_path}"

test_records = [
    json.loads(line)
    for line in test_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
for r in test_records:
    r["image_path"] = str(image_root / r["img_name"])

test_kg = [
    r for r in test_records
    if r.get("base_type") == "kvqa" or r.get("content_type") == "KG"
]

# Same fixed sampling rule as validation: up to four per relation
groups = defaultdict(list)
for r in test_kg:
    relation = str((r.get("triple") or ["_", "_", "_"])[1]).lower()
    groups[relation].append(r)

rng = random.Random(42)
test_sample = []
for relation in sorted(groups):
    test_sample.extend(rng.sample(groups[relation], min(4, len(groups[relation]))))

print("Test KG questions:", len(test_kg))
print("Test questions to evaluate:", len(test_sample))

test_results = []
for i, r in enumerate(test_sample, start=1):
    plain_text, plain_json = predict_one(r, use_rag=False)
    rag_text, rag_json = predict_one(r, use_rag=True)
    gold = norm(r["answer"])

    test_results.append({
        "qid": r.get("qid"),
        "question": r["question"],
        "gold": r["answer"],
        "without_retrieval": plain_text,
        "with_retrieval": rag_text,
        "correct_without_retrieval": (
            plain_json is not None and norm(plain_json.get("answer", "")) == gold
        ),
        "correct_with_retrieval": (
            rag_json is not None and norm(rag_json.get("answer", "")) == gold
        ),
    })

    if i % 5 == 0 or i == len(test_sample):
        print(f"Completed {i}/{len(test_sample)}")

n = len(test_results)
plain = sum(x["correct_without_retrieval"] for x in test_results)
rag = sum(x["correct_with_retrieval"] for x in test_results)

test_metrics = {
    "test_sample_questions": n,
    "without_retrieval": {
        "correct": plain, "accuracy": plain / n, "wilson_95_ci": wilson(plain, n)
    },
    "with_retrieval": {
        "correct": rag, "accuracy": rag / n, "wilson_95_ci": wilson(rag, n)
    },
    "improved_by_retrieval": sum(
        x["correct_with_retrieval"] and not x["correct_without_retrieval"]
        for x in test_results
    ),
    "worsened_by_retrieval": sum(
        x["correct_without_retrieval"] and not x["correct_with_retrieval"]
        for x in test_results
    ),
}

Path("/kaggle/working/test_rag_paired_eval.jsonl").write_text(
    "".join(json.dumps(x, ensure_ascii=False) + "\n" for x in test_results),
    encoding="utf-8",
)
Path("/kaggle/working/test_rag_paired_metrics.json").write_text(
    json.dumps(test_metrics, indent=2),
    encoding="utf-8",
)

print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))

Test KG questions: 148
Test questions to evaluate: 49
Completed 5/49
Completed 10/49
Completed 15/49
Completed 20/49
Completed 25/49
Completed 30/49
Completed 35/49
Completed 40/49
Completed 45/49
Completed 49/49

Test metrics:
{
  "test_sample_questions": 49,
  "without_retrieval": {
    "correct": 26,
    "accuracy": 0.5306122448979592,
    "wilson_95_ci": [
      0.39380627019919845,
      0.6629671810210522
    ]
  },
  "with_retrieval": {
    "correct": 22,
    "accuracy": 0.4489795918367347,
    "wilson_95_ci": [
      0.3185262492963633,
      0.5868513320032188
    ]
  },
  "improved_by_retrieval": 1,
  "worsened_by_retrieval": 5
}
